# Auc_High.ipynb — Research Revision 2

## High-Performance, Leakage-Audited Tox21 Toxicity Prediction

This revision was rebuilt after reviewing the complete previous notebook, its actual run logs, and the uploaded ToxKG-GNN paper.

### Evidence-based changes after the previous run

- The previous **GPS–R-GCN branch produced only 8 KG graphs**, so the chemical-node identity mapping has been rewritten with robust PubChem CID normalization and coverage diagnostics.
- The previous neural multimodal fusion was unstable and reduced cross-validation performance; it has been removed.
- The final model is now **Model 5 — Cross-Fitted Endpoint-Wise Ensemble**, which uses only OOF predictions and automatically rejects weak or redundant branches.
- The molecular preprocessing pipeline now includes a larger, NaN-safe RDKit 2D descriptor panel, train-only descriptor winsorization, fold-local variance filtering, SVD/PCA and scaling.
- DeepTox has been upgraded with **NR/SR hierarchical task sharing**.
- The final ensemble uses a **performance-and-diversity gate**, so a weak ChemBERTa or incomplete KG model cannot reduce the final score.
- XAI has been intentionally excluded. This notebook is a black-box predictive research pipeline.

> **Scientific note:** Mean ROC-AUC ≥ 0.95 is an aggressive target, not a guaranteed output. The notebook never hard-codes scores, uses test labels for tuning, or keeps target-assay edges in ToxKG.


## How to read this notebook

The notebook is organized as a thesis-style executable workflow:

- **Part I** — environment, paths, CUDA verification and reproducibility
- **Part II** — dataset loading, audit and high-performance chemical preprocessing
- **Part III** — frozen scaffold split, 5-fold CV, per-task masking and molecular representations
- **Part IV** — Model 1: Hierarchical DeepTox-Residual Multitask DNN
- **Part V** — Model 2: Optimized AttentiveFP Multitask GNN
- **Part VI** — Model 3: ChemBERTa Multitask Transformer
- **Part VII** — Model 4: Leakage-Safe GPS–R-GCN ToxKG Hybrid
- **Part VIII** — Model 5: Cross-Fitted Endpoint-Wise Ensemble
- **Part IX** — final evaluation, clean tables and visualizations

Every reported model-selection decision is based on training/CV OOF predictions only. The frozen test split is evaluated after weights and thresholds are fixed.


# Part I — Environment, paths and reproducibility

## 1. User configuration
Edit only this cell before running the notebook.

In [ ]:
# ============================
# USER CONFIGURATION
# ============================

EXECUTION_MODE = "STANDARD"   # "QUICK", "STANDARD", or "RESEARCH"
PROJECT_ROOT_OVERRIDE = r"D:\Drug_Toxicity"  # Set None for auto-detection

# Data download
AUTO_DOWNLOAD_TOX21 = True
AUTO_DOWNLOAD_TOXKG = True

# Base-model switches
RUN_MODEL_1_DEEPTOX = True
RUN_MODEL_2_ATTENTIVEFP = True
RUN_MODEL_3_CHEMBERTA = True
RUN_MODEL_4_GPS_TOXKG = True

# Model 5 is always the final cross-fitted endpoint-wise ensemble.
RUN_MODEL_5_ENSEMBLE = True

# Frozen scientific protocol
TRAIN_FRACTION = 0.75
N_CV_FOLDS = 5
PRIMARY_SEED = 13
TARGET_MEAN_AUC = 0.95

# Ensemble gate:
# A weak model is retained only if it is sufficiently close to the best
# OOF model or provides useful error diversity.
ENSEMBLE_MAX_AUC_GAP = 0.040
ENSEMBLE_DIVERSITY_CORR = 0.965
MIN_MODEL_COVERAGE = 0.80

# KG quality gate
MIN_KG_GRAPH_COVERAGE = 0.60
MAX_KG_FIRST_HOP = 128
MAX_KG_SECOND_HOP_PER_GENE = 16
MAX_KG_NODES_PER_MOLECULE = 192

# Low-memory settings for 6–8 GB NVIDIA GPUs
CHEMBERTA_MAX_LENGTH = 192
GRADIENT_ACCUMULATION_STEPS = 4

print("গবেষণা কনফিগারেশন লোড হয়েছে।")
print(f"Execution mode: {EXECUTION_MODE}")
print(f"Cross-validation: {N_CV_FOLDS}-fold")
print("Final model: Model 5 — Cross-Fitted Endpoint-Wise Ensemble")


## 2. Package check and automatic installation

Missing libraries are installed automatically. Before installation, the notebook prints a **Bangla message**.

> **Recommended:** Create an Anaconda environment with Python 3.10 or 3.11 and install a CUDA-enabled PyTorch build manually. If PyTorch is already installed, this cell will not replace it.

In [ ]:
import importlib.util
import subprocess
import sys

PACKAGE_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "yaml": "pyyaml",
    "joblib": "joblib",
    "pyarrow": "pyarrow",
    "rdkit": "rdkit",
    "torch": "torch",
    "torchvision": "torchvision",
    "torch_geometric": "torch-geometric",
    "transformers": "transformers>=4.45",
    "accelerate": "accelerate",
    "optuna": "optuna",
    "huggingface_hub": "huggingface_hub",
}

missing = [
    pip_name
    for module_name, pip_name in PACKAGE_MAP.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print("বাংলা বার্তা: প্রয়োজনীয় কিছু Python package পাওয়া যায়নি। এখন install করা হচ্ছে...")
    print("Install list:", missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--upgrade", "--quiet", *missing]
    )
    print(
        "বাংলা বার্তা: Package installation সম্পন্ন হয়েছে। "
        "নতুন package import না হলে Kernel → Restart Kernel করে Run All দিন।"
    )
else:
    print("বাংলা বার্তা: সব প্রয়োজনীয় package আগে থেকেই install করা আছে।")

## 3. Core imports and clean output style

In [ ]:
import os
import re
import gc
import json
import math
import time
import random
import hashlib
import shutil
import platform
import warnings
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import minimize
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
)
from sklearn.svm import SVC

import joblib
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import (
    AllChem,
    MACCSkeys,
    Descriptors,
    Crippen,
    Lipinski,
    rdMolDescriptors,
)
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.warning")
RDLogger.DisableLog("rdApp.error")

sns.set_context("notebook")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

HTML("""
<style>
.jp-RenderedHTMLCommon h1,
.jp-RenderedHTMLCommon h2,
.jp-RenderedHTMLCommon h3 {
    color: #173B7A;
    font-weight: 700;
}
.output_area pre {
    font-size: 13px;
    line-height: 1.35;
}
.dataframe {
    font-size: 12px;
}
.research-card {
    padding: 14px 18px;
    border-left: 5px solid #2D6CDF;
    background: #F4F7FF;
    border-radius: 8px;
    margin: 8px 0;
}
.success-card {
    padding: 12px 16px;
    border-left: 5px solid #1A9B76;
    background: #EFFBF7;
    border-radius: 8px;
}
.warning-card {
    padding: 12px 16px;
    border-left: 5px solid #E6A700;
    background: #FFF9E8;
    border-radius: 8px;
}
</style>
""")

print("সব core library সফলভাবে import হয়েছে।")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)

## 4. Cross-platform project path

In [ ]:
IN_COLAB = "google.colab" in sys.modules

def resolve_project_root(override=None) -> Path:
    candidates = []

    if override:
        candidates.append(Path(override))

    if IN_COLAB:
        candidates.extend([
            Path("/content/drive/MyDrive/Drug_Toxicity"),
            Path("/content/Drug_Toxicity"),
        ])

    candidates.extend([
        Path(r"D:\Drug_Toxicity"),
        Path.cwd(),
        Path.cwd().parent,
    ])

    for path in candidates:
        try:
            if path.exists() and (path / "datasets").exists():
                return path.resolve()
        except Exception:
            continue

    fallback = Path.cwd() / "Drug_Toxicity"
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback.resolve()


ROOT = resolve_project_root(PROJECT_ROOT_OVERRIDE)

DIRS = {
    "datasets": ROOT / "datasets",
    "raw": ROOT / "datasets" / "raw",
    "processed": ROOT / "datasets" / "processed",
    "splits": ROOT / "datasets" / "splits",
    "cache": ROOT / "datasets" / "cache",
    "external": ROOT / "datasets" / "external",
    "kg_raw": ROOT / "knowledge_graph" / "raw",
    "kg_safe": ROOT / "knowledge_graph" / "processed" / "toxkg_safe_v2",
    "kg_provenance": ROOT / "knowledge_graph" / "provenance",
    "models": ROOT / "models",
    "outputs": ROOT / "outputs",
    "figures": ROOT / "figures",
    "logs": ROOT / "logs",
    "configs": ROOT / "configs",
    "reports": ROOT / "reports",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Notebook output:", DIRS["outputs"])

## 5. Reproducibility and hardware-aware profile

In [ ]:
def seed_everything(seed: int = 13) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(PRIMARY_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU_GB = 0.0
if DEVICE.type == "cuda":
    GPU_GB = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    )

LOW_MEMORY = (DEVICE.type == "cpu") or (GPU_GB < 10)
USE_AMP = DEVICE.type == "cuda"

# The CPU profile is intentionally a reproducibility/debug profile.
# The high-AUC research profile requires CUDA and longer training.
MODE_CONFIG = {
    "QUICK": {
        "epochs": 5,
        "patience": 2,
        "batch_size": 16 if LOW_MEMORY else 32,
        "graph_pretrain_epochs": 0,
        "chemberta_head_epochs": 1,
        "chemberta_finetune_epochs": 1,
    },
    "STANDARD": {
        "epochs": 45 if DEVICE.type == "cpu" else 100,
        "patience": 8 if DEVICE.type == "cpu" else 15,
        "batch_size": 16 if LOW_MEMORY else 48,
        "graph_pretrain_epochs": 2 if DEVICE.type == "cpu" else 8,
        "chemberta_head_epochs": 4 if DEVICE.type == "cpu" else 8,
        "chemberta_finetune_epochs": 8 if DEVICE.type == "cpu" else 24,
    },
    "RESEARCH": {
        "epochs": 80 if DEVICE.type == "cpu" else 180,
        "patience": 12 if DEVICE.type == "cpu" else 22,
        "batch_size": 16 if LOW_MEMORY else 64,
        "graph_pretrain_epochs": 4 if DEVICE.type == "cpu" else 15,
        "chemberta_head_epochs": 6 if DEVICE.type == "cpu" else 12,
        "chemberta_finetune_epochs": 16 if DEVICE.type == "cpu" else 40,
    },
}

if EXECUTION_MODE not in MODE_CONFIG:
    raise ValueError("EXECUTION_MODE must be QUICK, STANDARD, or RESEARCH.")

CFG = MODE_CONFIG[EXECUTION_MODE]


def amp_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )
    return nullcontext()


print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        f"GPU: {torch.cuda.get_device_name(0)} "
        f"({GPU_GB:.1f} GB)"
    )

print("Low-memory mode:", LOW_MEMORY)
print("Training profile:", CFG)

if DEVICE.type != "cuda":
    print(
        "বাংলা সতর্কতা: CUDA GPU detect হয়নি। "
        "এই CPU run-টি reproducibility/debug benchmark হিসেবে ধরা হবে; "
        "high-AUC final research run হিসেবে নয়।"
    )
    print(
        "বাংলা পরামর্শ: আপনার NVIDIA GPU থাকলে নতুন conda environment-এ "
        "CUDA-enabled PyTorch install করে kernel restart করুন। "
        "CPU-only PyTorch থাকলে ChemBERTa, GPS এবং GNN যথেষ্ট train নাও হতে পারে।"
    )


In [ ]:
RUN_CONFIG = {
    "revision": "Auc_High_v2",
    "execution_mode": EXECUTION_MODE,
    "train_fraction": TRAIN_FRACTION,
    "n_cv_folds": N_CV_FOLDS,
    "primary_seed": PRIMARY_SEED,
    "target_mean_auc": TARGET_MEAN_AUC,
    "device": str(DEVICE),
    "gpu_gb": GPU_GB,
    "low_memory": LOW_MEMORY,
    "mode_config": CFG,
    "ensemble_max_auc_gap": ENSEMBLE_MAX_AUC_GAP,
    "ensemble_diversity_corr": ENSEMBLE_DIVERSITY_CORR,
    "minimum_kg_graph_coverage": MIN_KG_GRAPH_COVERAGE,
}

CONFIG_FILE = DIRS["configs"] / "auc_high_v2_run_config.json"

with open(CONFIG_FILE, "w", encoding="utf-8") as file:
    json.dump(
        RUN_CONFIG,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Frozen run configuration saved:", CONFIG_FILE)


# Part II — Data loading, audit and high-performance chemical preprocessing

## 6. Locate or download the Tox21 dataset

In [ ]:
def find_tox21_file() -> Optional[Path]:
    candidates = [
        DIRS["raw"] / "tox21.csv",
        DIRS["datasets"] / "tox21.csv",
        ROOT / "tox21.csv",
        Path.cwd() / "tox21.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


TOX21_PATH = find_tox21_file()

if TOX21_PATH is None and AUTO_DOWNLOAD_TOX21:
    print(
        "বাংলা বার্তা: tox21.csv পাওয়া যায়নি। "
        "এখন public Tox21 dataset download করা হচ্ছে..."
    )

    import gzip
    import urllib.request

    url = (
        "https://deepchemdata.s3-us-west-1.amazonaws.com/"
        "datasets/tox21.csv.gz"
    )

    gz_path = DIRS["raw"] / "tox21.csv.gz"
    csv_path = DIRS["raw"] / "tox21.csv"

    try:
        urllib.request.urlretrieve(url, gz_path)

        with gzip.open(gz_path, "rb") as source:
            with open(csv_path, "wb") as target:
                shutil.copyfileobj(source, target)

        TOX21_PATH = csv_path

        print(
            "বাংলা বার্তা: Tox21 dataset download ও extract "
            "সম্পন্ন হয়েছে।"
        )

    except Exception as error:
        print(
            "বাংলা বার্তা: Dataset automatic download ব্যর্থ হয়েছে."
        )
        print(error)


if TOX21_PATH is None:
    raise FileNotFoundError(
        "tox21.csv পাওয়া যায়নি। ফাইলটি "
        "D:/Drug_Toxicity/datasets/raw/ অথবা datasets/ "
        "ফোল্ডারে রাখুন।"
    )

print("Dataset path:", TOX21_PATH)

## 7. Load data and detect the 12 toxicity endpoints

In [ ]:
raw_df = pd.read_csv(TOX21_PATH)

SMILES_COL = next(
    (
        column
        for column in raw_df.columns
        if column.lower() == "smiles"
    ),
    None,
)

ID_COL = next(
    (
        column
        for column in raw_df.columns
        if column.lower()
        in {
            "mol_id",
            "molid",
            "id",
            "compound_id",
        }
    ),
    None,
)

if SMILES_COL is None:
    raise ValueError("SMILES column পাওয়া যায়নি।")

NON_TARGET_COLUMNS = {SMILES_COL}

if ID_COL:
    NON_TARGET_COLUMNS.add(ID_COL)

TARGET_COLS = [
    column
    for column in raw_df.columns
    if column not in NON_TARGET_COLUMNS
]

for target in TARGET_COLS:
    raw_df[target] = pd.to_numeric(
        raw_df[target],
        errors="coerce",
    )

    invalid_label = ~raw_df[target].isin([0, 1])

    raw_df.loc[
        invalid_label,
        target,
    ] = np.nan


if len(TARGET_COLS) != 12:
    print(
        "সতর্কতা: 12টির পরিবর্তে "
        f"{len(TARGET_COLS)}টি endpoint detected হয়েছে।"
    )

print(
    f"Raw dataset: {raw_df.shape[0]:,} molecules × "
    f"{raw_df.shape[1]} columns"
)

print("Detected endpoints:", TARGET_COLS)

display(raw_df.head(3))

## 8. Initial endpoint audit

This audit is essential because Tox21 is both **partially labeled** and **strongly imbalanced**. Missing values remain missing; they are never interpreted as non-toxic.

In [ ]:
def endpoint_audit(
    dataframe: pd.DataFrame,
    targets: List[str],
) -> pd.DataFrame:

    rows = []

    for target in targets:
        series = dataframe[target]

        n_positive = int((series == 1).sum())
        n_negative = int((series == 0).sum())
        n_missing = int(series.isna().sum())
        n_labeled = n_positive + n_negative

        rows.append({
            "Endpoint": target,
            "Labeled": n_labeled,
            "Toxic (1)": n_positive,
            "Non-Toxic (0)": n_negative,
            "Missing": n_missing,
            "Missing %": (
                100 * n_missing / max(len(dataframe), 1)
            ),
            "Positive %": (
                100 * n_positive / max(n_labeled, 1)
            ),
            "Neg:Pos": (
                n_negative / max(n_positive, 1)
            ),
        })

    return pd.DataFrame(rows)


raw_audit = endpoint_audit(
    raw_df,
    TARGET_COLS,
)

display(
    raw_audit.round(3)
)

raw_audit.to_csv(
    DIRS["outputs"] / "01_raw_endpoint_audit.csv",
    index=False,
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6),
)

missing_plot = raw_audit.sort_values(
    "Missing %"
)

axes[0].barh(
    missing_plot["Endpoint"],
    missing_plot["Missing %"],
)

axes[0].axvline(
    15,
    linestyle="--",
    linewidth=1.2,
    label="15% reference",
)

axes[0].set_title(
    "Missing Labels per Endpoint",
    fontweight="bold",
)

axes[0].set_xlabel(
    "Missing labels (%)"
)

axes[0].legend()


positive_plot = raw_audit.sort_values(
    "Positive %"
)

axes[1].barh(
    positive_plot["Endpoint"],
    positive_plot["Positive %"],
)

axes[1].set_title(
    "Class Imbalance — Toxic Positive Rate",
    fontweight="bold",
)

axes[1].set_xlabel(
    "Toxic positive rate (%)"
)

plt.suptitle(
    "Tox21 Dataset Challenges",
    fontsize=16,
    fontweight="bold",
)

plt.tight_layout()

plt.savefig(
    DIRS["figures"] / "01_dataset_challenges.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()

## 9. Chemical standardization pipeline

The same standardized molecular identity is used by every representation.

**Pipeline**

`Original SMILES → RDKit parse → largest organic fragment → cleanup/normalize → uncharge → sanitize → canonical SMILES → InChIKey → Bemis–Murcko scaffold`

Additional quality controls:

- Invalid molecules are removed.
- Salts and multi-fragment records are reduced to the largest organic fragment.
- Charges are normalized where possible.
- Stereochemistry is preserved in canonical SMILES.
- Transformations are fully audited.

In [ ]:
fragment_chooser = (
    rdMolStandardize.LargestFragmentChooser(
        preferOrganic=True
    )
)

normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()


def standardize_smiles(
    smiles: str,
) -> Dict[str, Any]:

    result = {
        "original_smiles": smiles,
        "canonical_smiles": None,
        "inchikey": None,
        "scaffold": None,
        "status": "invalid",
        "reason": None,
    }

    try:
        molecule = Chem.MolFromSmiles(
            str(smiles),
            sanitize=True,
        )

        if molecule is None:
            result["reason"] = "MolFromSmiles failed"
            return result

        molecule = fragment_chooser.choose(
            molecule
        )

        molecule = normalizer.normalize(
            molecule
        )

        molecule = uncharger.uncharge(
            molecule
        )

        Chem.SanitizeMol(
            molecule
        )

        canonical_smiles = Chem.MolToSmiles(
            molecule,
            canonical=True,
            isomericSmiles=True,
        )

        inchikey = Chem.MolToInchiKey(
            molecule
        )

        scaffold = (
            MurckoScaffold.MurckoScaffoldSmiles(
                mol=molecule,
                includeChirality=True,
            )
        )

        if scaffold == "":
            scaffold = (
                "ACYCLIC::"
                + Chem.MolToSmiles(
                    molecule,
                    canonical=True,
                )
            )

        result.update({
            "canonical_smiles": canonical_smiles,
            "inchikey": inchikey,
            "scaffold": scaffold,
            "status": "valid",
            "reason": "",
        })

        return result

    except Exception as error:
        result["reason"] = str(error)[:180]
        return result

In [ ]:
print(
    "বাংলা বার্তা: RDKit chemical standardization শুরু হয়েছে..."
)

curation_records = [
    standardize_smiles(smiles)
    for smiles in tqdm(
        raw_df[SMILES_COL],
        desc="Chemical curation",
    )
]

curation_audit = pd.DataFrame(
    curation_records
)

curation_audit.insert(
    0,
    "original_row_id",
    np.arange(len(raw_df)),
)

work_df = pd.concat(
    [
        raw_df.reset_index(drop=True),
        curation_audit.drop(
            columns=["original_smiles"]
        ),
    ],
    axis=1,
)

invalid_df = work_df[
    work_df["status"] != "valid"
].copy()

valid_df = work_df[
    work_df["status"] == "valid"
].copy()

print(
    f"Valid molecules: {len(valid_df):,}"
)

print(
    f"Invalid molecules removed: {len(invalid_df):,}"
)

curation_audit.to_csv(
    DIRS["outputs"] / "02_chemical_curation_audit.csv",
    index=False,
)

invalid_df.to_csv(
    DIRS["outputs"] / "03_invalid_smiles_records.csv",
    index=False,
)

## 10. Duplicate merge and endpoint-specific conflict policy

For duplicate standardized molecules:

- Consistent endpoint labels are merged.
- If duplicate rows disagree for one endpoint, **only that endpoint** is marked missing.
- Valid labels for the other endpoints are preserved.

In [ ]:
grouped = valid_df.groupby(
    "canonical_smiles",
    sort=False,
)

curated_df = (
    valid_df
    .drop_duplicates(
        "canonical_smiles",
        keep="first",
    )
    .copy()
)

curated_df = curated_df.set_index(
    "canonical_smiles",
    drop=False,
)

source_ids = grouped[
    "original_row_id"
].agg(
    lambda values: ",".join(
        map(
            str,
            values.tolist(),
        )
    )
)

duplicate_counts = grouped.size()

curated_df[
    "source_row_ids"
] = source_ids.reindex(
    curated_df.index
).values

curated_df[
    "duplicate_count"
] = duplicate_counts.reindex(
    curated_df.index
).values

curated_df[
    "conflicting_endpoints"
] = 0


for target in TARGET_COLS:

    statistics_table = grouped[
        target
    ].agg(
        [
            "min",
            "max",
            "count",
        ]
    )

    merged_values = (
        statistics_table["min"]
        .astype(float)
    )

    no_observation = (
        statistics_table["count"] == 0
    )

    merged_values[
        no_observation
    ] = np.nan

    conflict = (
        (statistics_table["count"] > 0)
        & (
            statistics_table["min"]
            != statistics_table["max"]
        )
    )

    merged_values[
        conflict
    ] = np.nan

    curated_df[
        target
    ] = merged_values.reindex(
        curated_df.index
    ).values

    curated_df[
        "conflicting_endpoints"
    ] += (
        conflict
        .reindex(
            curated_df.index,
            fill_value=False,
        )
        .astype(int)
        .values
    )


curated_df = curated_df.reset_index(
    drop=True
)

print(
    f"Unique standardized molecules: "
    f"{len(curated_df):,}"
)

print(
    "Conflicting endpoint labels set to missing:",
    int(
        curated_df[
            "conflicting_endpoints"
        ].sum()
    ),
)

curated_audit = endpoint_audit(
    curated_df,
    TARGET_COLS,
)

display(
    curated_audit.round(3)
)

In [ ]:
CURATED_CSV = (
    DIRS["processed"]
    / "tox21_curated.csv"
)

CURATED_PARQUET = (
    DIRS["processed"]
    / "tox21_curated.parquet"
)

curated_df.to_csv(
    CURATED_CSV,
    index=False,
)

try:
    curated_df.to_parquet(
        CURATED_PARQUET,
        index=False,
    )
except Exception as error:
    print(
        "Parquet save skipped:",
        error,
    )

print(
    "Curated dataset saved:",
    CURATED_CSV,
)

## 11. Chemical-space audit

In [ ]:
CHEMICAL_SPACE_FUNCTIONS = {
    "MW": Descriptors.MolWt,
    "LogP": Crippen.MolLogP,
    "TPSA": rdMolDescriptors.CalcTPSA,
    "HBD": Lipinski.NumHDonors,
    "HBA": Lipinski.NumHAcceptors,
    "RotBonds": Lipinski.NumRotatableBonds,
    "Rings": Lipinski.RingCount,
    "AromaticRings": Lipinski.NumAromaticRings,
    "HeavyAtoms": Lipinski.HeavyAtomCount,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
}


def chemical_space_row(
    smiles: str,
) -> Dict[str, float]:

    molecule = Chem.MolFromSmiles(
        smiles
    )

    row = {
        name: float(function(molecule))
        for name, function
        in CHEMICAL_SPACE_FUNCTIONS.items()
    }

    row[
        "SMILES_Length"
    ] = len(smiles)

    return row


chemical_space = pd.DataFrame([
    chemical_space_row(smiles)
    for smiles in tqdm(
        curated_df["canonical_smiles"],
        desc="Chemical-space descriptors",
    )
])

chemical_space.to_csv(
    DIRS["outputs"] / "04_chemical_space.csv",
    index=False,
)

display(
    chemical_space
    .describe()
    .T
    .round(3)
)

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(16, 9),
)

selected_columns = [
    "MW",
    "LogP",
    "TPSA",
    "Rings",
    "SMILES_Length",
    "FractionCSP3",
]

for axis, column in zip(
    axes.ravel(),
    selected_columns,
):

    axis.hist(
        chemical_space[
            column
        ].dropna(),
        bins=35,
        alpha=0.85,
    )

    axis.set_title(
        column,
        fontweight="bold",
    )

    axis.set_ylabel(
        "Molecules"
    )


plt.suptitle(
    "Curated Tox21 Chemical Space",
    fontsize=16,
    fontweight="bold",
)

plt.tight_layout()

plt.savefig(
    DIRS["figures"] / "02_chemical_space.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()

# Part III — Frozen scaffold split, 5-fold CV and molecular representations

## 12. Frozen 75/25 scaffold split

The final test set is created before model development. Molecules with the same Bemis–Murcko scaffold remain in the same partition.

The split optimizer balances:

- total molecule count,
- endpoint-specific positive counts,
- endpoint-specific labeled counts,
- scaffold separation.

In [ ]:
def label_statistics_for_indices(
    label_matrix: np.ndarray,
    indices: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:

    selected = label_matrix[indices]

    observed = ~np.isnan(
        selected
    )

    positives = np.nansum(
        selected,
        axis=0,
    )

    labeled = observed.sum(
        axis=0,
    )

    return (
        positives.astype(float),
        labeled.astype(float),
    )


def scaffold_holdout_assignment(
    dataframe: pd.DataFrame,
    targets: List[str],
    test_fraction: float = 0.25,
    seed: int = 13,
    n_starts: int = 250,
) -> Tuple[np.ndarray, float]:

    labels = dataframe[
        targets
    ].to_numpy(
        dtype=float
    )

    groups = dataframe[
        "scaffold"
    ].astype(
        str
    ).values

    unique_groups = np.unique(
        groups
    )

    group_indices = {
        group: np.where(
            groups == group
        )[0]
        for group in unique_groups
    }

    total_positive, total_labeled = (
        label_statistics_for_indices(
            labels,
            np.arange(
                len(dataframe)
            ),
        )
    )

    target_size = (
        len(dataframe)
        * test_fraction
    )

    target_positive = (
        total_positive
        * test_fraction
    )

    target_labeled = (
        total_labeled
        * test_fraction
    )

    random_generator = (
        np.random.default_rng(
            seed
        )
    )

    best_result = None


    def assignment_cost(
        size,
        positive,
        labeled,
    ):

        size_cost = (
            abs(
                size - target_size
            )
            / max(
                target_size,
                1,
            )
        )

        positive_cost = np.mean(
            np.abs(
                positive
                - target_positive
            )
            / np.maximum(
                target_positive,
                3,
            )
        )

        labeled_cost = np.mean(
            np.abs(
                labeled
                - target_labeled
            )
            / np.maximum(
                target_labeled,
                10,
            )
        )

        overflow = (
            max(
                0,
                size
                - target_size * 1.08,
            )
            / max(
                target_size,
                1,
            )
        )

        return (
            0.45 * size_cost
            + 0.40 * positive_cost
            + 0.15 * labeled_cost
            + 2.0 * overflow
        )


    for _ in range(
        n_starts
    ):

        group_order = list(
            unique_groups
        )

        random_generator.shuffle(
            group_order
        )

        group_order.sort(
            key=lambda group: (
                len(
                    group_indices[
                        group
                    ]
                )
            ),
            reverse=True,
        )

        test_groups = []

        current_size = 0

        current_positive = np.zeros(
            len(targets)
        )

        current_labeled = np.zeros(
            len(targets)
        )


        for group in group_order:

            indices = group_indices[
                group
            ]

            group_positive, group_labeled = (
                label_statistics_for_indices(
                    labels,
                    indices,
                )
            )

            add_cost = assignment_cost(
                current_size
                + len(indices),
                current_positive
                + group_positive,
                current_labeled
                + group_labeled,
            )

            skip_cost = assignment_cost(
                current_size,
                current_positive,
                current_labeled,
            )

            if (
                add_cost <= skip_cost
                or current_size
                < target_size * 0.88
            ):

                test_groups.append(
                    group
                )

                current_size += len(
                    indices
                )

                current_positive += (
                    group_positive
                )

                current_labeled += (
                    group_labeled
                )


        final_cost = assignment_cost(
            current_size,
            current_positive,
            current_labeled,
        )

        if (
            best_result is None
            or final_cost
            < best_result[0]
        ):

            best_result = (
                final_cost,
                set(test_groups),
            )


    test_group_set = (
        best_result[1]
    )

    split = np.where(
        np.isin(
            groups,
            list(
                test_group_set
            ),
        ),
        "test",
        "train",
    )

    return (
        split,
        best_result[0],
    )

In [ ]:
SPLIT_FILE = (
    DIRS["splits"]
    / "scaffold_split_75_25_v2.csv"
)

if SPLIT_FILE.exists():

    print(
        "বাংলা বার্তা: আগে freeze করা scaffold split পাওয়া গেছে। "
        "সেটিই ব্যবহার করা হচ্ছে।"
    )

    split_manifest = pd.read_csv(
        SPLIT_FILE
    )

    curated_df = (
        curated_df
        .drop(
            columns=["split"],
            errors="ignore",
        )
        .merge(
            split_manifest[
                [
                    "canonical_smiles",
                    "split",
                ]
            ],
            on="canonical_smiles",
            how="left",
        )
    )

    if curated_df[
        "split"
    ].isna().any():

        raise ValueError(
            "Curated dataset ও split manifest mismatch. "
            "পুরনো split file version audit করুন।"
        )

else:

    print(
        "বাংলা বার্তা: নতুন leakage-safe 75/25 scaffold split "
        "তৈরি ও freeze করা হচ্ছে..."
    )

    split_labels, split_cost = (
        scaffold_holdout_assignment(
            curated_df,
            TARGET_COLS,
            test_fraction=(
                1 - TRAIN_FRACTION
            ),
            seed=PRIMARY_SEED,
            n_starts=(
                50
                if EXECUTION_MODE == "QUICK"
                else 350
            ),
        )
    )

    curated_df[
        "split"
    ] = split_labels

    split_manifest = curated_df[
        [
            "canonical_smiles",
            "inchikey",
            "scaffold",
            "split",
        ]
    ].copy()

    split_manifest.to_csv(
        SPLIT_FILE,
        index=False,
    )

    print(
        "Scaffold split freeze সম্পন্ন হয়েছে। "
        f"Assignment cost={split_cost:.4f}"
    )


train_idx = np.where(
    curated_df[
        "split"
    ].values == "train"
)[0]

test_idx = np.where(
    curated_df[
        "split"
    ].values == "test"
)[0]


print(
    "বাংলা বার্তা: Dataset ভাগ করা হয়েছে — "
    f"Train = {len(train_idx):,} "
    f"({len(train_idx)/len(curated_df):.1%}), "
    f"Test = {len(test_idx):,} "
    f"({len(test_idx)/len(curated_df):.1%})"
)

## 13. Split integrity and endpoint coverage checks

In [ ]:
train_scaffolds = set(
    curated_df.loc[
        train_idx,
        "scaffold",
    ]
)

test_scaffolds = set(
    curated_df.loc[
        test_idx,
        "scaffold",
    ]
)

train_inchikeys = set(
    curated_df.loc[
        train_idx,
        "inchikey",
    ]
)

test_inchikeys = set(
    curated_df.loc[
        test_idx,
        "inchikey",
    ]
)

assert train_scaffolds.isdisjoint(
    test_scaffolds
), "Scaffold leakage detected!"

assert train_inchikeys.isdisjoint(
    test_inchikeys
), "Molecular identity leakage detected!"


split_quality = pd.concat([
    endpoint_audit(
        curated_df.iloc[
            train_idx
        ],
        TARGET_COLS,
    ).assign(
        Split="Train"
    ),

    endpoint_audit(
        curated_df.iloc[
            test_idx
        ],
        TARGET_COLS,
    ).assign(
        Split="Test"
    ),
])


display(
    split_quality[
        [
            "Split",
            "Endpoint",
            "Labeled",
            "Toxic (1)",
            "Non-Toxic (0)",
            "Missing",
            "Positive %",
        ]
    ].round(2)
)

print(
    "Leakage check passed: "
    "zero molecule and scaffold overlap."
)

## 14. Scaffold-aware 5-fold cross-validation inside the training pool

All five models use the **same five folds**.

The final test set is never used for:

- hyperparameter selection,
- early stopping,
- feature fitting,
- class weighting,
- threshold selection,
- ensemble weight learning.

In [ ]:
def balanced_scaffold_folds(
    dataframe: pd.DataFrame,
    targets: List[str],
    n_splits: int = 5,
    seed: int = 13,
    n_starts: int = 250,
) -> Tuple[np.ndarray, float]:

    labels = dataframe[
        targets
    ].to_numpy(
        dtype=float
    )

    groups = dataframe[
        "scaffold"
    ].astype(
        str
    ).values

    unique_groups = np.unique(
        groups
    )

    group_indices = {
        group: np.where(
            groups == group
        )[0]
        for group in unique_groups
    }

    group_positive = {}
    group_labeled = {}

    for group, indices in group_indices.items():

        (
            group_positive[group],
            group_labeled[group],
        ) = label_statistics_for_indices(
            labels,
            indices,
        )


    total_positive, total_labeled = (
        label_statistics_for_indices(
            labels,
            np.arange(
                len(dataframe)
            ),
        )
    )

    target_size = (
        len(dataframe)
        / n_splits
    )

    target_positive = (
        total_positive
        / n_splits
    )

    target_labeled = (
        total_labeled
        / n_splits
    )

    random_generator = (
        np.random.default_rng(
            seed
        )
    )

    best_result = None


    def global_cost(
        sizes,
        positives,
        labeled,
    ):

        size_term = np.std(
            sizes
            / max(
                target_size,
                1,
            )
        )

        positive_term = np.mean(
            np.std(
                positives
                / np.maximum(
                    target_positive,
                    3,
                ),
                axis=0,
            )
        )

        labeled_term = np.mean(
            np.std(
                labeled
                / np.maximum(
                    target_labeled,
                    10,
                ),
                axis=0,
            )
        )

        empty_penalty = (
            5.0
            * np.sum(
                sizes == 0
            )
        )

        return (
            0.45 * size_term
            + 0.40 * positive_term
            + 0.15 * labeled_term
            + empty_penalty
        )


    for _ in range(
        n_starts
    ):

        group_order = list(
            unique_groups
        )

        random_generator.shuffle(
            group_order
        )

        group_order.sort(
            key=lambda group: (
                len(
                    group_indices[
                        group
                    ]
                )
                + 0.25
                * group_positive[
                    group
                ].sum()
                + random_generator.random()
                * 1e-3
            ),
            reverse=True,
        )

        fold_groups = [
            set()
            for _ in range(
                n_splits
            )
        ]

        sizes = np.zeros(
            n_splits,
            dtype=float,
        )

        positives = np.zeros(
            (
                n_splits,
                len(targets),
            ),
            dtype=float,
        )

        labeled = np.zeros(
            (
                n_splits,
                len(targets),
            ),
            dtype=float,
        )


        for fold, group in enumerate(
            group_order[
                :n_splits
            ]
        ):

            fold_groups[
                fold
            ].add(
                group
            )

            sizes[
                fold
            ] += len(
                group_indices[
                    group
                ]
            )

            positives[
                fold
            ] += (
                group_positive[
                    group
                ]
            )

            labeled[
                fold
            ] += (
                group_labeled[
                    group
                ]
            )


        for group in group_order[
            n_splits:
        ]:

            candidate_costs = []

            for fold in range(
                n_splits
            ):

                trial_sizes = (
                    sizes.copy()
                )

                trial_positives = (
                    positives.copy()
                )

                trial_labeled = (
                    labeled.copy()
                )

                trial_sizes[
                    fold
                ] += len(
                    group_indices[
                        group
                    ]
                )

                trial_positives[
                    fold
                ] += (
                    group_positive[
                        group
                    ]
                )

                trial_labeled[
                    fold
                ] += (
                    group_labeled[
                        group
                    ]
                )

                overflow = (
                    max(
                        0.0,
                        trial_sizes[
                            fold
                        ]
                        - 1.18
                        * target_size,
                    )
                    / max(
                        target_size,
                        1,
                    )
                )

                candidate_costs.append(
                    global_cost(
                        trial_sizes,
                        trial_positives,
                        trial_labeled,
                    )
                    + 0.5
                    * overflow
                )


            minimum_cost = min(
                candidate_costs
            )

            candidate_folds = [
                fold
                for fold, cost
                in enumerate(
                    candidate_costs
                )
                if abs(
                    cost
                    - minimum_cost
                ) < 1e-12
            ]

            selected_fold = int(
                random_generator.choice(
                    candidate_folds
                )
            )

            fold_groups[
                selected_fold
            ].add(
                group
            )

            sizes[
                selected_fold
            ] += len(
                group_indices[
                    group
                ]
            )

            positives[
                selected_fold
            ] += (
                group_positive[
                    group
                ]
            )

            labeled[
                selected_fold
            ] += (
                group_labeled[
                    group
                ]
            )


        cost = global_cost(
            sizes,
            positives,
            labeled,
        )

        if (
            best_result is None
            or cost < best_result[0]
        ):

            best_result = (
                cost,
                fold_groups,
                sizes.copy(),
            )


    fold_ids = np.full(
        len(dataframe),
        -1,
        dtype=int,
    )

    for fold, groups_in_fold in enumerate(
        best_result[1]
    ):

        fold_ids[
            np.isin(
                groups,
                list(
                    groups_in_fold
                ),
            )
        ] = fold


    if (
        (fold_ids < 0).any()
        or len(
            np.unique(
                fold_ids
            )
        )
        != n_splits
    ):

        raise RuntimeError(
            "Balanced scaffold fold assignment failed."
        )


    return (
        fold_ids,
        best_result[0],
    )

In [ ]:
train_df = (
    curated_df
    .iloc[
        train_idx
    ]
    .reset_index()
    .rename(
        columns={
            "index": "global_index"
        }
    )
)

CV_FILE = (
    DIRS["splits"]
    / "scaffold_cv5_v2.csv"
)

regenerate_cv = False

if CV_FILE.exists():

    cv_manifest = pd.read_csv(
        CV_FILE
    )

    train_df = train_df.merge(
        cv_manifest[
            [
                "canonical_smiles",
                "cv_fold",
            ]
        ],
        on="canonical_smiles",
        how="left",
    )

    if (
        train_df[
            "cv_fold"
        ].nunique()
        != N_CV_FOLDS
        or train_df[
            "cv_fold"
        ].isna().any()
    ):

        regenerate_cv = True

else:

    regenerate_cv = True


if regenerate_cv:

    print(
        "বাংলা বার্তা: নতুন balanced scaffold-aware "
        "5-fold CV manifest তৈরি হচ্ছে..."
    )

    train_df = train_df.drop(
        columns=["cv_fold"],
        errors="ignore",
    )

    fold_ids, fold_cost = (
        balanced_scaffold_folds(
            train_df,
            TARGET_COLS,
            n_splits=N_CV_FOLDS,
            seed=PRIMARY_SEED,
            n_starts=(
                50
                if EXECUTION_MODE == "QUICK"
                else 300
            ),
        )
    )

    train_df[
        "cv_fold"
    ] = fold_ids

    train_df[
        [
            "canonical_smiles",
            "inchikey",
            "scaffold",
            "cv_fold",
        ]
    ].to_csv(
        CV_FILE,
        index=False,
    )

    print(
        f"5-fold manifest saved. "
        f"Assignment cost={fold_cost:.4f}"
    )


assert (
    train_df[
        "cv_fold"
    ].notna().all()
)

assert (
    train_df[
        "cv_fold"
    ].nunique()
    == N_CV_FOLDS
)


fold_rows = []

for fold in range(
    N_CV_FOLDS
):

    validation_fold = (
        train_df[
            train_df[
                "cv_fold"
            ] == fold
        ]
    )

    fold_rows.append({
        "Fold": fold + 1,
        "Molecules": len(
            validation_fold
        ),
        "Scaffolds": (
            validation_fold[
                "scaffold"
            ].nunique()
        ),
    })


fold_summary = pd.DataFrame(
    fold_rows
)

display(
    fold_summary
)

### 14.1 Fold-level label support audit

Every validation fold should contain both positive and negative examples for each endpoint. This audit is performed before model training.

In [ ]:
fold_support_rows = []

for fold in range(
    N_CV_FOLDS
):

    validation_rows = (
        train_df[
            "cv_fold"
        ].values
        == fold
    )

    validation_global = (
        train_df.loc[
            validation_rows,
            "global_index",
        ].to_numpy(
            dtype=int
        )
    )

    for task, endpoint in enumerate(
        TARGET_COLS
    ):

        observed = (
            M[
                validation_global,
                task
            ]
            .astype(
                bool
            )
        )

        labels = (
            Y_FILLED[
                validation_global,
                task
            ][
                observed
            ]
            .astype(
                int
            )
        )

        n_positive = int(
            labels.sum()
        )

        n_negative = int(
            len(
                labels
            )
            - n_positive
        )

        fold_support_rows.append({
            "Fold": fold + 1,
            "Endpoint": endpoint,
            "Labeled": len(
                labels
            ),
            "Positive": n_positive,
            "Negative": n_negative,
            "Both classes": (
                n_positive > 0
                and n_negative > 0
            ),
        })


FOLD_SUPPORT = pd.DataFrame(
    fold_support_rows
)


display(
    FOLD_SUPPORT.pivot(
        index="Endpoint",
        columns="Fold",
        values="Positive",
    )
    .rename_axis(
        columns="Validation fold"
    )
)


missing_class_rows = (
    FOLD_SUPPORT[
        ~FOLD_SUPPORT[
            "Both classes"
        ]
    ]
)


if len(
    missing_class_rows
):

    print(
        "বাংলা সতর্কতা: কিছু fold-endpoint-এ উভয় class নেই। "
        "AUC ওই fold-task-এর জন্য undefined হবে। "
        "Scaffold separation ভাঙা হবে না; support table report করুন।"
    )

    display(
        missing_class_rows
    )

else:

    print(
        "বাংলা বার্তা: সব validation fold-এর সব endpoint-এ "
        "positive ও negative দুই class-ই আছে।"
    )


FOLD_SUPPORT.to_csv(
    DIRS["outputs"]
    / "fold_endpoint_support.csv",
    index=False,
)

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    fold_summary[
        "Fold"
    ].astype(str),
    fold_summary[
        "Molecules"
    ],
)

plt.title(
    "Scaffold-Aware 5-Fold CV — Fold Sizes",
    fontweight="bold",
)

plt.xlabel(
    "Fold"
)

plt.ylabel(
    "Molecules"
)

plt.tight_layout()

plt.savefig(
    DIRS["figures"]
    / "03_cv_fold_sizes.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()

## 15. Per-task labels and missing-label masks

In [ ]:
Y = curated_df[
    TARGET_COLS
].to_numpy(
    dtype=np.float32
)

M = (
    ~np.isnan(
        Y
    )
).astype(
    np.float32
)

Y_FILLED = np.nan_to_num(
    Y,
    nan=0.0,
).astype(
    np.float32
)

assert np.all(
    (Y_FILLED == 0)
    | (Y_FILLED == 1)
)

assert np.all(
    (M == 0)
    | (M == 1)
)

print(
    "বাংলা বার্তা: Per-task mask তৈরি হয়েছে। "
    "Missing label কোনো অবস্থাতেই 0/1 হিসেবে impute করা হয়নি।"
)

print(
    f"Observed label cells: {int(M.sum()):,}"
)

print(
    f"Missing label cells: {int((1-M).sum()):,}"
)

## 16. High-performance multi-fingerprint and descriptor representation

The structural branch combines six complementary fingerprint views:

- chirality-aware ECFP4 bits,
- chirality-aware Morgan count fingerprint,
- MACCS keys,
- hashed Atom-Pair fingerprint,
- RDKit path fingerprint,
- hashed topological torsion fingerprint.

The previous 21-descriptor panel is expanded with the stable RDKit 2D descriptor registry. Failed, infinite or undefined values are preserved as `NaN` and handled **inside each training fold only**.

Fold-local preprocessing order:

1. fingerprint variance filtering,
2. TruncatedSVD,
3. SVD-component standardization,
4. descriptor train-only winsorization,
5. median imputation,
6. robust scaling,
7. descriptor variance filtering,
8. PCA retaining 99.5% variance.

No imputer, clipping boundary, scaler, variance selector, SVD or PCA is fitted on validation/test data.


In [ ]:
FP_SIZES = {
    "ecfp4": 2048,
    "morgan_count": 2048,
    "maccs": 167,
    "atom_pair": 2048,
    "rdkit": 2048,
    "torsion": 2048,
}


# Curated core descriptors are kept explicitly.
CORE_DESCRIPTOR_FUNCTIONS = {
    "MolWt": Descriptors.MolWt,
    "MolLogP": Crippen.MolLogP,
    "TPSA": rdMolDescriptors.CalcTPSA,
    "HBD": Lipinski.NumHDonors,
    "HBA": Lipinski.NumHAcceptors,
    "RotatableBonds": Lipinski.NumRotatableBonds,
    "RingCount": Lipinski.RingCount,
    "AromaticRings": Lipinski.NumAromaticRings,
    "AliphaticRings": Lipinski.NumAliphaticRings,
    "HeavyAtoms": Lipinski.HeavyAtomCount,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
    "FormalCharge": Chem.GetFormalCharge,
    "NumHeteroatoms": Lipinski.NumHeteroatoms,
    "NumValenceElectrons": Descriptors.NumValenceElectrons,
    "MolMR": Crippen.MolMR,
    "LabuteASA": rdMolDescriptors.CalcLabuteASA,
    "NHOHCount": Lipinski.NHOHCount,
    "NOCount": Lipinski.NOCount,
    "NumRadicalElectrons": Descriptors.NumRadicalElectrons,
    "BalabanJ": Descriptors.BalabanJ,
    "BertzCT": Descriptors.BertzCT,
}


# Add the complete stable RDKit 2D descriptor registry.
# Duplicate names are automatically ignored.
DESCRIPTOR_FUNCTIONS = dict(CORE_DESCRIPTOR_FUNCTIONS)

for descriptor_name, descriptor_function in Descriptors._descList:
    if descriptor_name not in DESCRIPTOR_FUNCTIONS:
        DESCRIPTOR_FUNCTIONS[descriptor_name] = descriptor_function


DESCRIPTOR_NAMES = list(DESCRIPTOR_FUNCTIONS)


def safe_descriptor_value(function, molecule) -> float:
    try:
        value = float(function(molecule))
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan


def bit_vector_to_array(
    fingerprint,
    n_bits: int,
) -> np.ndarray:

    array = np.zeros(
        n_bits,
        dtype=np.float32,
    )

    DataStructs.ConvertToNumpyArray(
        fingerprint,
        array,
    )

    return array


def sparse_count_to_hashed_array(
    sparse_vector,
    n_bits: int,
) -> np.ndarray:

    array = np.zeros(
        n_bits,
        dtype=np.float32,
    )

    for index, value in (
        sparse_vector
        .GetNonzeroElements()
        .items()
    ):

        array[
            index % n_bits
        ] += float(
            value
        )

    return array


def molecule_features(
    smiles: str,
) -> Tuple[np.ndarray, np.ndarray]:

    molecule = Chem.MolFromSmiles(
        smiles
    )

    ecfp4 = (
        AllChem
        .GetMorganFingerprintAsBitVect(
            molecule,
            radius=2,
            nBits=FP_SIZES[
                "ecfp4"
            ],
            useChirality=True,
        )
    )

    morgan_count = (
        AllChem
        .GetMorganFingerprint(
            molecule,
            radius=2,
            useChirality=True,
        )
    )

    maccs = (
        MACCSkeys
        .GenMACCSKeys(
            molecule
        )
    )

    atom_pair = (
        rdMolDescriptors
        .GetHashedAtomPairFingerprintAsBitVect(
            molecule,
            nBits=FP_SIZES[
                "atom_pair"
            ],
        )
    )

    rdkit_fp = (
        Chem
        .RDKFingerprint(
            molecule,
            fpSize=FP_SIZES[
                "rdkit"
            ],
            minPath=1,
            maxPath=7,
        )
    )

    torsion = (
        rdMolDescriptors
        .GetHashedTopologicalTorsionFingerprintAsBitVect(
            molecule,
            nBits=FP_SIZES[
                "torsion"
            ],
        )
    )

    fingerprint_vector = np.concatenate([
        bit_vector_to_array(
            ecfp4,
            FP_SIZES[
                "ecfp4"
            ],
        ),

        sparse_count_to_hashed_array(
            morgan_count,
            FP_SIZES[
                "morgan_count"
            ],
        ),

        bit_vector_to_array(
            maccs,
            FP_SIZES[
                "maccs"
            ],
        ),

        bit_vector_to_array(
            atom_pair,
            FP_SIZES[
                "atom_pair"
            ],
        ),

        bit_vector_to_array(
            rdkit_fp,
            FP_SIZES[
                "rdkit"
            ],
        ),

        bit_vector_to_array(
            torsion,
            FP_SIZES[
                "torsion"
            ],
        ),
    ]).astype(
        np.float32
    )

    descriptor_vector = np.array([
        safe_descriptor_value(
            function,
            molecule,
        )
        for function
        in DESCRIPTOR_FUNCTIONS.values()
    ], dtype=np.float32)

    return (
        fingerprint_vector,
        descriptor_vector,
    )


print(
    "Structural descriptor count:",
    len(DESCRIPTOR_NAMES),
)


In [ ]:
FEATURE_CACHE = (
    DIRS["cache"]
    / "multifingerprint_descriptors_v3.npz"
)

if FEATURE_CACHE.exists():

    print(
        "বাংলা বার্তা: Cached fingerprint/descriptor "
        "features load করা হচ্ছে..."
    )

    feature_cache = np.load(
        FEATURE_CACHE
    )

    X_FP = feature_cache[
        "X_FP"
    ]

    X_DESC = feature_cache[
        "X_DESC"
    ]

else:

    print(
        "বাংলা বার্তা: Multiple fingerprints ও "
        "molecular descriptors generate করা হচ্ছে..."
    )

    fingerprint_rows = []
    descriptor_rows = []

    for smiles in tqdm(
        curated_df[
            "canonical_smiles"
        ],
        desc="Molecular features",
    ):

        (
            fingerprint_vector,
            descriptor_vector,
        ) = molecule_features(
            smiles
        )

        fingerprint_rows.append(
            fingerprint_vector
        )

        descriptor_rows.append(
            descriptor_vector
        )


    X_FP = np.stack(
        fingerprint_rows
    ).astype(
        np.float32
    )

    X_DESC = np.stack(
        descriptor_rows
    ).astype(
        np.float32
    )


    np.savez_compressed(
        FEATURE_CACHE,
        X_FP=X_FP,
        X_DESC=X_DESC,
        descriptor_names=np.asarray(DESCRIPTOR_NAMES, dtype=str),
    )

    print(
        "বাংলা বার্তা: Feature generation complete; "
        "cache save হয়েছে।"
    )


print(
    "Fingerprint matrix:",
    X_FP.shape,
)

print(
    "Descriptor matrix:",
    X_DESC.shape,
)

In [ ]:
class FoldLocalFeatureProcessor:
    """
    Every transformation is fitted on fold-train only.

    Improvements in revision 2:
    - train-only winsorization for unstable descriptor tails,
    - variance filtering for both feature families,
    - standardization of SVD components,
    - NaN/inf-safe transformation.
    """

    def __init__(
        self,
        max_svd: int = 512,
        descriptor_pca_variance: float = 0.995,
        descriptor_clip_quantiles=(0.005, 0.995),
        random_state: int = 13,
    ):

        self.max_svd = max_svd
        self.descriptor_pca_variance = descriptor_pca_variance
        self.descriptor_clip_quantiles = descriptor_clip_quantiles
        self.random_state = random_state

        self.fingerprint_variance = VarianceThreshold(
            threshold=0.0
        )
        self.fingerprint_svd = None
        self.fingerprint_scaler = StandardScaler()

        self.descriptor_lower = None
        self.descriptor_upper = None
        self.descriptor_imputer = SimpleImputer(
            strategy="median"
        )
        self.descriptor_scaler = RobustScaler(
            quantile_range=(5.0, 95.0)
        )
        self.descriptor_variance = VarianceThreshold(
            threshold=1e-12
        )
        self.descriptor_pca = None


    def _clean_descriptor_array(
        self,
        descriptors,
    ):

        array = np.asarray(
            descriptors,
            dtype=np.float64,
        ).copy()

        array[
            ~np.isfinite(
                array
            )
        ] = np.nan

        return array


    def _clip_descriptors(
        self,
        descriptors,
    ):

        return np.minimum(
            np.maximum(
                descriptors,
                self.descriptor_lower,
            ),
            self.descriptor_upper,
        )


    def fit(
        self,
        fingerprints,
        descriptors,
    ):

        filtered_fingerprints = (
            self.fingerprint_variance
            .fit_transform(
                fingerprints
            )
        )

        n_components = min(
            self.max_svd,
            max(
                2,
                filtered_fingerprints.shape[0] - 1,
            ),
            max(
                2,
                filtered_fingerprints.shape[1] - 1,
            ),
        )

        self.fingerprint_svd = TruncatedSVD(
            n_components=n_components,
            n_iter=7,
            random_state=self.random_state,
        )

        fingerprint_embedding = (
            self.fingerprint_svd
            .fit_transform(
                filtered_fingerprints
            )
        )

        self.fingerprint_scaler.fit(
            fingerprint_embedding
        )


        clean_descriptors = (
            self._clean_descriptor_array(
                descriptors
            )
        )

        q_low, q_high = (
            self.descriptor_clip_quantiles
        )

        with np.errstate(
            all="ignore"
        ):

            self.descriptor_lower = (
                np.nanquantile(
                    clean_descriptors,
                    q_low,
                    axis=0,
                )
            )

            self.descriptor_upper = (
                np.nanquantile(
                    clean_descriptors,
                    q_high,
                    axis=0,
                )
            )


        self.descriptor_lower = np.where(
            np.isfinite(
                self.descriptor_lower
            ),
            self.descriptor_lower,
            -np.inf,
        )

        self.descriptor_upper = np.where(
            np.isfinite(
                self.descriptor_upper
            ),
            self.descriptor_upper,
            np.inf,
        )


        clipped_descriptors = (
            self._clip_descriptors(
                clean_descriptors
            )
        )

        processed_descriptors = (
            self.descriptor_imputer
            .fit_transform(
                clipped_descriptors
            )
        )

        processed_descriptors = (
            self.descriptor_scaler
            .fit_transform(
                processed_descriptors
            )
        )

        processed_descriptors = (
            self.descriptor_variance
            .fit_transform(
                processed_descriptors
            )
        )


        n_descriptor_components = min(
            processed_descriptors.shape[0] - 1,
            processed_descriptors.shape[1],
        )


        if n_descriptor_components >= 2:

            self.descriptor_pca = PCA(
                n_components=(
                    self
                    .descriptor_pca_variance
                ),
                svd_solver="full",
                random_state=(
                    self.random_state
                ),
            )

            self.descriptor_pca.fit(
                processed_descriptors
            )

        else:

            self.descriptor_pca = None


        return self


    def transform(
        self,
        fingerprints,
        descriptors,
    ):

        fingerprint_embedding = (
            self.fingerprint_svd
            .transform(
                self
                .fingerprint_variance
                .transform(
                    fingerprints
                )
            )
        )

        fingerprint_embedding = (
            self.fingerprint_scaler
            .transform(
                fingerprint_embedding
            )
        )


        clean_descriptors = (
            self._clean_descriptor_array(
                descriptors
            )
        )

        descriptor_embedding = (
            self._clip_descriptors(
                clean_descriptors
            )
        )

        descriptor_embedding = (
            self.descriptor_imputer
            .transform(
                descriptor_embedding
            )
        )

        descriptor_embedding = (
            self.descriptor_scaler
            .transform(
                descriptor_embedding
            )
        )

        descriptor_embedding = (
            self.descriptor_variance
            .transform(
                descriptor_embedding
            )
        )


        if self.descriptor_pca is not None:

            descriptor_embedding = (
                self.descriptor_pca
                .transform(
                    descriptor_embedding
                )
            )


        combined = np.concatenate(
            [
                fingerprint_embedding,
                descriptor_embedding,
            ],
            axis=1,
        )

        combined = np.nan_to_num(
            combined,
            nan=0.0,
            posinf=8.0,
            neginf=-8.0,
        )

        return combined.astype(
            np.float32
        )


    def fit_transform(
        self,
        fingerprints,
        descriptors,
    ):

        return (
            self
            .fit(
                fingerprints,
                descriptors,
            )
            .transform(
                fingerprints,
                descriptors,
            )
        )


## 17. Molecular graph representation

In [ ]:
try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader as PyGDataLoader
    from torch_geometric.nn.models import AttentiveFP
    from torch_geometric.nn import (
        GPSConv,
        GINEConv,
        RGCNConv,
    )

    PYG_AVAILABLE = True
    print("PyTorch Geometric ready.")

except Exception as error:

    PYG_AVAILABLE = False

    print(
        "PyTorch Geometric import হয়নি:",
        error,
    )

In [ ]:
ATOM_NUMBERS = [
    1, 5, 6, 7, 8, 9,
    14, 15, 16, 17, 35, 53,
]

HYBRIDIZATIONS = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]


def one_hot_with_unknown(
    value,
    choices,
) -> List[float]:

    return [
        float(
            value == choice
        )
        for choice in choices
    ] + [
        float(
            value not in choices
        )
    ]


def atom_features(
    atom,
) -> np.ndarray:

    return np.array(

        one_hot_with_unknown(
            atom.GetAtomicNum(),
            ATOM_NUMBERS,
        )

        + one_hot_with_unknown(
            atom.GetDegree(),
            [0, 1, 2, 3, 4, 5],
        )

        + one_hot_with_unknown(
            atom.GetFormalCharge(),
            [-2, -1, 0, 1, 2],
        )

        + one_hot_with_unknown(
            atom.GetHybridization(),
            HYBRIDIZATIONS,
        )

        + [
            float(
                atom.GetIsAromatic()
            ),

            float(
                atom.HasProp(
                    "_ChiralityPossible"
                )
            ),

            float(
                atom.GetTotalNumHs()
            ) / 4.0,

            float(
                atom.GetTotalValence()
            ) / 8.0,

            float(
                atom.IsInRing()
            ),

            float(
                atom.GetMass()
            ) / 200.0,

            float(
                atom.GetAtomicNum()
                in {
                    7, 8, 9, 15, 16,
                }
            ),
        ],

        dtype=np.float32,
    )


def bond_features(
    bond,
) -> np.ndarray:

    return np.array(

        one_hot_with_unknown(
            bond.GetBondType(),
            BOND_TYPES,
        )

        + [
            float(
                bond.GetBondTypeAsDouble()
            ) / 3.0,

            float(
                bond.GetIsConjugated()
            ),

            float(
                bond.IsInRing()
            ),
        ]

        + one_hot_with_unknown(
            bond.GetStereo(),
            [
                Chem.rdchem
                .BondStereo
                .STEREONONE,

                Chem.rdchem
                .BondStereo
                .STEREOZ,

                Chem.rdchem
                .BondStereo
                .STEREOE,
            ],
        ),

        dtype=np.float32,
    )


_BOND_FEATURE_DIM = len(
    bond_features(
        Chem
        .MolFromSmiles("CC")
        .GetBondWithIdx(0)
    )
)


def smiles_to_pyg(
    smiles: str,
    y_row=None,
    mask_row=None,
    index=None,
):

    molecule = Chem.MolFromSmiles(
        smiles
    )

    node_matrix = np.stack([
        atom_features(atom)
        for atom
        in molecule.GetAtoms()
    ])

    x = torch.tensor(
        node_matrix,
        dtype=torch.float32,
    )

    edges = []
    edge_attributes = []

    for bond in molecule.GetBonds():

        source = bond.GetBeginAtomIdx()
        target = bond.GetEndAtomIdx()

        features = bond_features(
            bond
        )

        edges.extend([
            [source, target],
            [target, source],
        ])

        edge_attributes.extend([
            features,
            features,
        ])


    if edges:

        edge_index = torch.tensor(
            edges,
            dtype=torch.long,
        ).t().contiguous()

        edge_attr = torch.tensor(
            np.stack(
                edge_attributes
            ),
            dtype=torch.float32,
        )

    else:

        edge_index = torch.zeros(
            (2, 0),
            dtype=torch.long,
        )

        edge_attr = torch.zeros(
            (
                0,
                _BOND_FEATURE_DIM,
            ),
            dtype=torch.float32,
        )


    graph = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
    )

    if y_row is not None:

        graph.y = torch.tensor(
            y_row,
            dtype=torch.float32,
        ).view(
            1,
            -1,
        )

        graph.mask = torch.tensor(
            mask_row,
            dtype=torch.float32,
        ).view(
            1,
            -1,
        )


    if index is not None:

        graph.sample_idx = (
            torch.tensor(
                [index],
                dtype=torch.long,
            )
        )


    return graph

In [ ]:
GRAPH_CACHE = (
    DIRS["cache"]
    / "molecular_graphs_v2.pt"
)

MOLECULAR_GRAPHS = None

if PYG_AVAILABLE:

    if GRAPH_CACHE.exists():

        print(
            "বাংলা বার্তা: Cached molecular graphs load করা হচ্ছে..."
        )

        MOLECULAR_GRAPHS = torch.load(
            GRAPH_CACHE,
            weights_only=False,
        )

    else:

        print(
            "বাংলা বার্তা: Canonical SMILES থেকে "
            "molecular graph তৈরি করা হচ্ছে..."
        )

        MOLECULAR_GRAPHS = [
            smiles_to_pyg(
                smiles,
                Y_FILLED[index],
                M[index],
                index,
            )
            for index, smiles
            in enumerate(
                tqdm(
                    curated_df[
                        "canonical_smiles"
                    ],
                    desc="Molecular graphs",
                )
            )
        ]

        torch.save(
            MOLECULAR_GRAPHS,
            GRAPH_CACHE,
        )

        print(
            "বাংলা বার্তা: Molecular graph cache save হয়েছে।"
        )

else:

    print(
        "PyG unavailable; graph-based models cannot run "
        "until the environment is fixed."
    )

## 18. Training-only randomized SMILES augmentation

In [ ]:
def randomized_smiles(
    canonical_smiles: str,
    n_views: int = 1,
    seed: int = 13,
) -> List[str]:

    molecule = Chem.MolFromSmiles(
        canonical_smiles
    )

    if molecule is None:

        return [
            canonical_smiles
        ] * n_views


    previous_state = random.getstate()

    random.seed(
        seed
    )

    views = []

    for _ in range(
        n_views
    ):

        views.append(
            Chem.MolToSmiles(
                molecule,
                canonical=False,
                doRandom=True,
                isomericSmiles=True,
            )
        )


    random.setstate(
        previous_state
    )

    return views


example_smiles = (
    curated_df[
        "canonical_smiles"
    ].iloc[0]
)

print(
    "Canonical:",
    example_smiles,
)

print(
    "Training views:",
    randomized_smiles(
        example_smiles,
        n_views=3,
        seed=PRIMARY_SEED,
    ),
)

## 19. Evaluation, threshold and statistical utilities

In [ ]:
def safe_auc(
    y_true,
    probability,
) -> float:

    y_true = np.asarray(
        y_true
    )

    probability = np.asarray(
        probability
    )

    if (
        len(
            np.unique(
                y_true
            )
        )
        != 2
    ):

        return np.nan

    return float(
        roc_auc_score(
            y_true,
            probability,
        )
    )


def safe_average_precision(
    y_true,
    probability,
) -> float:

    y_true = np.asarray(
        y_true
    )

    probability = np.asarray(
        probability
    )

    if (
        len(
            np.unique(
                y_true
            )
        )
        != 2
    ):

        return np.nan

    return float(
        average_precision_score(
            y_true,
            probability,
        )
    )


def sensitivity_score(
    y_true,
    prediction,
) -> float:

    matrix = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    )

    tn, fp, fn, tp = (
        matrix.ravel()
    )

    return (
        tp
        / max(
            tp + fn,
            1,
        )
    )


def specificity_score(
    y_true,
    prediction,
) -> float:

    matrix = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    )

    tn, fp, fn, tp = (
        matrix.ravel()
    )

    return (
        tn
        / max(
            tn + fp,
            1,
        )
    )


def optimize_threshold(
    y_true,
    probability,
    criterion: str = "balanced_accuracy",
) -> float:

    candidate_thresholds = np.unique(
        np.r_[
            np.linspace(
                0.02,
                0.98,
                97,
            ),
            probability,
        ]
    )

    best_threshold = 0.5
    best_score = -np.inf

    for threshold in candidate_thresholds:

        prediction = (
            probability
            >= threshold
        ).astype(
            int
        )

        if criterion == "accuracy":

            score = accuracy_score(
                y_true,
                prediction,
            )

        elif criterion == "youden":

            score = (
                sensitivity_score(
                    y_true,
                    prediction,
                )
                + specificity_score(
                    y_true,
                    prediction,
                )
                - 1
            )

        else:

            score = (
                balanced_accuracy_score(
                    y_true,
                    prediction,
                )
            )


        if score > best_score:

            best_score = score
            best_threshold = float(
                threshold
            )


    return best_threshold


def mean_masked_auc(
    y_true,
    mask,
    probability,
) -> float:

    endpoint_scores = []

    for task in range(
        y_true.shape[1]
    ):

        valid = (
            mask[
                :,
                task
            ]
            .astype(
                bool
            )
        )

        if (
            valid.sum() > 1
            and len(
                np.unique(
                    y_true[
                        valid,
                        task
                    ]
                )
            )
            == 2
        ):

            endpoint_scores.append(
                roc_auc_score(
                    y_true[
                        valid,
                        task
                    ],
                    probability[
                        valid,
                        task
                    ],
                )
            )


    return (
        float(
            np.mean(
                endpoint_scores
            )
        )
        if endpoint_scores
        else np.nan
    )


def evaluate_multitask(
    y_true,
    mask,
    probability,
    thresholds=None,
    endpoints=None,
) -> pd.DataFrame:

    endpoints = (
        endpoints
        or [
            f"Task_{index}"
            for index
            in range(
                y_true.shape[1]
            )
        ]
    )

    if thresholds is None:

        thresholds = np.full(
            y_true.shape[1],
            0.5,
        )


    rows = []

    for task, endpoint in enumerate(
        endpoints
    ):

        valid = (
            mask[
                :,
                task
            ]
            .astype(
                bool
            )
        )

        y_task = (
            y_true[
                valid,
                task
            ]
            .astype(
                int
            )
        )

        p_task = probability[
            valid,
            task
        ]

        if len(
            y_task
        ) == 0:

            continue


        prediction = (
            p_task
            >= thresholds[
                task
            ]
        ).astype(
            int
        )

        rows.append({
            "Endpoint": endpoint,
            "N": len(
                y_task
            ),
            "Positive": int(
                y_task.sum()
            ),
            "Negative": int(
                (
                    1
                    - y_task
                ).sum()
            ),
            "AUC": safe_auc(
                y_task,
                p_task,
            ),
            "Accuracy": accuracy_score(
                y_task,
                prediction,
            ),
            "PR_AUC": (
                safe_average_precision(
                    y_task,
                    p_task,
                )
            ),
            "Balanced_Accuracy": (
                balanced_accuracy_score(
                    y_task,
                    prediction,
                )
                if len(
                    np.unique(
                        y_task
                    )
                )
                == 2
                else np.nan
            ),
            "F1": f1_score(
                y_task,
                prediction,
                zero_division=0,
            ),
            "MCC": (
                matthews_corrcoef(
                    y_task,
                    prediction,
                )
                if len(
                    np.unique(
                        prediction
                    )
                )
                > 1
                else 0.0
            ),
            "Threshold": thresholds[
                task
            ],
        })


    return pd.DataFrame(
        rows
    )

In [ ]:
def thresholds_from_probabilities(
    y_true,
    mask,
    probability,
    criterion="balanced_accuracy",
):

    thresholds = np.full(
        y_true.shape[1],
        0.5,
        dtype=float,
    )

    for task in range(
        y_true.shape[1]
    ):

        valid = (
            mask[
                :,
                task
            ]
            .astype(
                bool
            )
            & np.isfinite(
                probability[
                    :,
                    task
                ]
            )
        )

        y_task = (
            y_true[
                valid,
                task
            ]
            .astype(
                int
            )
        )

        p_task = probability[
            valid,
            task
        ]

        if (
            len(
                np.unique(
                    y_task
                )
            )
            == 2
        ):

            thresholds[
                task
            ] = optimize_threshold(
                y_task,
                p_task,
                criterion=criterion,
            )


    return thresholds


def cross_fitted_thresholds_and_accuracy(
    y_true,
    mask,
    probability,
    fold_ids,
):

    fold_accuracy = []

    cross_fitted_thresholds = np.full(
        (
            len(
                np.unique(
                    fold_ids
                )
            ),
            y_true.shape[1],
        ),
        0.5,
        dtype=float,
    )


    for fold in sorted(
        np.unique(
            fold_ids
        )
    ):

        training_rows = (
            fold_ids != fold
        )

        validation_rows = (
            fold_ids == fold
        )

        thresholds = (
            thresholds_from_probabilities(
                y_true[
                    training_rows
                ],
                mask[
                    training_rows
                ],
                probability[
                    training_rows
                ],
                criterion="balanced_accuracy",
            )
        )

        cross_fitted_thresholds[
            fold
        ] = thresholds

        validation_report = (
            evaluate_multitask(
                y_true[
                    validation_rows
                ],
                mask[
                    validation_rows
                ],
                probability[
                    validation_rows
                ],
                thresholds=thresholds,
                endpoints=TARGET_COLS,
            )
        )

        fold_accuracy.append(
            float(
                validation_report[
                    "Accuracy"
                ].mean()
            )
        )


    final_thresholds = (
        thresholds_from_probabilities(
            y_true,
            mask,
            probability,
            criterion="balanced_accuracy",
        )
    )

    return (
        np.array(
            fold_accuracy
        ),
        final_thresholds,
        cross_fitted_thresholds,
    )


def bootstrap_macro_auc(
    y_true,
    mask,
    probability,
    n_bootstrap=1000,
    seed=13,
):

    random_generator = (
        np.random.default_rng(
            seed
        )
    )

    bootstrap_values = []

    n_samples = len(
        y_true
    )


    for _ in range(
        n_bootstrap
    ):

        sampled_rows = (
            random_generator.integers(
                0,
                n_samples,
                n_samples,
            )
        )

        endpoint_scores = []

        for task in range(
            y_true.shape[1]
        ):

            valid = (
                mask[
                    sampled_rows,
                    task
                ]
                .astype(
                    bool
                )
            )

            y_task = (
                y_true[
                    sampled_rows,
                    task
                ][
                    valid
                ]
            )

            p_task = (
                probability[
                    sampled_rows,
                    task
                ][
                    valid
                ]
            )

            if (
                len(
                    np.unique(
                        y_task
                    )
                )
                == 2
            ):

                endpoint_scores.append(
                    roc_auc_score(
                        y_task,
                        p_task,
                    )
                )


        if endpoint_scores:

            bootstrap_values.append(
                np.mean(
                    endpoint_scores
                )
            )


    if not bootstrap_values:

        return np.array([
            np.nan,
            np.nan,
            np.nan,
        ])


    return np.percentile(
        bootstrap_values,
        [
            2.5,
            50,
            97.5,
        ],
    )

## 20. Model artifact registry

In [ ]:
ARTIFACT_DIR = (
    DIRS["outputs"]
    / "model_artifacts"
)

ARTIFACT_DIR.mkdir(
    exist_ok=True
)

MODEL_REGISTRY = {}


def save_model_artifact(
    name,
    oof_probability,
    test_probability,
    oof_embedding=None,
    test_embedding=None,
    fold_auc=None,
    fold_accuracy=None,
    metadata=None,
):

    artifact_path = (
        ARTIFACT_DIR
        / f"{name}.npz"
    )

    payload = {
        "oof_prob": (
            oof_probability
            .astype(
                np.float32
            )
        ),
        "test_prob": (
            test_probability
            .astype(
                np.float32
            )
        ),
    }

    if oof_embedding is not None:

        payload[
            "oof_embed"
        ] = (
            oof_embedding
            .astype(
                np.float32
            )
        )


    if test_embedding is not None:

        payload[
            "test_embed"
        ] = (
            test_embedding
            .astype(
                np.float32
            )
        )


    if fold_auc is not None:

        payload[
            "fold_auc"
        ] = np.asarray(
            fold_auc,
            dtype=np.float32,
        )


    if fold_accuracy is not None:

        payload[
            "fold_accuracy"
        ] = np.asarray(
            fold_accuracy,
            dtype=np.float32,
        )


    np.savez_compressed(
        artifact_path,
        **payload,
    )


    metadata = (
        metadata
        or {}
    )

    with open(
        ARTIFACT_DIR
        / f"{name}.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            metadata,
            file,
            indent=2,
            ensure_ascii=False,
        )


    MODEL_REGISTRY[
        name
    ] = {
        "path": str(
            artifact_path
        ),
        "metadata": metadata,
    }

    print(
        f"Saved model artifact: {name}"
    )


def load_model_artifact(
    name,
):

    return dict(
        np.load(
            ARTIFACT_DIR
            / f"{name}.npz"
        )
    )


def discover_artifacts():

    return sorted(
        path.stem
        for path
        in ARTIFACT_DIR.glob(
            "*.npz"
        )
    )

## 21. Masked class-balanced focal and asymmetric losses

In [ ]:
def effective_number_weights(
    labels,
    mask,
    beta=0.999,
):

    positive_count = (
        (
            labels
            * mask
        )
        .sum(
            dim=0
        )
        .clamp_min(
            1.0
        )
    )

    negative_count = (
        (
            (
                1
                - labels
            )
            * mask
        )
        .sum(
            dim=0
        )
        .clamp_min(
            1.0
        )
    )

    beta_tensor = torch.tensor(
        beta,
        device=labels.device,
    )

    positive_weight = (
        (1 - beta)
        / torch.clamp(
            1
            - torch.pow(
                beta_tensor,
                positive_count,
            ),
            min=1e-8,
        )
    )

    negative_weight = (
        (1 - beta)
        / torch.clamp(
            1
            - torch.pow(
                beta_tensor,
                negative_count,
            ),
            min=1e-8,
        )
    )

    positive_weight = (
        positive_weight
        / positive_weight
        .mean()
        .clamp_min(
            1e-8
        )
    ).clamp(
        0.2,
        6.0,
    )

    negative_weight = (
        negative_weight
        / negative_weight
        .mean()
        .clamp_min(
            1e-8
        )
    ).clamp(
        0.2,
        6.0,
    )

    return (
        positive_weight.detach(),
        negative_weight.detach(),
    )


class MaskedClassBalancedFocalLoss(
    nn.Module
):

    def __init__(
        self,
        positive_weight,
        negative_weight,
        gamma=2.0,
    ):

        super().__init__()

        self.register_buffer(
            "positive_weight",
            positive_weight,
        )

        self.register_buffer(
            "negative_weight",
            negative_weight,
        )

        self.gamma = gamma


    def forward(
        self,
        logits,
        labels,
        mask,
    ):

        probability = torch.sigmoid(
            logits
        )

        bce = (
            nn.functional
            .binary_cross_entropy_with_logits(
                logits,
                labels,
                reduction="none",
            )
        )

        probability_true = (
            labels
            * probability
            + (
                1
                - labels
            )
            * (
                1
                - probability
            )
        )

        class_weight = (
            labels
            * self.positive_weight
            + (
                1
                - labels
            )
            * self.negative_weight
        )

        loss = (
            class_weight
            * torch.pow(
                1
                - probability_true,
                self.gamma,
            )
            * bce
            * mask
        )

        loss_per_task = (
            loss.sum(
                dim=0
            )
            / mask.sum(
                dim=0
            ).clamp_min(
                1.0
            )
        )

        return loss_per_task.mean()


class MaskedAsymmetricLoss(
    nn.Module
):

    def __init__(
        self,
        positive_weight,
        negative_weight,
        gamma_positive=1.0,
        gamma_negative=4.0,
        clip=0.05,
    ):

        super().__init__()

        self.register_buffer(
            "positive_weight",
            positive_weight,
        )

        self.register_buffer(
            "negative_weight",
            negative_weight,
        )

        self.gamma_positive = (
            gamma_positive
        )

        self.gamma_negative = (
            gamma_negative
        )

        self.clip = clip


    def forward(
        self,
        logits,
        labels,
        mask,
    ):

        probability = torch.sigmoid(
            logits
        )

        negative_probability = (
            1
            - probability
            + self.clip
        ).clamp(
            max=1
        )

        positive_loss = (
            -labels
            * torch.log(
                probability
                .clamp_min(
                    1e-8
                )
            )
            * torch.pow(
                1
                - probability,
                self.gamma_positive,
            )
            * self.positive_weight
        )

        negative_loss = (
            -(
                1
                - labels
            )
            * torch.log(
                negative_probability
                .clamp_min(
                    1e-8
                )
            )
            * torch.pow(
                probability,
                self.gamma_negative,
            )
            * self.negative_weight
        )

        loss = (
            (
                positive_loss
                + negative_loss
            )
            * mask
        )

        loss_per_task = (
            loss.sum(
                dim=0
            )
            / mask.sum(
                dim=0
            ).clamp_min(
                1.0
            )
        )

        return loss_per_task.mean()

# Part IV — Model 1: DeepTox-Residual Multitask DNN

## Model 1 — Hierarchical DeepTox-Residual Multitask DNN

**Purpose:** strong structure-only multitask baseline using multi-fingerprint and extended RDKit descriptors.

**Revision 2 upgrades**

- fold-local high-performance preprocessing,
- residual shared trunk,
- separate Nuclear Receptor (NR) and Stress Response (SR) adapters,
- endpoint-specific outputs,
- strict per-task missing-label masking,
- class-balanced focal/asymmetric loss,
- early stopping on mean validation ROC-AUC.

The NR/SR hierarchy encourages related endpoints to share information without forcing all 12 tasks to use exactly the same final representation.


## 22. Reusable neural training utilities

In [ ]:
@dataclass
class TrainingHistory:
    train_loss: List[float]
    validation_auc: List[float]
    best_epoch: int
    best_auc: float


def make_grad_scaler():

    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP,
        )

    except Exception:
        return torch.cuda.amp.GradScaler(
            enabled=USE_AMP,
        )


def train_array_model(
    model,
    x_train,
    y_train,
    mask_train,
    x_validation,
    y_validation,
    mask_validation,
    seed=13,
    epochs=None,
    batch_size=None,
    learning_rate=3e-4,
    weight_decay=1e-5,
    loss_name="focal",
    patience=None,
):

    seed_everything(
        seed
    )

    epochs = (
        epochs
        or CFG["epochs"]
    )

    batch_size = (
        batch_size
        or CFG["batch_size"]
    )

    patience = (
        patience
        or CFG["patience"]
    )

    model = model.to(
        DEVICE
    )

    training_dataset = TensorDataset(
        torch.tensor(
            x_train,
            dtype=torch.float32,
        ),
        torch.tensor(
            y_train,
            dtype=torch.float32,
        ),
        torch.tensor(
            mask_train,
            dtype=torch.float32,
        ),
    )

    training_loader = DataLoader(
        training_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
    )

    positive_weight, negative_weight = (
        effective_number_weights(
            torch.tensor(
                y_train,
                device=DEVICE,
            ),
            torch.tensor(
                mask_train,
                device=DEVICE,
            ),
        )
    )

    if loss_name == "asymmetric":

        criterion = (
            MaskedAsymmetricLoss(
                positive_weight,
                negative_weight,
            )
        )

    else:

        criterion = (
            MaskedClassBalancedFocalLoss(
                positive_weight,
                negative_weight,
                gamma=2.0,
            )
        )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=max(
                epochs,
                1,
            ),
        )
    )

    scaler = make_grad_scaler()

    best_auc = -np.inf
    best_state = None
    wait = 0

    history = TrainingHistory(
        train_loss=[],
        validation_auc=[],
        best_epoch=0,
        best_auc=-np.inf,
    )

    x_validation_tensor = (
        torch.tensor(
            x_validation,
            dtype=torch.float32,
            device=DEVICE,
        )
    )


    for epoch in range(
        epochs
    ):

        model.train()

        epoch_losses = []

        for (
            x_batch,
            y_batch,
            mask_batch,
        ) in training_loader:

            x_batch = x_batch.to(
                DEVICE
            )

            y_batch = y_batch.to(
                DEVICE
            )

            mask_batch = (
                mask_batch.to(
                    DEVICE
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with amp_context():

                output = model(
                    x_batch
                )

                logits = (
                    output[0]
                    if isinstance(
                        output,
                        tuple,
                    )
                    else output
                )

                loss = criterion(
                    logits,
                    y_batch,
                    mask_batch,
                )


            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            epoch_losses.append(
                float(
                    loss.item()
                )
            )


        scheduler.step()


        model.eval()

        with torch.no_grad():

            with amp_context():

                output = model(
                    x_validation_tensor
                )

                logits = (
                    output[0]
                    if isinstance(
                        output,
                        tuple,
                    )
                    else output
                )

            validation_probability = (
                torch.sigmoid(
                    logits
                )
                .float()
                .cpu()
                .numpy()
            )


        validation_auc = (
            mean_masked_auc(
                y_validation,
                mask_validation,
                validation_probability,
            )
        )

        history.train_loss.append(
            float(
                np.mean(
                    epoch_losses
                )
            )
        )

        history.validation_auc.append(
            validation_auc
        )


        if (
            validation_auc
            > best_auc
            + 1e-5
        ):

            best_auc = (
                validation_auc
            )

            best_state = {
                key: value
                .detach()
                .cpu()
                .clone()

                for key, value
                in model
                .state_dict()
                .items()
            }

            wait = 0

            history.best_epoch = (
                epoch
            )

            history.best_auc = (
                validation_auc
            )

        else:

            wait += 1


        if wait >= patience:

            break


    if best_state is not None:

        model.load_state_dict(
            best_state
        )


    return (
        model,
        history,
    )


def predict_array_model(
    model,
    features,
    batch_size=512,
):

    model.eval()

    probabilities = []
    embeddings = []


    with torch.no_grad():

        for start in range(
            0,
            len(features),
            batch_size,
        ):

            x_batch = torch.tensor(
                features[
                    start:
                    start
                    + batch_size
                ],
                dtype=torch.float32,
                device=DEVICE,
            )

            with amp_context():

                output = model(
                    x_batch
                )


            if isinstance(
                output,
                tuple,
            ):

                logits, embedding = (
                    output
                )

            else:

                logits = output
                embedding = None


            probabilities.append(
                torch.sigmoid(
                    logits
                )
                .float()
                .cpu()
                .numpy()
            )


            if embedding is not None:

                embeddings.append(
                    embedding
                    .float()
                    .cpu()
                    .numpy()
                )


    return (
        np.vstack(
            probabilities
        ),
        (
            np.vstack(
                embeddings
            )
            if embeddings
            else None
        ),
    )

In [ ]:
def fold_level_auc(
    y_true,
    mask,
    probability,
    fold_ids,
) -> np.ndarray:

    scores = []

    for fold in sorted(
        np.unique(
            fold_ids
        )
    ):

        rows = (
            fold_ids
            == fold
        )

        scores.append(
            mean_masked_auc(
                y_true[
                    rows
                ],
                mask[
                    rows
                ],
                probability[
                    rows
                ],
            )
        )


    return np.asarray(
        scores,
        dtype=float,
    )


def finalize_oof_metrics(
    y_true,
    mask,
    probability,
    fold_ids,
):

    fold_auc = fold_level_auc(
        y_true,
        mask,
        probability,
        fold_ids,
    )

    (
        fold_accuracy,
        final_thresholds,
        cross_fitted_thresholds,
    ) = (
        cross_fitted_thresholds_and_accuracy(
            y_true,
            mask,
            probability,
            fold_ids,
        )
    )

    return {
        "fold_auc": fold_auc,
        "fold_accuracy": fold_accuracy,
        "final_thresholds": final_thresholds,
        "cross_fitted_thresholds": (
            cross_fitted_thresholds
        ),
    }


Y_TRAIN = Y_FILLED[
    train_df[
        "global_index"
    ].values
]

M_TRAIN = M[
    train_df[
        "global_index"
    ].values
]

CV_FOLD_IDS = (
    train_df[
        "cv_fold"
    ]
    .to_numpy(
        dtype=int
    )
)

## 23. Model 1 architecture

In [ ]:
class ResidualMLPBlock(
    nn.Module
):

    def __init__(
        self,
        dimension,
        dropout=0.25,
    ):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                dimension,
                dimension,
            ),
            nn.BatchNorm1d(
                dimension
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                dimension,
                dimension,
            ),
            nn.BatchNorm1d(
                dimension
            ),
        )

        self.activation = nn.GELU()


    def forward(
        self,
        x,
    ):

        return self.activation(
            x
            + self.network(
                x
            )
        )


class DeepToxResidualDNN(
    nn.Module
):

    def __init__(
        self,
        input_dimension,
        n_tasks,
        hidden_dimension=640,
        embedding_dimension=256,
        dropout=0.25,
    ):

        super().__init__()

        if n_tasks != 12:
            raise ValueError(
                "This hierarchical head expects the 12 standard Tox21 endpoints."
            )

        self.stem = nn.Sequential(
            nn.Linear(
                input_dimension,
                hidden_dimension,
            ),
            nn.BatchNorm1d(
                hidden_dimension
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
        )

        self.residual_blocks = nn.Sequential(
            ResidualMLPBlock(
                hidden_dimension,
                dropout,
            ),
            ResidualMLPBlock(
                hidden_dimension,
                dropout,
            ),
            ResidualMLPBlock(
                hidden_dimension,
                dropout * 0.8,
            ),
        )

        self.shared_embedding = nn.Sequential(
            nn.Linear(
                hidden_dimension,
                embedding_dimension,
            ),
            nn.LayerNorm(
                embedding_dimension
            ),
            nn.GELU(),
            nn.Dropout(
                dropout / 2
            ),
        )

        self.nr_adapter = nn.Sequential(
            nn.Linear(
                embedding_dimension,
                embedding_dimension,
            ),
            nn.LayerNorm(
                embedding_dimension
            ),
            nn.GELU(),
            nn.Dropout(
                dropout / 2
            ),
        )

        self.sr_adapter = nn.Sequential(
            nn.Linear(
                embedding_dimension,
                embedding_dimension,
            ),
            nn.LayerNorm(
                embedding_dimension
            ),
            nn.GELU(),
            nn.Dropout(
                dropout / 2
            ),
        )

        self.nr_head = nn.Linear(
            embedding_dimension,
            7,
        )

        self.sr_head = nn.Linear(
            embedding_dimension,
            5,
        )


    def forward(
        self,
        x,
    ):

        hidden = self.stem(
            x
        )

        hidden = self.residual_blocks(
            hidden
        )

        shared_embedding = (
            self.shared_embedding(
                hidden
            )
        )

        nr_embedding = (
            shared_embedding
            + self.nr_adapter(
                shared_embedding
            )
        )

        sr_embedding = (
            shared_embedding
            + self.sr_adapter(
                shared_embedding
            )
        )

        logits = torch.cat(
            [
                self.nr_head(
                    nr_embedding
                ),
                self.sr_head(
                    sr_embedding
                ),
            ],
            dim=1,
        )

        return (
            logits,
            shared_embedding,
        )


## 24. Model 1 — 5-fold training

In [ ]:
def run_deeptox_cv():

    model_name = (
        "Model_1_DeepTox_Residual"
    )

    oof_probability = np.full(
        (
            len(
                train_df
            ),
            len(
                TARGET_COLS
            ),
        ),
        np.nan,
        dtype=np.float32,
    )

    oof_embedding = np.zeros(
        (
            len(
                train_df
            ),
            256,
        ),
        dtype=np.float32,
    )

    test_probabilities = []
    test_embeddings = []


    for fold in range(
        N_CV_FOLDS
    ):

        print(
            f"\nModel 1 — Fold {fold+1}/{N_CV_FOLDS}"
        )

        training_local = np.where(
            CV_FOLD_IDS
            != fold
        )[0]

        validation_local = np.where(
            CV_FOLD_IDS
            == fold
        )[0]

        training_global = (
            train_df[
                "global_index"
            ].values[
                training_local
            ]
        )

        validation_global = (
            train_df[
                "global_index"
            ].values[
                validation_local
            ]
        )


        processor = (
            FoldLocalFeatureProcessor(
                max_svd=(
                    256
                    if LOW_MEMORY
                    else 512
                ),
                descriptor_pca_variance=0.995,
                random_state=(
                    PRIMARY_SEED
                    + fold
                ),
            )
        )

        x_train = (
            processor.fit_transform(
                X_FP[
                    training_global
                ],
                X_DESC[
                    training_global
                ],
            )
        )

        x_validation = (
            processor.transform(
                X_FP[
                    validation_global
                ],
                X_DESC[
                    validation_global
                ],
            )
        )

        x_test = (
            processor.transform(
                X_FP[
                    test_idx
                ],
                X_DESC[
                    test_idx
                ],
            )
        )


        model = (
            DeepToxResidualDNN(
                input_dimension=(
                    x_train.shape[1]
                ),
                n_tasks=len(
                    TARGET_COLS
                ),
                hidden_dimension=(
                    384
                    if LOW_MEMORY
                    else 512
                ),
                embedding_dimension=256,
                dropout=0.28,
            )
        )


        model, history = (
            train_array_model(
                model,
                x_train,
                Y_FILLED[
                    training_global
                ],
                M[
                    training_global
                ],
                x_validation,
                Y_FILLED[
                    validation_global
                ],
                M[
                    validation_global
                ],
                seed=(
                    PRIMARY_SEED
                    + fold
                ),
                loss_name="focal",
            )
        )


        (
            validation_probability,
            validation_embedding,
        ) = predict_array_model(
            model,
            x_validation,
        )

        (
            test_probability,
            test_embedding,
        ) = predict_array_model(
            model,
            x_test,
        )


        oof_probability[
            validation_local
        ] = (
            validation_probability
        )

        oof_embedding[
            validation_local
        ] = (
            validation_embedding
        )

        test_probabilities.append(
            test_probability
        )

        test_embeddings.append(
            test_embedding
        )


        checkpoint_path = (
            DIRS["models"]
            / f"{model_name}_fold{fold}.pt"
        )

        torch.save({
            "state_dict": (
                model.state_dict()
            ),
            "fold": fold,
            "processor": processor,
            "best_epoch": (
                history.best_epoch
            ),
            "best_auc": (
                history.best_auc
            ),
        }, checkpoint_path)


        print(
            "Best validation AUC:",
            f"{history.best_auc:.4f}",
        )


        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    test_probability = np.mean(
        test_probabilities,
        axis=0,
    )

    test_embedding = np.mean(
        test_embeddings,
        axis=0,
    )


    metric_bundle = (
        finalize_oof_metrics(
            Y_TRAIN,
            M_TRAIN,
            oof_probability,
            CV_FOLD_IDS,
        )
    )


    save_model_artifact(
        name=model_name,
        oof_probability=oof_probability,
        test_probability=test_probability,
        oof_embedding=oof_embedding,
        test_embedding=test_embedding,
        fold_auc=(
            metric_bundle[
                "fold_auc"
            ]
        ),
        fold_accuracy=(
            metric_bundle[
                "fold_accuracy"
            ]
        ),
        metadata={
            "model_number": 1,
            "description": (
                "DeepTox-style residual multitask DNN"
            ),
            "cv_folds": N_CV_FOLDS,
            "loss": (
                "masked class-balanced focal BCE"
            ),
        },
    )


    np.save(
        ARTIFACT_DIR
        / f"{model_name}_thresholds.npy",
        metric_bundle[
            "final_thresholds"
        ],
    )


    return {
        "oof_probability": (
            oof_probability
        ),
        "test_probability": (
            test_probability
        ),
        "oof_embedding": (
            oof_embedding
        ),
        "test_embedding": (
            test_embedding
        ),
        **metric_bundle,
    }


if RUN_MODEL_1_DEEPTOX:

    MODEL_1_RESULT = (
        run_deeptox_cv()
    )

else:

    print(
        "Model 1 disabled."
    )

# Part V — Model 2: Optimized AttentiveFP Multitask GNN

## Model 2 — Optimized AttentiveFP Multitask GNN

**Purpose:** Learn local molecular structure directly from atoms and bonds.

**Important implementation upgrade:** The AttentiveFP backbone outputs a genuine **256-dimensional molecular embedding**. The embedding is not reconstructed from the 12 prediction logits.

**Optimization**

- Edge-aware attentive message passing
- Graph-level attentive readout
- Optional fold-local graph contrastive pretraining
- Masked class-balanced focal loss
- AdamW, mixed precision, gradient clipping and early stopping

## 25. Model 2 architecture

In [ ]:
if PYG_AVAILABLE:

    NODE_FEATURE_DIMENSION = (
        MOLECULAR_GRAPHS[
            0
        ].x.shape[1]
    )

    EDGE_FEATURE_DIMENSION = (
        MOLECULAR_GRAPHS[
            0
        ].edge_attr.shape[1]
    )


    class OptimizedAttentiveFP(
        nn.Module
    ):

        def __init__(
            self,
            n_tasks,
            hidden_dimension=192,
            embedding_dimension=256,
            num_layers=4,
            num_timesteps=3,
            dropout=0.25,
        ):

            super().__init__()

            self.encoder = (
                AttentiveFP(
                    in_channels=(
                        NODE_FEATURE_DIMENSION
                    ),
                    hidden_channels=(
                        hidden_dimension
                    ),
                    out_channels=(
                        embedding_dimension
                    ),
                    edge_dim=(
                        EDGE_FEATURE_DIMENSION
                    ),
                    num_layers=(
                        num_layers
                    ),
                    num_timesteps=(
                        num_timesteps
                    ),
                    dropout=(
                        dropout
                    ),
                )
            )

            self.embedding_norm = (
                nn.Sequential(
                    nn.LayerNorm(
                        embedding_dimension
                    ),
                    nn.LeakyReLU(
                        0.1
                    ),
                    nn.Dropout(
                        dropout
                        / 2
                    ),
                )
            )

            self.output_head = (
                nn.Linear(
                    embedding_dimension,
                    n_tasks,
                )
            )


        def forward(
            self,
            batch,
        ):

            graph_embedding = (
                self.encoder(
                    batch.x,
                    batch.edge_index,
                    batch.edge_attr,
                    batch.batch,
                )
            )

            graph_embedding = (
                self.embedding_norm(
                    graph_embedding
                )
            )

            logits = (
                self.output_head(
                    graph_embedding
                )
            )

            return (
                logits,
                graph_embedding,
            )

## 26. Graph prediction and fold-local contrastive pretraining

In [ ]:
def graph_predict(
    model,
    loader,
):

    model.eval()

    all_indices = []
    all_probabilities = []
    all_embeddings = []


    with torch.no_grad():

        for batch in loader:

            batch = batch.to(
                DEVICE
            )

            with amp_context():

                logits, embedding = (
                    model(
                        batch
                    )
                )


            all_probabilities.append(
                torch.sigmoid(
                    logits
                )
                .float()
                .cpu()
                .numpy()
            )

            all_embeddings.append(
                embedding
                .float()
                .cpu()
                .numpy()
            )

            all_indices.append(
                batch
                .sample_idx
                .view(-1)
                .cpu()
                .numpy()
            )


    indices = np.concatenate(
        all_indices
    )

    probabilities = np.vstack(
        all_probabilities
    )

    embeddings = np.vstack(
        all_embeddings
    )

    order = np.argsort(
        indices
    )

    return (
        indices[
            order
        ],
        probabilities[
            order
        ],
        embeddings[
            order
        ],
    )


def graph_nt_xent(
    first_embedding,
    second_embedding,
    temperature=0.12,
):

    first_embedding = (
        nn.functional
        .normalize(
            first_embedding,
            dim=-1,
        )
    )

    second_embedding = (
        nn.functional
        .normalize(
            second_embedding,
            dim=-1,
        )
    )

    similarity = (
        first_embedding
        @ second_embedding.T
        / temperature
    )

    labels = torch.arange(
        len(
            first_embedding
        ),
        device=(
            first_embedding
            .device
        ),
    )

    return 0.5 * (
        nn.functional
        .cross_entropy(
            similarity,
            labels,
        )
        + nn.functional
        .cross_entropy(
            similarity.T,
            labels,
        )
    )


def augment_graph_batch(
    batch,
    edge_drop_probability=0.10,
    feature_mask_probability=0.05,
):

    augmented = batch.clone()


    if (
        augmented
        .edge_index
        .size(1)
        > 2
    ):

        keep_edges = (
            torch.rand(
                augmented
                .edge_index
                .size(1),
                device=(
                    augmented
                    .edge_index
                    .device
                ),
            )
            > edge_drop_probability
        )

        if keep_edges.sum() < 2:

            keep_edges[
                :2
            ] = True


        augmented.edge_index = (
            augmented
            .edge_index[
                :,
                keep_edges,
            ]
        )

        augmented.edge_attr = (
            augmented
            .edge_attr[
                keep_edges
            ]
        )


    feature_keep = (
        torch.rand_like(
            augmented.x
        )
        > feature_mask_probability
    ).float()

    augmented.x = (
        augmented.x
        * feature_keep
    )

    return augmented


def pretrain_attentivefp_contrastive(
    model,
    loader,
    epochs,
):

    if epochs <= 0:

        return model


    print(
        "বাংলা বার্তা: Fold-train molecular graphs দিয়ে "
        "label-free contrastive pretraining হচ্ছে..."
    )

    model = model.to(
        DEVICE
    )

    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=2e-4,
            weight_decay=1e-5,
        )
    )


    for _ in range(
        epochs
    ):

        model.train()

        for batch in loader:

            batch = batch.to(
                DEVICE
            )

            augmented = (
                augment_graph_batch(
                    batch
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with amp_context():

                _, embedding_1 = (
                    model(
                        batch
                    )
                )

                _, embedding_2 = (
                    model(
                        augmented
                    )
                )

                loss = graph_nt_xent(
                    embedding_1,
                    embedding_2,
                )


            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            optimizer.step()


    return model

## 27. Model 2 — 5-fold training

In [ ]:
def train_attentivefp_fold(
    training_graphs,
    validation_graphs,
    seed,
):

    seed_everything(
        seed
    )

    model = (
        OptimizedAttentiveFP(
            n_tasks=len(
                TARGET_COLS
            ),
            hidden_dimension=(
                160
                if LOW_MEMORY
                else 256
            ),
            embedding_dimension=256,
            num_layers=4,
            num_timesteps=3,
            dropout=0.25,
        )
        .to(
            DEVICE
        )
    )


    training_loader = (
        PyGDataLoader(
            training_graphs,
            batch_size=(
                CFG[
                    "batch_size"
                ]
            ),
            shuffle=True,
            num_workers=0,
        )
    )

    validation_loader = (
        PyGDataLoader(
            validation_graphs,
            batch_size=(
                CFG[
                    "batch_size"
                ]
            ),
            shuffle=False,
            num_workers=0,
        )
    )


    model = (
        pretrain_attentivefp_contrastive(
            model,
            training_loader,
            epochs=(
                CFG[
                    "graph_pretrain_epochs"
                ]
            ),
        )
    )


    y_train = np.vstack([
        graph.y.numpy()
        for graph
        in training_graphs
    ])

    mask_train = np.vstack([
        graph.mask.numpy()
        for graph
        in training_graphs
    ])


    positive_weight, negative_weight = (
        effective_number_weights(
            torch.tensor(
                y_train,
                device=DEVICE,
            ),
            torch.tensor(
                mask_train,
                device=DEVICE,
            ),
        )
    )


    criterion = (
        MaskedClassBalancedFocalLoss(
            positive_weight,
            negative_weight,
            gamma=2.0,
        )
    )


    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=3e-4,
            weight_decay=1e-5,
        )
    )


    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=(
                CFG[
                    "epochs"
                ]
            ),
        )
    )


    scaler = make_grad_scaler()

    best_auc = -np.inf
    best_state = None
    wait = 0


    validation_sorted = sorted(
        validation_graphs,
        key=lambda graph: int(
            graph
            .sample_idx
            .item()
        ),
    )


    y_validation = np.vstack([
        graph.y.numpy()
        for graph
        in validation_sorted
    ])

    mask_validation = np.vstack([
        graph.mask.numpy()
        for graph
        in validation_sorted
    ])


    for epoch in range(
        CFG[
            "epochs"
        ]
    ):

        model.train()

        for batch in training_loader:

            batch = batch.to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with amp_context():

                logits, _ = (
                    model(
                        batch
                    )
                )

                loss = criterion(
                    logits,
                    batch.y,
                    batch.mask,
                )


            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            scaler.step(
                optimizer
            )

            scaler.update()


        scheduler.step()


        (
            _,
            validation_probability,
            _,
        ) = graph_predict(
            model,
            validation_loader,
        )


        validation_auc = (
            mean_masked_auc(
                y_validation,
                mask_validation,
                validation_probability,
            )
        )


        if (
            validation_auc
            > best_auc
            + 1e-5
        ):

            best_auc = (
                validation_auc
            )

            best_state = {
                key: value
                .detach()
                .cpu()
                .clone()

                for key, value
                in model
                .state_dict()
                .items()
            }

            wait = 0

        else:

            wait += 1


        if (
            wait
            >= CFG[
                "patience"
            ]
        ):

            break


    if best_state is not None:

        model.load_state_dict(
            best_state
        )


    return (
        model,
        best_auc,
    )

In [ ]:
def run_attentivefp_cv():

    if (
        not PYG_AVAILABLE
        or MOLECULAR_GRAPHS is None
    ):

        print(
            "Model 2 skipped: PyTorch Geometric "
            "or graph cache unavailable."
        )

        return None


    model_name = (
        "Model_2_AttentiveFP"
    )


    oof_probability = np.full(
        (
            len(
                train_df
            ),
            len(
                TARGET_COLS
            ),
        ),
        np.nan,
        dtype=np.float32,
    )

    oof_embedding = np.zeros(
        (
            len(
                train_df
            ),
            256,
        ),
        dtype=np.float32,
    )

    test_probabilities = []
    test_embeddings = []


    for fold in range(
        N_CV_FOLDS
    ):

        print(
            f"\nModel 2 — Fold {fold+1}/{N_CV_FOLDS}"
        )

        training_local = np.where(
            CV_FOLD_IDS
            != fold
        )[0]

        validation_local = np.where(
            CV_FOLD_IDS
            == fold
        )[0]

        training_global = (
            train_df[
                "global_index"
            ].values[
                training_local
            ]
        )

        validation_global = (
            train_df[
                "global_index"
            ].values[
                validation_local
            ]
        )


        training_graphs = [
            MOLECULAR_GRAPHS[
                index
            ]
            for index
            in training_global
        ]

        validation_graphs = [
            MOLECULAR_GRAPHS[
                index
            ]
            for index
            in validation_global
        ]

        test_graphs = [
            MOLECULAR_GRAPHS[
                index
            ]
            for index
            in test_idx
        ]


        model, best_auc = (
            train_attentivefp_fold(
                training_graphs,
                validation_graphs,
                seed=(
                    PRIMARY_SEED
                    + fold
                ),
            )
        )


        validation_loader = (
            PyGDataLoader(
                validation_graphs,
                batch_size=(
                    CFG[
                        "batch_size"
                    ]
                ),
                shuffle=False,
            )
        )

        test_loader = (
            PyGDataLoader(
                test_graphs,
                batch_size=(
                    CFG[
                        "batch_size"
                    ]
                ),
                shuffle=False,
            )
        )


        (
            validation_indices,
            validation_probability,
            validation_embedding,
        ) = graph_predict(
            model,
            validation_loader,
        )

        (
            test_indices,
            test_probability,
            test_embedding,
        ) = graph_predict(
            model,
            test_loader,
        )


        validation_position = {
            global_index: position
            for position, global_index
            in enumerate(
                validation_global
            )
        }

        validation_order = np.argsort([
            validation_position[
                int(index)
            ]
            for index
            in validation_indices
        ])


        test_position = {
            global_index: position
            for position, global_index
            in enumerate(
                test_idx
            )
        }

        test_order = np.argsort([
            test_position[
                int(index)
            ]
            for index
            in test_indices
        ])


        oof_probability[
            validation_local
        ] = validation_probability[
            validation_order
        ]

        oof_embedding[
            validation_local
        ] = validation_embedding[
            validation_order
        ]

        test_probabilities.append(
            test_probability[
                test_order
            ]
        )

        test_embeddings.append(
            test_embedding[
                test_order
            ]
        )


        torch.save({
            "state_dict": (
                model.state_dict()
            ),
            "fold": fold,
            "best_auc": best_auc,
        },
        DIRS["models"]
        / f"{model_name}_fold{fold}.pt")


        print(
            "Best validation AUC:",
            f"{best_auc:.4f}",
        )


        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    test_probability = np.mean(
        test_probabilities,
        axis=0,
    )

    test_embedding = np.mean(
        test_embeddings,
        axis=0,
    )


    metric_bundle = (
        finalize_oof_metrics(
            Y_TRAIN,
            M_TRAIN,
            oof_probability,
            CV_FOLD_IDS,
        )
    )


    save_model_artifact(
        name=model_name,
        oof_probability=oof_probability,
        test_probability=test_probability,
        oof_embedding=oof_embedding,
        test_embedding=test_embedding,
        fold_auc=(
            metric_bundle[
                "fold_auc"
            ]
        ),
        fold_accuracy=(
            metric_bundle[
                "fold_accuracy"
            ]
        ),
        metadata={
            "model_number": 2,
            "description": (
                "Optimized AttentiveFP multitask GNN"
            ),
            "cv_folds": N_CV_FOLDS,
            "pretraining": (
                "fold-local graph contrastive"
            ),
        },
    )


    np.save(
        ARTIFACT_DIR
        / f"{model_name}_thresholds.npy",
        metric_bundle[
            "final_thresholds"
        ],
    )


    return {
        "oof_probability": (
            oof_probability
        ),
        "test_probability": (
            test_probability
        ),
        "oof_embedding": (
            oof_embedding
        ),
        "test_embedding": (
            test_embedding
        ),
        **metric_bundle,
    }


if RUN_MODEL_2_ATTENTIVEFP:

    MODEL_2_RESULT = (
        run_attentivefp_cv()
    )

else:

    print(
        "Model 2 disabled."
    )

# Part VI — Model 3: ChemBERTa Multitask Transformer

## Model 3 — ChemBERTa Multitask Transformer

**Purpose:** learn contextual chemical information directly from canonical and randomized SMILES.

This branch remains because sequence representations can make complementary errors. However, the previous run showed that ChemBERTa was weaker than DeepTox and AttentiveFP under CPU/low-epoch training. Therefore:

- it is trained with freeze-first and staged unfreezing,
- longer schedules are used automatically when CUDA is available,
- its predictions are admitted to Model 5 only when the OOF performance/diversity gate is passed.

The Hugging Face `UNEXPECTED lm_head` and `MISSING pooler` messages are expected when an MLM checkpoint is loaded into `AutoModel`; the notebook uses mean token pooling and does not rely on the missing pooler.


## 28. Transformer imports and checkpoint

In [ ]:
try:
    from transformers import (
        AutoTokenizer,
        AutoModel,
        AutoModelForMaskedLM,
        DataCollatorForLanguageModeling,
        get_cosine_schedule_with_warmup,
    )

    TRANSFORMERS_AVAILABLE = True

    print("Hugging Face Transformers ready.")

except Exception as error:

    TRANSFORMERS_AVAILABLE = False

    print(
        "Transformers import হয়নি:",
        error,
    )


CHEMBERTA_CHECKPOINT = (
    "DeepChem/ChemBERTa-77M-MLM"
)

RUN_CHEMBERTA_DOMAIN_ADAPTATION = (
    EXECUTION_MODE == "RESEARCH"
)

## 29. SMILES dataset and dynamic padding

In [ ]:
class SmilesMultitaskDataset(
    Dataset
):

    def __init__(
        self,
        smiles,
        labels,
        masks,
        indices,
        tokenizer,
        max_length=192,
        augment=False,
        seed=13,
    ):

        self.smiles = list(
            smiles
        )

        self.labels = np.asarray(
            labels,
            dtype=np.float32,
        )

        self.masks = np.asarray(
            masks,
            dtype=np.float32,
        )

        self.indices = np.asarray(
            indices,
            dtype=int,
        )

        self.tokenizer = tokenizer

        self.max_length = (
            max_length
        )

        self.augment = augment

        self.seed = seed


    def __len__(
        self,
    ):

        return len(
            self.smiles
        )


    def __getitem__(
        self,
        index,
    ):

        smiles = self.smiles[
            index
        ]

        if self.augment:

            smiles = (
                randomized_smiles(
                    smiles,
                    n_views=1,
                    seed=(
                        self.seed
                        + index
                    ),
                )[0]
            )


        encoded = (
            self.tokenizer(
                smiles,
                truncation=True,
                max_length=(
                    self.max_length
                ),
                add_special_tokens=True,
            )
        )


        return {
            "input_ids": encoded[
                "input_ids"
            ],
            "attention_mask": encoded[
                "attention_mask"
            ],
            "labels": self.labels[
                index
            ],
            "mask": self.masks[
                index
            ],
            "index": int(
                self.indices[
                    index
                ]
            ),
        }


class SmilesBatchCollator:

    def __init__(
        self,
        tokenizer,
    ):

        self.tokenizer = tokenizer


    def __call__(
        self,
        batch,
    ):

        token_batch = [
            {
                "input_ids": row[
                    "input_ids"
                ],
                "attention_mask": row[
                    "attention_mask"
                ],
            }
            for row
            in batch
        ]

        padded = (
            self.tokenizer.pad(
                token_batch,
                padding=True,
                return_tensors="pt",
            )
        )

        padded[
            "labels"
        ] = torch.tensor(
            np.stack([
                row[
                    "labels"
                ]
                for row
                in batch
            ]),
            dtype=torch.float32,
        )

        padded[
            "mask"
        ] = torch.tensor(
            np.stack([
                row[
                    "mask"
                ]
                for row
                in batch
            ]),
            dtype=torch.float32,
        )

        padded[
            "index"
        ] = torch.tensor(
            [
                row[
                    "index"
                ]
                for row
                in batch
            ],
            dtype=torch.long,
        )

        return padded

## 30. Optional fold-local domain-adaptive masked-language pretraining

In [ ]:
class UnlabeledSmilesDataset(
    Dataset
):

    def __init__(
        self,
        smiles,
        tokenizer,
        max_length=192,
    ):

        self.smiles = list(
            smiles
        )

        self.tokenizer = (
            tokenizer
        )

        self.max_length = (
            max_length
        )


    def __len__(
        self,
    ):

        return len(
            self.smiles
        )


    def __getitem__(
        self,
        index,
    ):

        return self.tokenizer(
            self.smiles[
                index
            ],
            truncation=True,
            max_length=(
                self.max_length
            ),
            return_special_tokens_mask=True,
        )


def fold_local_domain_adaptation(
    base_checkpoint,
    training_smiles,
    fold,
):

    output_directory = (
        DIRS["models"]
        / f"ChemBERTa_MLM_fold{fold}"
    )


    if output_directory.exists():

        return str(
            output_directory
        )


    if (
        not RUN_CHEMBERTA_DOMAIN_ADAPTATION
    ):

        return base_checkpoint


    print(
        "বাংলা বার্তা: শুধু fold-train SMILES দিয়ে "
        "ChemBERTa domain-adaptive MLM pretraining হচ্ছে..."
    )


    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            base_checkpoint
        )
    )

    model = (
        AutoModelForMaskedLM
        .from_pretrained(
            base_checkpoint
        )
        .to(
            DEVICE
        )
    )


    dataset = (
        UnlabeledSmilesDataset(
            training_smiles,
            tokenizer,
            max_length=(
                CHEMBERTA_MAX_LENGTH
            ),
        )
    )


    collator = (
        DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=True,
            mlm_probability=0.15,
        )
    )


    loader = DataLoader(
        dataset,
        batch_size=(
            4
            if LOW_MEMORY
            else 16
        ),
        shuffle=True,
        collate_fn=collator,
        num_workers=0,
    )


    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=2e-5,
            weight_decay=1e-5,
        )
    )


    model.train()


    for batch in loader:

        batch = {
            key: value.to(
                DEVICE
            )
            for key, value
            in batch.items()
        }

        optimizer.zero_grad(
            set_to_none=True
        )

        with amp_context():

            loss = model(
                **batch
            ).loss


        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0,
        )

        optimizer.step()


    model.save_pretrained(
        output_directory
    )

    tokenizer.save_pretrained(
        output_directory
    )


    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return str(
        output_directory
    )

## 31. Model 3 architecture

In [ ]:
class ChemBERTaMultitask(
    nn.Module
):

    def __init__(
        self,
        checkpoint,
        n_tasks,
        embedding_dimension=256,
        dropout=0.20,
    ):

        super().__init__()

        self.encoder = (
            AutoModel
            .from_pretrained(
                checkpoint
            )
        )

        hidden_size = (
            self.encoder
            .config
            .hidden_size
        )

        self.projection = (
            nn.Sequential(
                nn.Linear(
                    hidden_size,
                    embedding_dimension,
                ),
                nn.LayerNorm(
                    embedding_dimension
                ),
                nn.GELU(),
                nn.Dropout(
                    dropout
                ),
            )
        )

        self.output_head = (
            nn.Linear(
                embedding_dimension,
                n_tasks,
            )
        )


    def freeze_encoder(
        self,
    ):

        for parameter in (
            self.encoder
            .parameters()
        ):

            parameter.requires_grad = (
                False
            )


    def unfreeze_top_layers(
        self,
        n_layers=2,
    ):

        for parameter in (
            self.encoder
            .parameters()
        ):

            parameter.requires_grad = (
                False
            )


        encoder_module = getattr(
            self.encoder,
            "encoder",
            None,
        )

        layers = getattr(
            encoder_module,
            "layer",
            None,
        )


        if layers is not None:

            for layer in layers[
                -n_layers:
            ]:

                for parameter in (
                    layer.parameters()
                ):

                    parameter.requires_grad = (
                        True
                    )


        embeddings = getattr(
            self.encoder,
            "embeddings",
            None,
        )

        if (
            embeddings is not None
            and EXECUTION_MODE == "RESEARCH"
        ):

            for parameter in (
                embeddings.parameters()
            ):

                parameter.requires_grad = (
                    True
                )


    def forward(
        self,
        input_ids,
        attention_mask,
    ):

        output = self.encoder(
            input_ids=input_ids,
            attention_mask=(
                attention_mask
            ),
        )

        token_embeddings = (
            output
            .last_hidden_state
        )

        mask = (
            attention_mask
            .unsqueeze(-1)
            .float()
        )

        pooled = (
            (
                token_embeddings
                * mask
            )
            .sum(
                dim=1
            )
            / mask
            .sum(
                dim=1
            )
            .clamp_min(
                1.0
            )
        )

        embedding = (
            self.projection(
                pooled
            )
        )

        logits = (
            self.output_head(
                embedding
            )
        )

        return (
            logits,
            embedding,
        )

## 32. ChemBERTa training and prediction

In [ ]:
def predict_chemberta(
    model,
    loader,
):

    model.eval()

    records = []


    with torch.no_grad():

        for batch in loader:

            input_ids = (
                batch[
                    "input_ids"
                ]
                .to(
                    DEVICE
                )
            )

            attention_mask = (
                batch[
                    "attention_mask"
                ]
                .to(
                    DEVICE
                )
            )


            with amp_context():

                logits, embedding = (
                    model(
                        input_ids,
                        attention_mask,
                    )
                )


            probability = (
                torch.sigmoid(
                    logits
                )
                .float()
                .cpu()
                .numpy()
            )

            embedding = (
                embedding
                .float()
                .cpu()
                .numpy()
            )


            for (
                sample_index,
                sample_probability,
                sample_embedding,
            ) in zip(
                batch[
                    "index"
                ].numpy(),
                probability,
                embedding,
            ):

                records.append((
                    int(
                        sample_index
                    ),
                    sample_probability,
                    sample_embedding,
                ))


    records.sort(
        key=lambda row: row[0]
    )


    return (
        np.array([
            row[0]
            for row
            in records
        ]),
        np.stack([
            row[1]
            for row
            in records
        ]),
        np.stack([
            row[2]
            for row
            in records
        ]),
    )


def train_chemberta_phase(
    model,
    training_loader,
    validation_loader,
    y_validation,
    mask_validation,
    criterion,
    epochs,
    learning_rate,
    gradient_accumulation_steps,
):

    trainable_parameters = [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = (
        torch.optim.AdamW(
            trainable_parameters,
            lr=learning_rate,
            weight_decay=1e-5,
        )
    )

    total_updates = max(
        1,
        math.ceil(
            len(
                training_loader
            )
            / gradient_accumulation_steps
        )
        * epochs,
    )

    scheduler = (
        get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=max(
                1,
                int(
                    0.08
                    * total_updates
                ),
            ),
            num_training_steps=(
                total_updates
            ),
        )
    )

    scaler = make_grad_scaler()

    best_auc = -np.inf
    best_state = None
    wait = 0


    for epoch in range(
        epochs
    ):

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )


        for step, batch in enumerate(
            training_loader
        ):

            input_ids = (
                batch[
                    "input_ids"
                ]
                .to(
                    DEVICE
                )
            )

            attention_mask = (
                batch[
                    "attention_mask"
                ]
                .to(
                    DEVICE
                )
            )

            labels = (
                batch[
                    "labels"
                ]
                .to(
                    DEVICE
                )
            )

            mask = (
                batch[
                    "mask"
                ]
                .to(
                    DEVICE
                )
            )


            with amp_context():

                logits, _ = (
                    model(
                        input_ids,
                        attention_mask,
                    )
                )

                loss = (
                    criterion(
                        logits,
                        labels,
                        mask,
                    )
                    / gradient_accumulation_steps
                )


            scaler.scale(
                loss
            ).backward()


            should_step = (
                (
                    step + 1
                )
                % gradient_accumulation_steps
                == 0
                or (
                    step + 1
                )
                == len(
                    training_loader
                )
            )


            if should_step:

                scaler.unscale_(
                    optimizer
                )

                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0,
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

                scheduler.step()


        (
            _,
            validation_probability,
            _,
        ) = predict_chemberta(
            model,
            validation_loader,
        )


        validation_auc = (
            mean_masked_auc(
                y_validation,
                mask_validation,
                validation_probability,
            )
        )


        if (
            validation_auc
            > best_auc
            + 1e-5
        ):

            best_auc = (
                validation_auc
            )

            best_state = {
                key: value
                .detach()
                .cpu()
                .clone()

                for key, value
                in model
                .state_dict()
                .items()
            }

            wait = 0

        else:

            wait += 1


        if (
            wait
            >= max(
                2,
                CFG[
                    "patience"
                ]
                // 2,
            )
        ):

            break


    if best_state is not None:

        model.load_state_dict(
            best_state
        )


    return (
        model,
        best_auc,
    )

## 33. Model 3 — 5-fold training

In [ ]:
def run_chemberta_cv():

    if not TRANSFORMERS_AVAILABLE:

        print(
            "Model 3 skipped: Transformers unavailable."
        )

        return None


    model_name = (
        "Model_3_ChemBERTa"
    )


    print(
        "বাংলা বার্তা: ChemBERTa pretrained weights/tokenizer "
        "প্রথমবার Hugging Face থেকে download হতে পারে। "
        "Internet connection প্রয়োজন।"
    )


    oof_probability = np.full(
        (
            len(
                train_df
            ),
            len(
                TARGET_COLS
            ),
        ),
        np.nan,
        dtype=np.float32,
    )

    oof_embedding = np.zeros(
        (
            len(
                train_df
            ),
            256,
        ),
        dtype=np.float32,
    )

    test_probabilities = []
    test_embeddings = []


    for fold in range(
        N_CV_FOLDS
    ):

        print(
            f"\nModel 3 — Fold {fold+1}/{N_CV_FOLDS}"
        )


        training_local = np.where(
            CV_FOLD_IDS
            != fold
        )[0]

        validation_local = np.where(
            CV_FOLD_IDS
            == fold
        )[0]


        training_global = (
            train_df[
                "global_index"
            ].values[
                training_local
            ]
        )

        validation_global = (
            train_df[
                "global_index"
            ].values[
                validation_local
            ]
        )


        checkpoint = (
            fold_local_domain_adaptation(
                CHEMBERTA_CHECKPOINT,
                curated_df.loc[
                    training_global,
                    "canonical_smiles",
                ].tolist(),
                fold,
            )
        )


        tokenizer = (
            AutoTokenizer
            .from_pretrained(
                checkpoint
            )
        )


        model = (
            ChemBERTaMultitask(
                checkpoint=(
                    checkpoint
                ),
                n_tasks=len(
                    TARGET_COLS
                ),
                embedding_dimension=256,
                dropout=0.20,
            )
            .to(
                DEVICE
            )
        )


        collator = (
            SmilesBatchCollator(
                tokenizer
            )
        )


        training_dataset = (
            SmilesMultitaskDataset(
                smiles=(
                    curated_df.loc[
                        training_global,
                        "canonical_smiles",
                    ]
                ),
                labels=(
                    Y_FILLED[
                        training_global
                    ]
                ),
                masks=(
                    M[
                        training_global
                    ]
                ),
                indices=(
                    training_global
                ),
                tokenizer=(
                    tokenizer
                ),
                max_length=(
                    CHEMBERTA_MAX_LENGTH
                ),
                augment=True,
                seed=(
                    PRIMARY_SEED
                    + fold
                ),
            )
        )


        validation_dataset = (
            SmilesMultitaskDataset(
                smiles=(
                    curated_df.loc[
                        validation_global,
                        "canonical_smiles",
                    ]
                ),
                labels=(
                    Y_FILLED[
                        validation_global
                    ]
                ),
                masks=(
                    M[
                        validation_global
                    ]
                ),
                indices=(
                    validation_global
                ),
                tokenizer=(
                    tokenizer
                ),
                max_length=(
                    CHEMBERTA_MAX_LENGTH
                ),
                augment=False,
            )
        )


        test_dataset = (
            SmilesMultitaskDataset(
                smiles=(
                    curated_df.loc[
                        test_idx,
                        "canonical_smiles",
                    ]
                ),
                labels=(
                    Y_FILLED[
                        test_idx
                    ]
                ),
                masks=(
                    M[
                        test_idx
                    ]
                ),
                indices=(
                    test_idx
                ),
                tokenizer=(
                    tokenizer
                ),
                max_length=(
                    CHEMBERTA_MAX_LENGTH
                ),
                augment=False,
            )
        )


        batch_size = (
            4
            if LOW_MEMORY
            else 12
        )


        training_loader = (
            DataLoader(
                training_dataset,
                batch_size=(
                    batch_size
                ),
                shuffle=True,
                collate_fn=(
                    collator
                ),
                num_workers=0,
            )
        )


        validation_loader = (
            DataLoader(
                validation_dataset,
                batch_size=(
                    batch_size
                    * 2
                ),
                shuffle=False,
                collate_fn=(
                    collator
                ),
                num_workers=0,
            )
        )


        test_loader = (
            DataLoader(
                test_dataset,
                batch_size=(
                    batch_size
                    * 2
                ),
                shuffle=False,
                collate_fn=(
                    collator
                ),
                num_workers=0,
            )
        )


        positive_weight, negative_weight = (
            effective_number_weights(
                torch.tensor(
                    Y_FILLED[
                        training_global
                    ],
                    device=DEVICE,
                ),
                torch.tensor(
                    M[
                        training_global
                    ],
                    device=DEVICE,
                ),
            )
        )


        criterion = (
            MaskedClassBalancedFocalLoss(
                positive_weight,
                negative_weight,
                gamma=2.0,
            )
        )


        model.freeze_encoder()


        model, head_auc = (
            train_chemberta_phase(
                model,
                training_loader,
                validation_loader,
                Y_FILLED[
                    validation_global
                ],
                M[
                    validation_global
                ],
                criterion,
                epochs=(
                    CFG[
                        "chemberta_head_epochs"
                    ]
                ),
                learning_rate=3e-4,
                gradient_accumulation_steps=(
                    GRADIENT_ACCUMULATION_STEPS
                ),
            )
        )


        model.unfreeze_top_layers(
            n_layers=(
                2
                if LOW_MEMORY
                else 3
            )
        )


        model, fine_tune_auc = (
            train_chemberta_phase(
                model,
                training_loader,
                validation_loader,
                Y_FILLED[
                    validation_global
                ],
                M[
                    validation_global
                ],
                criterion,
                epochs=(
                    CFG[
                        "chemberta_finetune_epochs"
                    ]
                ),
                learning_rate=2e-5,
                gradient_accumulation_steps=(
                    GRADIENT_ACCUMULATION_STEPS
                ),
            )
        )


        (
            validation_indices,
            validation_probability,
            validation_embedding,
        ) = predict_chemberta(
            model,
            validation_loader,
        )


        (
            test_indices,
            test_probability,
            test_embedding,
        ) = predict_chemberta(
            model,
            test_loader,
        )


        validation_position = {
            global_index: position
            for position, global_index
            in enumerate(
                validation_global
            )
        }

        validation_order = np.argsort([
            validation_position[
                int(index)
            ]
            for index
            in validation_indices
        ])


        test_position = {
            global_index: position
            for position, global_index
            in enumerate(
                test_idx
            )
        }

        test_order = np.argsort([
            test_position[
                int(index)
            ]
            for index
            in test_indices
        ])


        oof_probability[
            validation_local
        ] = validation_probability[
            validation_order
        ]

        oof_embedding[
            validation_local
        ] = validation_embedding[
            validation_order
        ]

        test_probabilities.append(
            test_probability[
                test_order
            ]
        )

        test_embeddings.append(
            test_embedding[
                test_order
            ]
        )


        model_directory = (
            DIRS["models"]
            / f"{model_name}_fold{fold}"
        )

        model_directory.mkdir(
            exist_ok=True
        )


        torch.save({
            "state_dict": (
                model.state_dict()
            ),
            "fold": fold,
            "head_phase_auc": (
                head_auc
            ),
            "fine_tune_auc": (
                fine_tune_auc
            ),
            "checkpoint": (
                checkpoint
            ),
        },
        model_directory
        / "pytorch_model_state.pt")


        tokenizer.save_pretrained(
            model_directory
        )


        print(
            "Best validation AUC:",
            f"{max(head_auc, fine_tune_auc):.4f}",
        )


        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    test_probability = np.mean(
        test_probabilities,
        axis=0,
    )

    test_embedding = np.mean(
        test_embeddings,
        axis=0,
    )


    metric_bundle = (
        finalize_oof_metrics(
            Y_TRAIN,
            M_TRAIN,
            oof_probability,
            CV_FOLD_IDS,
        )
    )


    save_model_artifact(
        name=model_name,
        oof_probability=oof_probability,
        test_probability=test_probability,
        oof_embedding=oof_embedding,
        test_embedding=test_embedding,
        fold_auc=(
            metric_bundle[
                "fold_auc"
            ]
        ),
        fold_accuracy=(
            metric_bundle[
                "fold_accuracy"
            ]
        ),
        metadata={
            "model_number": 3,
            "description": (
                "ChemBERTa multitask SMILES Transformer"
            ),
            "cv_folds": N_CV_FOLDS,
            "checkpoint": (
                CHEMBERTA_CHECKPOINT
            ),
            "randomized_smiles_training": True,
            "staged_unfreezing": True,
            "domain_adaptation": (
                RUN_CHEMBERTA_DOMAIN_ADAPTATION
            ),
        },
    )


    np.save(
        ARTIFACT_DIR
        / f"{model_name}_thresholds.npy",
        metric_bundle[
            "final_thresholds"
        ],
    )


    return {
        "oof_probability": (
            oof_probability
        ),
        "test_probability": (
            test_probability
        ),
        "oof_embedding": (
            oof_embedding
        ),
        "test_embedding": (
            test_embedding
        ),
        **metric_bundle,
    }


if RUN_MODEL_3_CHEMBERTA:

    MODEL_3_RESULT = (
        run_chemberta_cv()
    )

else:

    print(
        "Model 3 disabled."
    )

# Part VII — Model 4: Leakage-Safe GPS–R-GCN ToxKG Hybrid

## Model 4 — Leakage-Safe GPS–R-GCN ToxKG Hybrid

**Purpose:** Combine molecular structural information with external chemical–gene–pathway context.

The paper reports that heterogeneous knowledge-graph models outperform homogeneous structural baselines on many Tox21 endpoints. This implementation preserves that insight while applying stricter leakage controls.

### Mandatory leakage policy

The model **removes**:

- `CHEMICALHASACTIVEASSAY`
- `CHEMICALHASINACTIVEASSAY`
- any relation containing target activity/outcome semantics
- any feature derived from the 12 Tox21 labels

The safe graph uses only mechanism-oriented relations such as:

- chemical binds gene,
- chemical increases/decreases gene expression,
- gene belongs to pathway,
- gene interacts with gene.

The model is evaluated only where KG coverage exists; coverage is reported explicitly.

**Hybrid upgrade:** Each knowledge-graph layer learns both a GPS representation (local GINE + global Transformer attention) and an R-GCN relation-specific representation. A learned gate merges the two, allowing the model to exploit GPS's strong average performance and R-GCN's endpoint-specific strengths within one optimized model.

## 34. Download the public ToxKG repository subset

In [ ]:
TOXKG_REPO_ID = (
    "xiejunjie/"
    "Molecular-toxicity-prediction"
)

TOXKG_SNAPSHOT = (
    DIRS["kg_raw"]
    / "Molecular-toxicity-prediction"
)


if (
    RUN_MODEL_4_GPS_TOXKG
    and AUTO_DOWNLOAD_TOXKG
    and not TOXKG_SNAPSHOT.exists()
):

    print(
        "বাংলা বার্তা: ToxKG paper-এর public Hugging Face "
        "repository থেকে প্রয়োজনীয় CSV files download করা হচ্ছে। "
        "Internet connection প্রয়োজন।"
    )

    try:

        from huggingface_hub import (
            snapshot_download
        )

        snapshot_download(
            repo_id=(
                TOXKG_REPO_ID
            ),
            repo_type="model",
            local_dir=str(
                TOXKG_SNAPSHOT
            ),
            allow_patterns=[
                "**/*.csv",
                "**/*.json",
                "**/*.txt",
                "**/*.npy",
                "**/*.npz",
            ],
        )

        print(
            "বাংলা বার্তা: ToxKG repository subset "
            "download সম্পন্ন হয়েছে।"
        )

    except Exception as error:

        print(
            "বাংলা বার্তা: ToxKG automatic download ব্যর্থ হয়েছে."
        )

        print(error)

        print(
            "Manual option: repository download করে "
            "knowledge_graph/raw/Molecular-toxicity-prediction "
            "ফোল্ডারে রাখুন।"
        )


elif RUN_MODEL_4_GPS_TOXKG:

    print(
        "ToxKG snapshot path:",
        TOXKG_SNAPSHOT,
    )

## 35. Locate and inspect KG files

In [ ]:
def locate_repository_file(
    patterns,
):

    if not TOXKG_SNAPSHOT.exists():

        return None


    for pattern in patterns:

        matches = list(
            TOXKG_SNAPSHOT.rglob(
                pattern
            )
        )

        if matches:

            return matches[
                0
            ]


    return None


KG_FILES = {
    "triples": locate_repository_file([
        "filtered_triples.csv",
        "*triples*.csv",
    ]),

    "compound": locate_repository_file([
        "compound_master.csv",
        "common_cids.csv",
        "*compound*.csv",
    ]),

    "common_cids": locate_repository_file([
        "common_cids.csv",
    ]),

    "fingerprints": locate_repository_file([
        "gnn_input_fp.csv",
    ]),

    "genes": locate_repository_file([
        "gnn_input_genes.csv",
        "*genes*.csv",
    ]),

    "pathways": locate_repository_file([
        "gnn_input_pathways.csv",
        "*pathways*.csv",
    ]),
}


print(
    "Detected KG files:"
)

for name, path in (
    KG_FILES.items()
):

    print(
        f"  {name}: {path}"
    )

## 36. Strict relation audit and target-edge removal

In [ ]:
SAFE_RELATION_ALLOWLIST = {
    "CHEMICALBINDSGENE",
    "CHEMICALDECREASESEXPRESSION",
    "CHEMICALINCREASESEXPRESSION",
    "GENEINPATHWAY",
    "GENEINTERACTSWITHGENE",
}

SUSPECT_RELATION_PATTERNS = [
    "activeassay",
    "inactiveassay",
    "hasactive",
    "hasinactive",
    "tox21label",
    "targetlabel",
    "activitylabel",
    "assayoutcome",
]


def normalize_token(
    value,
):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(
            value
        ).lower(),
    )


def detect_column(
    dataframe,
    candidates,
):

    lowered = {
        column.lower(): column
        for column
        in dataframe.columns
    }

    for candidate in candidates:

        if candidate.lower() in lowered:

            return lowered[
                candidate.lower()
            ]


    for column in dataframe.columns:

        if any(
            candidate.lower()
            in column.lower()

            for candidate
            in candidates
        ):

            return column


    return None


def audit_and_filter_triples(
    triples,
):

    source_column = detect_column(
        triples,
        [
            "source",
            "head",
            "start",
            "subject",
            "from",
        ],
    )

    relation_column = detect_column(
        triples,
        [
            "relation",
            "predicate",
            "edge",
            "type",
        ],
    )

    target_column = detect_column(
        triples,
        [
            "target",
            "tail",
            "end",
            "object",
            "to",
        ],
    )


    if not all([
        source_column,
        relation_column,
        target_column,
    ]):

        raise ValueError(
            "KG triple columns detect করা যায়নি: "
            f"{triples.columns.tolist()}"
        )


    relation_normalized = (
        triples[
            relation_column
        ]
        .astype(str)
        .map(
            normalize_token
        )
    )


    suspect = np.zeros(
        len(
            triples
        ),
        dtype=bool,
    )


    for pattern in (
        SUSPECT_RELATION_PATTERNS
    ):

        suspect |= (
            relation_normalized
            .str
            .contains(
                pattern,
                regex=False,
            )
            .values
        )


    normalized_allowlist = {
        normalize_token(
            relation
        )
        for relation
        in SAFE_RELATION_ALLOWLIST
    }


    in_allowlist = (
        relation_normalized
        .isin(
            normalized_allowlist
        )
        .values
    )


    safe = triples.loc[
        (~suspect)
        & in_allowlist
    ].copy()


    removed = triples.loc[
        suspect
        | (
            ~in_allowlist
        )
    ].copy()


    return (
        safe,
        removed,
        {
            "source": source_column,
            "relation": relation_column,
            "target": target_column,
        },
    )


SAFE_TRIPLES = None
REMOVED_TRIPLES = None
TRIPLE_COLUMNS = None


if KG_FILES[
    "triples"
]:

    triples_raw = pd.read_csv(
        KG_FILES[
            "triples"
        ]
    )

    (
        SAFE_TRIPLES,
        REMOVED_TRIPLES,
        TRIPLE_COLUMNS,
    ) = audit_and_filter_triples(
        triples_raw
    )


    SAFE_TRIPLES.to_csv(
        DIRS["kg_safe"]
        / "safe_core_triples.csv",
        index=False,
    )

    REMOVED_TRIPLES.to_csv(
        DIRS["kg_provenance"]
        / "removed_noncore_or_suspect_edges.csv",
        index=False,
    )


    print(
        f"KG triples: raw={len(triples_raw):,}, "
        f"safe={len(SAFE_TRIPLES):,}, "
        f"removed={len(REMOVED_TRIPLES):,}"
    )


    print(
        "Safe relations:",
        sorted(
            SAFE_TRIPLES[
                TRIPLE_COLUMNS[
                    "relation"
                ]
            ]
            .astype(str)
            .unique()
            .tolist()
        ),
    )


else:

    print(
        "KG triple file পাওয়া যায়নি। "
        "Model 4 will skip gracefully."
    )

In [ ]:
leakage_report = {
    "molecule_overlap": len(
        train_inchikeys
        & test_inchikeys
    ),
    "scaffold_overlap": len(
        train_scaffolds
        & test_scaffolds
    ),
    "target_or_noncore_edges_removed": (
        int(
            len(
                REMOVED_TRIPLES
            )
        )
        if REMOVED_TRIPLES
        is not None
        else None
    ),
    "safe_triples_available": (
        SAFE_TRIPLES
        is not None
    ),
    "full_dataset_feature_fit": False,
    "test_used_for_tuning": False,
}


leakage_report[
    "pass"
] = (
    leakage_report[
        "molecule_overlap"
    ]
    == 0

    and leakage_report[
        "scaffold_overlap"
    ]
    == 0
)


with open(
    DIRS["outputs"]
    / "05_leakage_audit.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        leakage_report,
        file,
        indent=2,
    )


display(
    pd.DataFrame([
        leakage_report
    ])
)


assert (
    leakage_report[
        "pass"
    ]
), "Leakage audit failed."

## 37. KG identity mapping and explicit node typing

In [ ]:
def read_first_column_as_set(
    path,
):

    if path is None:

        return set()


    dataframe = pd.read_csv(
        path
    )

    if dataframe.empty:

        return set()


    likely_id_column = (
        detect_column(
            dataframe,
            [
                "id",
                "gene",
                "pathway",
                "node",
                "identifier",
            ],
        )
        or dataframe.columns[
            0
        ]
    )


    return set(
        dataframe[
            likely_id_column
        ]
        .astype(str)
        .tolist()
    )


GENE_IDS = read_first_column_as_set(
    KG_FILES[
        "genes"
    ]
)

PATHWAY_IDS = read_first_column_as_set(
    KG_FILES[
        "pathways"
    ]
)


KG_COMPOUND_DF = None
KG_SMILES_COLUMN = None
KG_CID_COLUMN = None


if KG_FILES[
    "compound"
]:

    KG_COMPOUND_DF = pd.read_csv(
        KG_FILES[
            "compound"
        ]
    )

    KG_SMILES_COLUMN = (
        detect_column(
            KG_COMPOUND_DF,
            [
                "smiles",
                "canonical_smiles",
            ],
        )
    )

    KG_CID_COLUMN = (
        detect_column(
            KG_COMPOUND_DF,
            [
                "cid",
                "pubchem_cid",
                "compound_id",
                "chemical_id",
            ],
        )
    )


    if KG_SMILES_COLUMN:

        print(
            "বাংলা বার্তা: KG compound SMILES standardize করা হচ্ছে..."
        )

        standardized = [
            standardize_smiles(
                smiles
            )[
                "canonical_smiles"
            ]

            for smiles
            in tqdm(
                KG_COMPOUND_DF[
                    KG_SMILES_COLUMN
                ],
                desc="KG compound identities",
            )
        ]

        KG_COMPOUND_DF[
            "canonical_smiles"
        ] = standardized


    display(
        KG_COMPOUND_DF.head(
            3
        )
    )


else:

    print(
        "KG compound mapping file পাওয়া যায়নি।"
    )

## 38. Build leakage-safe two-hop mechanism graphs — corrected CID mapping

The previous run created only 8 local KG graphs because the chemical identifiers in the triples were compared as raw strings. The repository may encode the same PubChem CID as numeric values, decimal-like strings, prefixed identifiers or URIs.

Revision 2:

- normalizes PubChem CIDs before any join,
- uses relation-aware node typing,
- canonicalizes chemical, gene and pathway node keys,
- reports mapping and graph coverage,
- saves unmatched-ID diagnostics,
- refuses to treat a tiny KG subset as a valid model branch.

Only the safe relation allowlist is used. Active/inactive assay outcome edges remain excluded.


In [ ]:
KG_LOCAL_GRAPHS = {}
KG_RELATION_VOCAB = {}
KG_NODE_ID_VOCAB = {}
KG_MATCHED_GLOBAL = np.array(
    [],
    dtype=int,
)


CHEMICAL_GENE_RELATIONS = {
    normalize_token("CHEMICALBINDSGENE"),
    normalize_token("CHEMICALDECREASESEXPRESSION"),
    normalize_token("CHEMICALINCREASESEXPRESSION"),
}

GENE_PATHWAY_RELATION = normalize_token(
    "GENEINPATHWAY"
)

GENE_GENE_RELATION = normalize_token(
    "GENEINTERACTSWITHGENE"
)


def normalize_pubchem_cid(
    value,
):

    if value is None or (
        isinstance(value, float)
        and np.isnan(value)
    ):
        return None


    text = str(
        value
    ).strip()


    # Plain integer or decimal-like CSV value.
    if re.fullmatch(
        r"\d+(?:\.0+)?",
        text,
    ):

        return str(
            int(
                float(
                    text
                )
            )
        )


    patterns = [
        r"(?:pubchem(?:\.compound)?|compound|chemical|cid)"
        r"[^0-9]{0,12}(\d+)",
        r"/compound/(\d+)",
        r"/cid/(\d+)",
        r"[:/_-](\d+)$",
    ]


    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        if match:

            return str(
                int(
                    match.group(
                        1
                    )
                )
            )


    return None


def normalize_entity_text(
    value,
):

    text = str(
        value
    ).strip()

    text = re.sub(
        r"\.0$",
        "",
        text,
    )

    return text


def chemical_node_key(
    cid,
):

    return (
        f"CHEM::{cid}"
    )


def gene_node_key(
    raw_id,
):

    return (
        "GENE::"
        + normalize_entity_text(
            raw_id
        )
    )


def pathway_node_key(
    raw_id,
):

    return (
        "PATH::"
        + normalize_entity_text(
            raw_id
        )
    )


def typed_edge_from_raw(
    source,
    relation,
    target,
    cid_to_global,
):

    relation_norm = (
        normalize_token(
            relation
        )
    )


    if relation_norm in (
        CHEMICAL_GENE_RELATIONS
    ):

        cid = (
            normalize_pubchem_cid(
                source
            )
        )


        if (
            cid is None
            or cid not in cid_to_global
        ):

            return None


        return (
            chemical_node_key(
                cid
            ),
            str(
                relation
            ),
            gene_node_key(
                target
            ),
            0,
            1,
        )


    if (
        relation_norm
        == GENE_PATHWAY_RELATION
    ):

        return (
            gene_node_key(
                source
            ),
            str(
                relation
            ),
            pathway_node_key(
                target
            ),
            1,
            2,
        )


    if (
        relation_norm
        == GENE_GENE_RELATION
    ):

        return (
            gene_node_key(
                source
            ),
            str(
                relation
            ),
            gene_node_key(
                target
            ),
            1,
            1,
        )


    return None


if (
    PYG_AVAILABLE
    and SAFE_TRIPLES is not None
    and KG_COMPOUND_DF is not None
    and KG_CID_COLUMN is not None
    and KG_SMILES_COLUMN is not None
):

    source_column = (
        TRIPLE_COLUMNS[
            "source"
        ]
    )

    relation_column = (
        TRIPLE_COLUMNS[
            "relation"
        ]
    )

    target_column = (
        TRIPLE_COLUMNS[
            "target"
        ]
    )


    curated_smiles_to_global = {
        smiles: index

        for index, smiles
        in enumerate(
            curated_df[
                "canonical_smiles"
            ]
        )
    }


    kg_rows = (
        KG_COMPOUND_DF[
            KG_COMPOUND_DF[
                "canonical_smiles"
            ].isin(
                curated_smiles_to_global
            )
        ]
        .dropna(
            subset=[
                "canonical_smiles",
                KG_CID_COLUMN,
            ]
        )
        .copy()
    )


    cid_to_global = {}


    for cid_value, smiles in zip(
        kg_rows[
            KG_CID_COLUMN
        ],
        kg_rows[
            "canonical_smiles"
        ],
    ):

        cid = (
            normalize_pubchem_cid(
                cid_value
            )
        )


        if cid is not None:

            cid_to_global[
                cid
            ] = (
                curated_smiles_to_global[
                    smiles
                ]
            )


    print(
        "Curated molecules with a normalized KG CID:",
        f"{len(cid_to_global):,}",
    )


    typed_edges = []

    unmatched_chemical_sources = (
        Counter()
    )


    triple_view = (
        SAFE_TRIPLES[
            [
                source_column,
                relation_column,
                target_column,
            ]
        ]
    )


    for row in tqdm(
        triple_view.itertuples(
            index=False,
            name=None,
        ),
        total=len(
            triple_view
        ),
        desc=(
            "Normalize safe ToxKG edges"
        ),
    ):

        source, relation, target = row

        typed_edge = (
            typed_edge_from_raw(
                source,
                relation,
                target,
                cid_to_global,
            )
        )


        if typed_edge is None:

            if (
                normalize_token(
                    relation
                )
                in CHEMICAL_GENE_RELATIONS
            ):

                unmatched_chemical_sources[
                    str(
                        source
                    )
                ] += 1

            continue


        typed_edges.append(
            typed_edge
        )


    print(
        "Usable leakage-safe typed edges:",
        f"{len(typed_edges):,}",
    )


    outgoing = defaultdict(
        list
    )

    incoming = defaultdict(
        list
    )

    inferred_node_types = {}


    for (
        source,
        relation,
        target,
        source_type,
        target_type,
    ) in typed_edges:

        outgoing[
            source
        ].append(
            (
                relation,
                target,
            )
        )

        incoming[
            target
        ].append(
            (
                relation,
                source,
            )
        )

        inferred_node_types[
            source
        ] = source_type

        inferred_node_types[
            target
        ] = target_type


    local_graph_specs = {}


    for (
        cid,
        global_index,
    ) in tqdm(
        cid_to_global.items(),
        desc=(
            "ToxKG local mechanism graphs"
        ),
    ):

        root_key = (
            chemical_node_key(
                cid
            )
        )

        node_ids = [
            root_key
        ]

        node_types = [
            0
        ]

        node_position = {
            root_key: 0
        }

        edges = []
        relations = []


        first_hop = (
            outgoing.get(
                root_key,
                []
            )
        )


        first_hop = sorted(
            first_hop,
            key=lambda item: (
                normalize_token(
                    item[0]
                ),
                item[1],
            ),
        )[
            :MAX_KG_FIRST_HOP
        ]


        for (
            relation,
            neighbor,
        ) in first_hop:

            if neighbor not in (
                node_position
            ):

                node_position[
                    neighbor
                ] = len(
                    node_ids
                )

                node_ids.append(
                    neighbor
                )

                node_types.append(
                    inferred_node_types.get(
                        neighbor,
                        1,
                    )
                )


            neighbor_position = (
                node_position[
                    neighbor
                ]
            )


            edges.extend([
                [
                    0,
                    neighbor_position,
                ],
                [
                    neighbor_position,
                    0,
                ],
            ])

            relations.extend([
                str(
                    relation
                )
                + "::forward",

                str(
                    relation
                )
                + "::reverse",
            ])


        first_hop_nodes = list(
            node_position
            .keys()
        )[1:]


        for first_node in (
            first_hop_nodes
        ):

            if (
                inferred_node_types.get(
                    first_node,
                    3,
                )
                != 1
            ):

                continue


            second_hop = []


            for (
                relation,
                neighbor,
            ) in outgoing.get(
                first_node,
                [],
            ):

                relation_norm = (
                    normalize_token(
                        relation
                    )
                )


                if relation_norm in {
                    GENE_PATHWAY_RELATION,
                    GENE_GENE_RELATION,
                }:

                    second_hop.append(
                        (
                            relation,
                            neighbor,
                            "out",
                        )
                    )


            for (
                relation,
                neighbor,
            ) in incoming.get(
                first_node,
                [],
            ):

                relation_norm = (
                    normalize_token(
                        relation
                    )
                )


                if (
                    relation_norm
                    == GENE_GENE_RELATION
                ):

                    second_hop.append(
                        (
                            relation,
                            neighbor,
                            "in",
                        )
                    )


            second_hop = sorted(
                second_hop,
                key=lambda item: (
                    normalize_token(
                        item[0]
                    ),
                    item[1],
                ),
            )[
                :MAX_KG_SECOND_HOP_PER_GENE
            ]


            for (
                relation,
                neighbor,
                direction,
            ) in second_hop:

                if (
                    neighbor
                    not in node_position
                ):

                    if (
                        len(
                            node_ids
                        )
                        >= MAX_KG_NODES_PER_MOLECULE
                    ):

                        break


                    node_position[
                        neighbor
                    ] = len(
                        node_ids
                    )

                    node_ids.append(
                        neighbor
                    )

                    node_types.append(
                        inferred_node_types.get(
                            neighbor,
                            3,
                        )
                    )


                first_position = (
                    node_position[
                        first_node
                    ]
                )

                second_position = (
                    node_position[
                        neighbor
                    ]
                )


                if direction == "out":

                    forward_edge = [
                        first_position,
                        second_position,
                    ]

                    reverse_edge = [
                        second_position,
                        first_position,
                    ]

                else:

                    forward_edge = [
                        second_position,
                        first_position,
                    ]

                    reverse_edge = [
                        first_position,
                        second_position,
                    ]


                edges.extend([
                    forward_edge,
                    reverse_edge,
                ])

                relations.extend([
                    str(
                        relation
                    )
                    + "::forward",

                    str(
                        relation
                    )
                    + "::reverse",
                ])


        if edges:

            local_graph_specs[
                global_index
            ] = {
                "node_ids": (
                    node_ids
                ),
                "node_types": (
                    node_types
                ),
                "edges": edges,
                "relations": (
                    relations
                ),
            }


    all_node_ids = sorted({
        node_id

        for specification
        in local_graph_specs.values()

        for node_id
        in specification[
            "node_ids"
        ][1:]
    })


    KG_NODE_ID_VOCAB = {
        node_id: index + 1

        for index, node_id
        in enumerate(
            all_node_ids
        )
    }


    all_relations = sorted({
        relation

        for specification
        in local_graph_specs.values()

        for relation
        in specification[
            "relations"
        ]
    })


    KG_RELATION_VOCAB = {
        relation: index

        for index, relation
        in enumerate(
            all_relations
        )
    }


    KG_LOCAL_GRAPHS = {}


    for (
        global_index,
        specification,
    ) in local_graph_specs.items():

        encoded_node_ids = [
            0
        ] + [
            KG_NODE_ID_VOCAB[
                node_id
            ]

            for node_id
            in specification[
                "node_ids"
            ][1:]
        ]


        encoded_relations = [
            KG_RELATION_VOCAB[
                relation
            ]

            for relation
            in specification[
                "relations"
            ]
        ]


        graph = Data(
            node_id=torch.tensor(
                encoded_node_ids,
                dtype=torch.long,
            ),

            node_type=torch.tensor(
                specification[
                    "node_types"
                ],
                dtype=torch.long,
            ),

            edge_index=torch.tensor(
                specification[
                    "edges"
                ],
                dtype=torch.long,
            ).t().contiguous(),

            edge_type=torch.tensor(
                encoded_relations,
                dtype=torch.long,
            ),
        )


        graph.y = torch.tensor(
            Y_FILLED[
                global_index
            ],
            dtype=torch.float32,
        ).view(
            1,
            -1,
        )


        graph.mask = torch.tensor(
            M[
                global_index
            ],
            dtype=torch.float32,
        ).view(
            1,
            -1,
        )


        graph.sample_idx = torch.tensor(
            [
                global_index
            ],
            dtype=torch.long,
        )


        KG_LOCAL_GRAPHS[
            global_index
        ] = graph


    KG_MATCHED_GLOBAL = np.array(
        sorted(
            KG_LOCAL_GRAPHS
            .keys()
        ),
        dtype=int,
    )


    expected_kg_matches = max(
        len(
            cid_to_global
        ),
        1,
    )

    KG_GRAPH_COVERAGE = (
        len(
            KG_LOCAL_GRAPHS
        )
        / expected_kg_matches
    )


    print(
        "Leakage-safe KG graphs:",
        f"{len(KG_LOCAL_GRAPHS):,}",
    )

    print(
        "KG graph coverage among CID-matched molecules:",
        f"{KG_GRAPH_COVERAGE:.1%}",
    )

    print(
        "Node-ID vocabulary:",
        f"{len(KG_NODE_ID_VOCAB):,}",
    )

    print(
        "Relation vocabulary:",
        f"{len(KG_RELATION_VOCAB):,}",
    )


    if (
        unmatched_chemical_sources
    ):

        unmatched_table = pd.DataFrame(
            unmatched_chemical_sources
            .most_common(
                50
            ),
            columns=[
                "raw_chemical_source",
                "edge_count",
            ],
        )

        unmatched_table.to_csv(
            DIRS[
                "kg_provenance"
            ]
            / "unmatched_chemical_source_examples.csv",
            index=False,
        )


    coverage_report = pd.DataFrame({
        "Measure": [
            "CID-matched curated molecules",
            "Local KG graphs",
            "Coverage",
            "Safe typed edges",
        ],
        "Value": [
            len(
                cid_to_global
            ),
            len(
                KG_LOCAL_GRAPHS
            ),
            KG_GRAPH_COVERAGE,
            len(
                typed_edges
            ),
        ],
    })


    coverage_report.to_csv(
        DIRS[
            "outputs"
        ]
        / "kg_mapping_coverage_v2.csv",
        index=False,
    )

    display(
        coverage_report
    )


    if (
        KG_GRAPH_COVERAGE
        < MIN_KG_GRAPH_COVERAGE
    ):

        print(
            "বাংলা সতর্কতা: KG coverage research gate-এর নিচে। "
            "Model 4 final ensemble-এ ঢুকবে না যতক্ষণ mapping ঠিক না হয়।"
        )

        print(
            "Diagnostic file:",
            DIRS[
                "kg_provenance"
            ]
            / "unmatched_chemical_source_examples.csv",
        )


else:

    KG_GRAPH_COVERAGE = 0.0

    print(
        "Safe KG inputs incomplete; "
        "Model 4 will skip gracefully."
    )


## 39. NaN-safe metric patch for partial KG coverage

In [ ]:
def mean_masked_auc(
    y_true,
    mask,
    probability,
) -> float:

    endpoint_scores = []

    for task in range(
        y_true.shape[1]
    ):

        valid = (
            mask[:, task].astype(bool)
            & np.isfinite(
                probability[:, task]
            )
        )

        if (
            valid.sum() > 1
            and len(
                np.unique(
                    y_true[valid, task]
                )
            )
            == 2
        ):

            endpoint_scores.append(
                roc_auc_score(
                    y_true[valid, task],
                    probability[valid, task],
                )
            )

    return (
        float(
            np.mean(
                endpoint_scores
            )
        )
        if endpoint_scores
        else np.nan
    )


def thresholds_from_probabilities(
    y_true,
    mask,
    probability,
    criterion="balanced_accuracy",
):

    thresholds = np.full(
        y_true.shape[1],
        0.5,
        dtype=float,
    )

    for task in range(
        y_true.shape[1]
    ):

        valid = (
            mask[:, task].astype(bool)
            & np.isfinite(
                probability[:, task]
            )
        )

        y_task = (
            y_true[valid, task]
            .astype(int)
        )

        p_task = (
            probability[valid, task]
        )

        if (
            len(
                np.unique(
                    y_task
                )
            )
            == 2
        ):

            thresholds[task] = (
                optimize_threshold(
                    y_task,
                    p_task,
                    criterion=criterion,
                )
            )

    return thresholds


def evaluate_multitask(
    y_true,
    mask,
    probability,
    thresholds=None,
    endpoints=None,
) -> pd.DataFrame:

    endpoints = (
        endpoints
        or [
            f"Task_{index}"
            for index
            in range(
                y_true.shape[1]
            )
        ]
    )

    if thresholds is None:

        thresholds = np.full(
            y_true.shape[1],
            0.5,
        )

    rows = []

    for task, endpoint in enumerate(
        endpoints
    ):

        valid = (
            mask[:, task].astype(bool)
            & np.isfinite(
                probability[:, task]
            )
        )

        y_task = (
            y_true[valid, task]
            .astype(int)
        )

        p_task = (
            probability[valid, task]
        )

        if len(y_task) == 0:
            continue

        prediction = (
            p_task
            >= thresholds[task]
        ).astype(int)

        rows.append({
            "Endpoint": endpoint,
            "N": len(y_task),
            "Positive": int(
                y_task.sum()
            ),
            "Negative": int(
                (1 - y_task).sum()
            ),
            "AUC": safe_auc(
                y_task,
                p_task,
            ),
            "Accuracy": (
                accuracy_score(
                    y_task,
                    prediction,
                )
            ),
            "PR_AUC": (
                safe_average_precision(
                    y_task,
                    p_task,
                )
            ),
            "Balanced_Accuracy": (
                balanced_accuracy_score(
                    y_task,
                    prediction,
                )
                if len(
                    np.unique(
                        y_task
                    )
                ) == 2
                else np.nan
            ),
            "F1": f1_score(
                y_task,
                prediction,
                zero_division=0,
            ),
            "MCC": (
                matthews_corrcoef(
                    y_task,
                    prediction,
                )
                if len(
                    np.unique(
                        prediction
                    )
                ) > 1
                else 0.0
            ),
            "Threshold": (
                thresholds[task]
            ),
        })

    return pd.DataFrame(
        rows
    )


def cross_fitted_thresholds_and_accuracy(
    y_true,
    mask,
    probability,
    fold_ids,
):

    unique_folds = sorted(
        np.unique(
            fold_ids
        )
    )

    fold_accuracy = []

    cross_fitted_thresholds = (
        np.full(
            (
                len(
                    unique_folds
                ),
                y_true.shape[1],
            ),
            0.5,
            dtype=float,
        )
    )

    for fold in unique_folds:

        training_rows = (
            fold_ids != fold
        )

        validation_rows = (
            fold_ids == fold
        )

        thresholds = (
            thresholds_from_probabilities(
                y_true[training_rows],
                mask[training_rows],
                probability[training_rows],
                criterion="balanced_accuracy",
            )
        )

        cross_fitted_thresholds[
            fold
        ] = thresholds

        validation_report = (
            evaluate_multitask(
                y_true[validation_rows],
                mask[validation_rows],
                probability[validation_rows],
                thresholds=thresholds,
                endpoints=TARGET_COLS,
            )
        )

        fold_accuracy.append(
            float(
                validation_report[
                    "Accuracy"
                ].mean()
            )
        )

    final_thresholds = (
        thresholds_from_probabilities(
            y_true,
            mask,
            probability,
            criterion="balanced_accuracy",
        )
    )

    return (
        np.asarray(
            fold_accuracy,
            dtype=float,
        ),
        final_thresholds,
        cross_fitted_thresholds,
    )


def finalize_oof_metrics(
    y_true,
    mask,
    probability,
    fold_ids,
):

    fold_auc = (
        fold_level_auc(
            y_true,
            mask,
            probability,
            fold_ids,
        )
    )

    (
        fold_accuracy,
        final_thresholds,
        cross_fitted_thresholds,
    ) = (
        cross_fitted_thresholds_and_accuracy(
            y_true,
            mask,
            probability,
            fold_ids,
        )
    )

    return {
        "fold_auc": fold_auc,
        "fold_accuracy": (
            fold_accuracy
        ),
        "final_thresholds": (
            final_thresholds
        ),
        "cross_fitted_thresholds": (
            cross_fitted_thresholds
        ),
    }

## 40. Model 4 architecture

In [ ]:
if PYG_AVAILABLE:

    class LeakageSafeGPSRGCNToxKG(
        nn.Module
    ):

        def __init__(
            self,
            n_node_ids,
            n_node_types,
            n_relations,
            structural_dimension,
            n_tasks,
            hidden_dimension=192,
            embedding_dimension=256,
            n_layers=3,
            n_heads=4,
            dropout=0.25,
        ):

            super().__init__()

            self.n_relations = max(
                n_relations,
                1,
            )

            self.node_id_embedding = (
                nn.Embedding(
                    max(
                        n_node_ids,
                        2,
                    ),
                    hidden_dimension,
                )
            )

            self.node_type_embedding = (
                nn.Embedding(
                    n_node_types,
                    hidden_dimension,
                )
            )

            self.relation_embedding = (
                nn.Embedding(
                    self.n_relations,
                    hidden_dimension,
                )
            )

            self.structural_projection = (
                nn.Sequential(
                    nn.Linear(
                        structural_dimension,
                        hidden_dimension,
                    ),
                    nn.LayerNorm(
                        hidden_dimension
                    ),
                    nn.GELU(),
                )
            )

            self.gps_layers = (
                nn.ModuleList()
            )

            self.rgcn_layers = (
                nn.ModuleList()
            )

            self.hybrid_gates = (
                nn.ModuleList()
            )

            self.hybrid_norms = (
                nn.ModuleList()
            )


            for _ in range(
                n_layers
            ):

                local_mlp = (
                    nn.Sequential(
                        nn.Linear(
                            hidden_dimension,
                            hidden_dimension
                            * 2,
                        ),
                        nn.LeakyReLU(
                            0.1
                        ),
                        nn.Linear(
                            hidden_dimension
                            * 2,
                            hidden_dimension,
                        ),
                    )
                )

                local_conv = (
                    GINEConv(
                        local_mlp,
                        edge_dim=(
                            hidden_dimension
                        ),
                    )
                )

                self.gps_layers.append(
                    GPSConv(
                        channels=(
                            hidden_dimension
                        ),
                        conv=(
                            local_conv
                        ),
                        heads=(
                            n_heads
                        ),
                        dropout=(
                            dropout
                        ),
                        attn_type=(
                            "multihead"
                        ),
                    )
                )

                self.rgcn_layers.append(
                    RGCNConv(
                        in_channels=(
                            hidden_dimension
                        ),
                        out_channels=(
                            hidden_dimension
                        ),
                        num_relations=(
                            self.n_relations
                        ),
                        num_bases=min(
                            16,
                            self.n_relations,
                        ),
                    )
                )

                self.hybrid_gates.append(
                    nn.Sequential(
                        nn.Linear(
                            hidden_dimension
                            * 2,
                            hidden_dimension,
                        ),
                        nn.GELU(),
                        nn.Linear(
                            hidden_dimension,
                            1,
                        ),
                        nn.Sigmoid(),
                    )
                )

                self.hybrid_norms.append(
                    nn.LayerNorm(
                        hidden_dimension
                    )
                )


            self.final_norm = (
                nn.LayerNorm(
                    hidden_dimension
                )
            )

            self.embedding_head = (
                nn.Sequential(
                    nn.Linear(
                        hidden_dimension,
                        embedding_dimension,
                    ),
                    nn.LayerNorm(
                        embedding_dimension
                    ),
                    nn.LeakyReLU(
                        0.1
                    ),
                    nn.Dropout(
                        dropout
                    ),
                )
            )

            self.output_head = (
                nn.Linear(
                    embedding_dimension,
                    n_tasks,
                )
            )


        def forward(
            self,
            batch,
        ):

            node_hidden = (
                self.node_id_embedding(
                    batch.node_id
                )
                + self.node_type_embedding(
                    batch.node_type
                )
            )


            root_positions = (
                batch.ptr[
                    :-1
                ]
            )


            root_structural = (
                self.structural_projection(
                    batch.root_structural
                )
            )


            node_hidden[
                root_positions
            ] = (
                node_hidden[
                    root_positions
                ]
                + root_structural
            )


            edge_hidden = (
                self.relation_embedding(
                    batch.edge_type
                )
            )


            for (
                gps_layer,
                rgcn_layer,
                gate_layer,
                norm_layer,
            ) in zip(
                self.gps_layers,
                self.rgcn_layers,
                self.hybrid_gates,
                self.hybrid_norms,
            ):

                gps_hidden = (
                    gps_layer(
                        node_hidden,
                        batch.edge_index,
                        batch.batch,
                        edge_attr=(
                            edge_hidden
                        ),
                    )
                )

                rgcn_hidden = (
                    rgcn_layer(
                        node_hidden,
                        batch.edge_index,
                        batch.edge_type,
                    )
                )

                gate = (
                    gate_layer(
                        torch.cat(
                            [
                                gps_hidden,
                                rgcn_hidden,
                            ],
                            dim=-1,
                        )
                    )
                )

                node_hidden = (
                    norm_layer(
                        gate
                        * gps_hidden
                        + (
                            1
                            - gate
                        )
                        * rgcn_hidden
                    )
                )


            root_hidden = (
                self.final_norm(
                    node_hidden[
                        root_positions
                    ]
                )
            )


            embedding = (
                self.embedding_head(
                    root_hidden
                )
            )


            logits = (
                self.output_head(
                    embedding
                )
            )


            return (
                logits,
                embedding,
            )

## 41. Model 4 — 5-fold training

In [ ]:
def prepare_kg_graphs_with_structural_features(
    global_indices,
    structural_features,
):

    graphs = []


    for global_index, feature in zip(
        global_indices,
        structural_features,
    ):

        if (
            global_index
            not in KG_LOCAL_GRAPHS
        ):

            continue


        graph = (
            KG_LOCAL_GRAPHS[
                int(
                    global_index
                )
            ]
            .clone()
        )


        graph.root_structural = (
            torch.tensor(
                feature,
                dtype=torch.float32,
            )
            .view(
                1,
                -1,
            )
        )


        graphs.append(
            graph
        )


    return graphs


def train_gps_kg_fold(
    training_graphs,
    validation_graphs,
    structural_dimension,
    seed,
):

    seed_everything(
        seed
    )


    model = (
        LeakageSafeGPSRGCNToxKG(
            n_node_ids=(
                len(
                    KG_NODE_ID_VOCAB
                )
                + 1
            ),
            n_node_types=4,
            n_relations=max(
                len(
                    KG_RELATION_VOCAB
                ),
                1,
            ),
            structural_dimension=(
                structural_dimension
            ),
            n_tasks=len(
                TARGET_COLS
            ),
            hidden_dimension=(
                160
                if LOW_MEMORY
                else 256
            ),
            embedding_dimension=256,
            n_layers=(
                2
                if LOW_MEMORY
                else 4
            ),
            n_heads=4,
            dropout=0.30,
        )
        .to(
            DEVICE
        )
    )


    training_loader = (
        PyGDataLoader(
            training_graphs,
            batch_size=(
                max(
                    4,
                    CFG[
                        "batch_size"
                    ]
                    // 2,
                )
            ),
            shuffle=True,
            num_workers=0,
        )
    )


    validation_loader = (
        PyGDataLoader(
            validation_graphs,
            batch_size=(
                max(
                    4,
                    CFG[
                        "batch_size"
                    ]
                    // 2,
                )
            ),
            shuffle=False,
            num_workers=0,
        )
    )


    y_train = np.vstack([
        graph.y.numpy()
        for graph
        in training_graphs
    ])

    mask_train = np.vstack([
        graph.mask.numpy()
        for graph
        in training_graphs
    ])


    positive_weight, negative_weight = (
        effective_number_weights(
            torch.tensor(
                y_train,
                device=DEVICE,
            ),
            torch.tensor(
                mask_train,
                device=DEVICE,
            ),
        )
    )


    criterion = (
        MaskedAsymmetricLoss(
            positive_weight,
            negative_weight,
            gamma_positive=1.0,
            gamma_negative=4.0,
            clip=0.05,
        )
    )


    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=4e-4,
            weight_decay=2e-5,
        )
    )


    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=(
                CFG[
                    "epochs"
                ]
            ),
        )
    )


    scaler = make_grad_scaler()

    best_auc = -np.inf
    best_state = None
    wait = 0


    validation_sorted = sorted(
        validation_graphs,
        key=lambda graph: int(
            graph
            .sample_idx
            .item()
        ),
    )


    y_validation = np.vstack([
        graph.y.numpy()
        for graph
        in validation_sorted
    ])

    mask_validation = np.vstack([
        graph.mask.numpy()
        for graph
        in validation_sorted
    ])


    for epoch in range(
        CFG[
            "epochs"
        ]
    ):

        model.train()

        for batch in training_loader:

            batch = batch.to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )


            with amp_context():

                logits, _ = (
                    model(
                        batch
                    )
                )

                loss = criterion(
                    logits,
                    batch.y,
                    batch.mask,
                )


            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            scaler.step(
                optimizer
            )

            scaler.update()


        scheduler.step()


        (
            _,
            validation_probability,
            _,
        ) = graph_predict(
            model,
            validation_loader,
        )


        validation_auc = (
            mean_masked_auc(
                y_validation,
                mask_validation,
                validation_probability,
            )
        )


        if (
            validation_auc
            > best_auc
            + 1e-5
        ):

            best_auc = (
                validation_auc
            )

            best_state = {
                key: value
                .detach()
                .cpu()
                .clone()

                for key, value
                in model
                .state_dict()
                .items()
            }

            wait = 0

        else:

            wait += 1


        if (
            wait
            >= CFG[
                "patience"
            ]
        ):

            break


    if best_state is not None:

        model.load_state_dict(
            best_state
        )


    return (
        model,
        best_auc,
    )

In [ ]:
def run_gps_toxkg_cv():

    if (
        not PYG_AVAILABLE
        or not KG_LOCAL_GRAPHS
        or KG_GRAPH_COVERAGE < MIN_KG_GRAPH_COVERAGE
    ):

        print(
            "Model 4 skipped: leakage-safe KG coverage gate was not passed."
        )

        return None


    model_name = (
        "Model_4_GPS_RGCN_ToxKG"
    )


    oof_probability = np.full(
        (
            len(
                train_df
            ),
            len(
                TARGET_COLS
            ),
        ),
        np.nan,
        dtype=np.float32,
    )

    oof_embedding = np.full(
        (
            len(
                train_df
            ),
            256,
        ),
        np.nan,
        dtype=np.float32,
    )

    test_fold_probabilities = []
    test_fold_embeddings = []

    test_position = {
        global_index: position
        for position, global_index
        in enumerate(
            test_idx
        )
    }


    for fold in range(
        N_CV_FOLDS
    ):

        print(
            f"\nModel 4 — Fold {fold+1}/{N_CV_FOLDS}"
        )


        training_local_all = np.where(
            CV_FOLD_IDS
            != fold
        )[0]

        validation_local_all = np.where(
            CV_FOLD_IDS
            == fold
        )[0]


        training_global_all = (
            train_df[
                "global_index"
            ].values[
                training_local_all
            ]
        )

        validation_global_all = (
            train_df[
                "global_index"
            ].values[
                validation_local_all
            ]
        )


        training_mask = np.isin(
            training_global_all,
            KG_MATCHED_GLOBAL,
        )

        validation_mask = np.isin(
            validation_global_all,
            KG_MATCHED_GLOBAL,
        )

        test_global_kg = np.array([
            index
            for index
            in test_idx
            if index
            in KG_LOCAL_GRAPHS
        ], dtype=int)


        training_local = (
            training_local_all[
                training_mask
            ]
        )

        validation_local = (
            validation_local_all[
                validation_mask
            ]
        )

        training_global = (
            training_global_all[
                training_mask
            ]
        )

        validation_global = (
            validation_global_all[
                validation_mask
            ]
        )


        if (
            len(
                training_global
            )
            < 100
            or len(
                validation_global
            )
            < 20
            or len(
                test_global_kg
            )
            == 0
        ):

            print(
                "Insufficient KG coverage in this fold; fold skipped."
            )

            continue


        processor = (
            FoldLocalFeatureProcessor(
                max_svd=(
                    192
                    if LOW_MEMORY
                    else 384
                ),
                descriptor_pca_variance=0.995,
                random_state=(
                    PRIMARY_SEED
                    + fold
                ),
            )
        )


        x_train = (
            processor.fit_transform(
                X_FP[
                    training_global
                ],
                X_DESC[
                    training_global
                ],
            )
        )

        x_validation = (
            processor.transform(
                X_FP[
                    validation_global
                ],
                X_DESC[
                    validation_global
                ],
            )
        )

        x_test = (
            processor.transform(
                X_FP[
                    test_global_kg
                ],
                X_DESC[
                    test_global_kg
                ],
            )
        )


        training_graphs = (
            prepare_kg_graphs_with_structural_features(
                training_global,
                x_train,
            )
        )

        validation_graphs = (
            prepare_kg_graphs_with_structural_features(
                validation_global,
                x_validation,
            )
        )

        test_graphs = (
            prepare_kg_graphs_with_structural_features(
                test_global_kg,
                x_test,
            )
        )


        model, best_auc = (
            train_gps_kg_fold(
                training_graphs,
                validation_graphs,
                structural_dimension=(
                    x_train.shape[1]
                ),
                seed=(
                    PRIMARY_SEED
                    + fold
                ),
            )
        )


        validation_loader = (
            PyGDataLoader(
                validation_graphs,
                batch_size=(
                    max(
                        4,
                        CFG[
                            "batch_size"
                        ]
                        // 2,
                    )
                ),
                shuffle=False,
            )
        )

        test_loader = (
            PyGDataLoader(
                test_graphs,
                batch_size=(
                    max(
                        4,
                        CFG[
                            "batch_size"
                        ]
                        // 2,
                    )
                ),
                shuffle=False,
            )
        )


        (
            validation_indices,
            validation_probability,
            validation_embedding,
        ) = graph_predict(
            model,
            validation_loader,
        )


        (
            test_indices,
            test_probability,
            test_embedding,
        ) = graph_predict(
            model,
            test_loader,
        )


        global_to_train_local = {
            global_index: local_index

            for local_index, global_index
            in enumerate(
                train_df[
                    "global_index"
                ].values
            )
        }


        for (
            global_index,
            probability,
            embedding,
        ) in zip(
            validation_indices,
            validation_probability,
            validation_embedding,
        ):

            local_index = (
                global_to_train_local[
                    int(
                        global_index
                    )
                ]
            )

            oof_probability[
                local_index
            ] = probability

            oof_embedding[
                local_index
            ] = embedding


        fold_test_probability = np.full(
            (
                len(
                    test_idx
                ),
                len(
                    TARGET_COLS
                ),
            ),
            np.nan,
            dtype=np.float32,
        )

        fold_test_embedding = np.full(
            (
                len(
                    test_idx
                ),
                256,
            ),
            np.nan,
            dtype=np.float32,
        )


        for (
            global_index,
            probability,
            embedding,
        ) in zip(
            test_indices,
            test_probability,
            test_embedding,
        ):

            position = (
                test_position[
                    int(
                        global_index
                    )
                ]
            )

            fold_test_probability[
                position
            ] = probability

            fold_test_embedding[
                position
            ] = embedding


        test_fold_probabilities.append(
            fold_test_probability
        )

        test_fold_embeddings.append(
            fold_test_embedding
        )


        torch.save({
            "state_dict": (
                model.state_dict()
            ),
            "fold": fold,
            "best_auc": best_auc,
            "processor": processor,
            "safe_relations": (
                sorted(
                    KG_RELATION_VOCAB
                    .keys()
                )
            ),
        },
        DIRS["models"]
        / f"{model_name}_fold{fold}.pt")


        print(
            "Best validation AUC:",
            f"{best_auc:.4f}",
        )


        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    if not test_fold_probabilities:

        print(
            "Model 4 produced no valid fold."
        )

        return None


    test_probability = np.nanmean(
        np.stack(
            test_fold_probabilities
        ),
        axis=0,
    )

    test_embedding = np.nanmean(
        np.stack(
            test_fold_embeddings
        ),
        axis=0,
    )


    metric_bundle = (
        finalize_oof_metrics(
            Y_TRAIN,
            M_TRAIN,
            oof_probability,
            CV_FOLD_IDS,
        )
    )


    train_coverage = float(
        np.isfinite(
            oof_probability
        ).any(
            axis=1
        ).mean()
    )

    test_coverage = float(
        np.isfinite(
            test_probability
        ).any(
            axis=1
        ).mean()
    )


    save_model_artifact(
        name=model_name,
        oof_probability=oof_probability,
        test_probability=test_probability,
        oof_embedding=oof_embedding,
        test_embedding=test_embedding,
        fold_auc=(
            metric_bundle[
                "fold_auc"
            ]
        ),
        fold_accuracy=(
            metric_bundle[
                "fold_accuracy"
            ]
        ),
        metadata={
            "model_number": 4,
            "description": (
                "Leakage-safe GPS–R-GCN hybrid with ToxKG mechanism context"
            ),
            "cv_folds": N_CV_FOLDS,
            "train_coverage": (
                train_coverage
            ),
            "test_coverage": (
                test_coverage
            ),
            "safe_relation_allowlist": (
                sorted(
                    SAFE_RELATION_ALLOWLIST
                )
            ),
        },
    )


    np.save(
        ARTIFACT_DIR
        / f"{model_name}_thresholds.npy",
        metric_bundle[
            "final_thresholds"
        ],
    )


    print(
        f"KG coverage — Train: {train_coverage:.1%}, "
        f"Test: {test_coverage:.1%}"
    )


    return {
        "oof_probability": (
            oof_probability
        ),
        "test_probability": (
            test_probability
        ),
        "oof_embedding": (
            oof_embedding
        ),
        "test_embedding": (
            test_embedding
        ),
        "train_coverage": (
            train_coverage
        ),
        "test_coverage": (
            test_coverage
        ),
        **metric_bundle,
    }


if RUN_MODEL_4_GPS_TOXKG:

    MODEL_4_RESULT = (
        run_gps_toxkg_cv()
    )

else:

    print(
        "Model 4 disabled."
    )

## 42. Automated label-permutation leakage check

In [ ]:
def label_permutation_check(
    max_training_rows=1000,
    seed=13,
):

    random_generator = (
        np.random.default_rng(
            seed
        )
    )


    selected_global = (
        train_df[
            "global_index"
        ]
        .values[
            :max_training_rows
        ]
    )


    processor = (
        FoldLocalFeatureProcessor(
            max_svd=64,
            descriptor_pca_variance=0.99,
            random_state=seed,
        )
    )


    features = (
        processor.fit_transform(
            X_FP[
                selected_global
            ],
            X_DESC[
                selected_global
            ],
        )
    )


    endpoint_scores = []


    for task in range(
        len(
            TARGET_COLS
        )
    ):

        observed = (
            M[
                selected_global,
                task
            ]
            .astype(
                bool
            )
        )

        if (
            observed.sum()
            < 60
        ):

            continue


        y_task = (
            Y_FILLED[
                selected_global,
                task
            ][
                observed
            ]
            .astype(
                int
            )
            .copy()
        )


        if (
            len(
                np.unique(
                    y_task
                )
            )
            < 2
        ):

            continue


        x_task = features[
            observed
        ]


        random_generator.shuffle(
            y_task
        )


        split_point = int(
            0.8
            * len(
                y_task
            )
        )


        classifier = SVC(
            kernel="rbf",
            C=2.0,
            gamma="scale",
            class_weight="balanced",
            probability=False,
            random_state=seed,
        )


        classifier.fit(
            x_task[
                :split_point
            ],
            y_task[
                :split_point
            ],
        )


        score = (
            classifier
            .decision_function(
                x_task[
                    split_point:
                ]
            )
        )


        y_test = (
            y_task[
                split_point:
            ]
        )


        if (
            len(
                np.unique(
                    y_test
                )
            )
            == 2
        ):

            endpoint_scores.append(
                roc_auc_score(
                    y_test,
                    score,
                )
            )


    return (
        float(
            np.mean(
                endpoint_scores
            )
        )
        if endpoint_scores
        else np.nan
    )


PERMUTATION_AUC = (
    label_permutation_check()
)

print(
    "Label-permutation mean AUC:",
    f"{PERMUTATION_AUC:.3f}",
)


if (
    np.isfinite(
        PERMUTATION_AUC
    )
    and PERMUTATION_AUC
    > 0.62
):

    raise RuntimeError(
        "Permutation AUC suspiciously high. "
        "Stop and audit leakage."
    )

else:

    print(
        "Permutation leakage check passed."
    )

# Part VIII — Model 5: Cross-Fitted Endpoint-Wise Ensemble

Model 5 is the final black-box predictor. It does not concatenate high-dimensional branch embeddings or train another unstable neural fusion network.

Instead it:

1. evaluates every completed base model from OOF predictions,
2. removes low-coverage branches,
3. applies a performance-and-diversity gate,
4. learns endpoint-specific non-negative weights using only training-side OOF predictions,
5. recreates the weight learning cross-fitted for unbiased CV estimates,
6. freezes the final weights before the untouched test set is evaluated.

This design directly addresses the previous run, where the neural fusion branch had much lower CV AUC than the individual branches.


## 49. Base-model evidence gate

Candidate base models:

1. Hierarchical DeepTox-Residual DNN
2. Optimized AttentiveFP GNN
3. ChemBERTa Transformer
4. Leakage-Safe GPS–R-GCN ToxKG Hybrid

A branch is not included merely because it completed. It must pass minimum coverage and either remain close to the strongest OOF model or contribute sufficiently different errors.


In [ ]:
FINAL_MODEL_NAMES = [
    "Model_1_DeepTox_Residual",
    "Model_2_AttentiveFP",
    "Model_3_ChemBERTa",
    "Model_4_GPS_RGCN_ToxKG",
]


def collect_prediction_artifacts(
    model_names,
):

    artifacts = {}


    for model_name in (
        model_names
    ):

        artifact_path = (
            ARTIFACT_DIR
            / f"{model_name}.npz"
        )


        if not artifact_path.exists():

            continue


        artifact = (
            load_model_artifact(
                model_name
            )
        )


        if (
            "oof_prob"
            in artifact
            and "test_prob"
            in artifact
        ):

            artifacts[
                model_name
            ] = artifact


    return artifacts


def artifact_mean_oof_auc(
    artifact,
):

    task_auc = []


    for task in range(
        len(
            TARGET_COLS
        )
    ):

        valid = (
            M_TRAIN[
                :,
                task
            ].astype(
                bool
            )
            & np.isfinite(
                artifact[
                    "oof_prob"
                ][
                    :,
                    task
                ]
            )
        )


        y_task = (
            Y_TRAIN[
                valid,
                task
            ]
        )

        p_task = (
            artifact[
                "oof_prob"
            ][
                valid,
                task
            ]
        )


        if (
            valid.sum() > 0
            and len(
                np.unique(
                    y_task
                )
            )
            == 2
        ):

            task_auc.append(
                roc_auc_score(
                    y_task,
                    p_task,
                )
            )


    return (
        float(
            np.mean(
                task_auc
            )
        )
        if task_auc
        else np.nan
    )


def mean_prediction_correlation(
    first,
    second,
):

    correlations = []


    for task in range(
        len(
            TARGET_COLS
        )
    ):

        valid = (
            M_TRAIN[
                :,
                task
            ].astype(
                bool
            )
            & np.isfinite(
                first[
                    :,
                    task
                ]
            )
            & np.isfinite(
                second[
                    :,
                    task
                ]
            )
        )


        if valid.sum() < 20:

            continue


        first_task = (
            first[
                valid,
                task
            ]
        )

        second_task = (
            second[
                valid,
                task
            ]
        )


        if (
            np.std(
                first_task
            )
            < 1e-12
            or np.std(
                second_task
            )
            < 1e-12
        ):

            continue


        correlations.append(
            np.corrcoef(
                first_task,
                second_task,
            )[
                0,
                1
            ]
        )


    return (
        float(
            np.nanmean(
                correlations
            )
        )
        if correlations
        else 1.0
    )


ALL_PREDICTION_ARTIFACTS = (
    collect_prediction_artifacts(
        FINAL_MODEL_NAMES
    )
)


if (
    len(
        ALL_PREDICTION_ARTIFACTS
    )
    < 2
):

    raise RuntimeError(
        "Model 5 requires at least two completed base models."
    )


gate_rows = []


for (
    model_name,
    artifact,
) in (
    ALL_PREDICTION_ARTIFACTS
    .items()
):

    oof_coverage = float(
        np.isfinite(
            artifact[
                "oof_prob"
            ]
        )
        .any(
            axis=1
        )
        .mean()
    )

    test_coverage = float(
        np.isfinite(
            artifact[
                "test_prob"
            ]
        )
        .any(
            axis=1
        )
        .mean()
    )

    gate_rows.append({
        "Model": model_name,
        "OOF Mean AUC": (
            artifact_mean_oof_auc(
                artifact
            )
        ),
        "OOF Coverage": (
            oof_coverage
        ),
        "Test Coverage": (
            test_coverage
        ),
    })


MODEL_GATE_TABLE = (
    pd.DataFrame(
        gate_rows
    )
    .sort_values(
        "OOF Mean AUC",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


best_model_name = (
    MODEL_GATE_TABLE
    .iloc[
        0
    ][
        "Model"
    ]
)


best_auc = float(
    MODEL_GATE_TABLE
    .iloc[
        0
    ][
        "OOF Mean AUC"
    ]
)


best_prediction = (
    ALL_PREDICTION_ARTIFACTS[
        best_model_name
    ][
        "oof_prob"
    ]
)


MODEL_GATE_TABLE[
    "Mean correlation vs best"
] = [
    (
        1.0
        if model_name
        == best_model_name
        else mean_prediction_correlation(
            best_prediction,
            ALL_PREDICTION_ARTIFACTS[
                model_name
            ][
                "oof_prob"
            ],
        )
    )

    for model_name
    in MODEL_GATE_TABLE[
        "Model"
    ]
]


MODEL_GATE_TABLE[
    "Pass gate"
] = (
    (
        MODEL_GATE_TABLE[
            "OOF Coverage"
        ]
        >= MIN_MODEL_COVERAGE
    )
    & (
        (
            MODEL_GATE_TABLE[
                "OOF Mean AUC"
            ]
            >= (
                best_auc
                - ENSEMBLE_MAX_AUC_GAP
            )
        )
        | (
            (
                MODEL_GATE_TABLE[
                    "Mean correlation vs best"
                ]
                < ENSEMBLE_DIVERSITY_CORR
            )
            & (
                MODEL_GATE_TABLE[
                    "OOF Mean AUC"
                ]
                >= (
                    best_auc
                    - 0.070
                )
            )
        )
    )
)


MODEL_GATE_TABLE.loc[
    MODEL_GATE_TABLE[
        "Model"
    ]
    == best_model_name,
    "Pass gate",
] = True


selected_model_names = (
    MODEL_GATE_TABLE.loc[
        MODEL_GATE_TABLE[
            "Pass gate"
        ],
        "Model",
    ]
    .tolist()
)


# Keep at least two models for ensemble learning.
if len(
    selected_model_names
) < 2:

    selected_model_names = (
        MODEL_GATE_TABLE[
            "Model"
        ]
        .head(
            2
        )
        .tolist()
    )


PREDICTION_ARTIFACTS = {
    name: (
        ALL_PREDICTION_ARTIFACTS[
            name
        ]
    )

    for name
    in selected_model_names
}


MODEL_GATE_TABLE.to_csv(
    DIRS[
        "outputs"
    ]
    / "07_model_evidence_gate.csv",
    index=False,
)


display(
    MODEL_GATE_TABLE
    .style
    .format({
        "OOF Mean AUC": "{:.4f}",
        "OOF Coverage": "{:.1%}",
        "Test Coverage": "{:.1%}",
        "Mean correlation vs best": "{:.3f}",
    })
)


print(
    "Model 5 selected branches:",
    list(
        PREDICTION_ARTIFACTS
    ),
)


## 50. NaN-safe evaluation utilities for partial KG coverage

In [ ]:
def evaluate_multitask_nan_safe(
    y_true,
    mask,
    probability,
    thresholds=None,
    endpoints=None,
):

    endpoints = (
        endpoints
        or [
            f"Task_{index}"
            for index
            in range(
                y_true.shape[1]
            )
        ]
    )


    if thresholds is None:

        thresholds = np.full(
            y_true.shape[1],
            0.5,
        )


    rows = []


    for task, endpoint in enumerate(
        endpoints
    ):

        valid = (
            mask[
                :,
                task
            ]
            .astype(
                bool
            )
            & np.isfinite(
                probability[
                    :,
                    task
                ]
            )
        )


        y_task = (
            y_true[
                valid,
                task
            ]
            .astype(
                int
            )
        )

        p_task = (
            probability[
                valid,
                task
            ]
        )


        if len(
            y_task
        ) == 0:

            continue


        prediction = (
            p_task
            >= thresholds[
                task
            ]
        ).astype(
            int
        )


        rows.append({
            "Endpoint": endpoint,
            "N": len(
                y_task
            ),
            "Positive": int(
                y_task.sum()
            ),
            "Negative": int(
                (
                    1
                    - y_task
                ).sum()
            ),
            "Coverage": (
                valid.mean()
            ),
            "AUC": safe_auc(
                y_task,
                p_task,
            ),
            "Accuracy": (
                accuracy_score(
                    y_task,
                    prediction,
                )
            ),
            "PR_AUC": (
                safe_average_precision(
                    y_task,
                    p_task,
                )
            ),
            "Balanced_Accuracy": (
                balanced_accuracy_score(
                    y_task,
                    prediction,
                )
                if len(
                    np.unique(
                        y_task
                    )
                )
                == 2
                else np.nan
            ),
            "F1": f1_score(
                y_task,
                prediction,
                zero_division=0,
            ),
            "MCC": (
                matthews_corrcoef(
                    y_task,
                    prediction,
                )
                if len(
                    np.unique(
                        prediction
                    )
                )
                > 1
                else 0.0
            ),
            "Threshold": (
                thresholds[
                    task
                ]
            ),
        })


    return pd.DataFrame(
        rows
    )


def thresholds_from_probabilities_nan_safe(
    y_true,
    mask,
    probability,
    criterion="balanced_accuracy",
):

    thresholds = np.full(
        y_true.shape[1],
        0.5,
        dtype=float,
    )


    for task in range(
        y_true.shape[1]
    ):

        valid = (
            mask[
                :,
                task
            ]
            .astype(
                bool
            )
            & np.isfinite(
                probability[
                    :,
                    task
                ]
            )
        )


        y_task = (
            y_true[
                valid,
                task
            ]
            .astype(
                int
            )
        )

        p_task = (
            probability[
                valid,
                task
            ]
        )


        if (
            len(
                np.unique(
                    y_task
                )
            )
            == 2
        ):

            thresholds[
                task
            ] = optimize_threshold(
                y_task,
                p_task,
                criterion=(
                    criterion
                ),
            )


    return thresholds


def recompute_cv_statistics(
    probability,
):

    fold_auc = []

    fold_accuracy = []


    for fold in range(
        N_CV_FOLDS
    ):

        training_rows = (
            CV_FOLD_IDS
            != fold
        )

        validation_rows = (
            CV_FOLD_IDS
            == fold
        )


        thresholds = (
            thresholds_from_probabilities_nan_safe(
                Y_TRAIN[
                    training_rows
                ],
                M_TRAIN[
                    training_rows
                ],
                probability[
                    training_rows
                ],
                criterion=(
                    "balanced_accuracy"
                ),
            )
        )


        validation_report = (
            evaluate_multitask_nan_safe(
                Y_TRAIN[
                    validation_rows
                ],
                M_TRAIN[
                    validation_rows
                ],
                probability[
                    validation_rows
                ],
                thresholds=(
                    thresholds
                ),
                endpoints=(
                    TARGET_COLS
                ),
            )
        )


        fold_auc.append(
            float(
                validation_report[
                    "AUC"
                ].mean()
            )
        )


        fold_accuracy.append(
            float(
                validation_report[
                    "Accuracy"
                ].mean()
            )
        )


    final_thresholds = (
        thresholds_from_probabilities_nan_safe(
            Y_TRAIN,
            M_TRAIN,
            probability,
            criterion=(
                "balanced_accuracy"
            ),
        )
    )


    return (
        np.asarray(
            fold_auc
        ),
        np.asarray(
            fold_accuracy
        ),
        final_thresholds,
    )

## 51. Regularized endpoint-wise weight learning

For every endpoint, candidate weights include:

- the best single branch,
- equal weighting,
- sparse and dense Dirichlet mixtures.

The objective maximizes ROC-AUC while applying a small concentration penalty. During the cross-fitted CV estimate, weights for a validation fold are learned only from the other four folds. Final test weights are learned from all OOF predictions and then frozen.


In [ ]:
def rowwise_weighted_blend(
    probability_matrix,
    weights,
):

    finite = np.isfinite(
        probability_matrix
    )

    weighted_probability = np.where(
        finite,
        probability_matrix
        * weights[
            None,
            :
        ],
        0.0,
    ).sum(
        axis=1
    )


    available_weight = np.where(
        finite,
        weights[
            None,
            :
        ],
        0.0,
    ).sum(
        axis=1
    )


    return np.divide(
        weighted_probability,
        available_weight,
        out=np.full(
            len(
                probability_matrix
            ),
            0.5,
            dtype=float,
        ),
        where=(
            available_weight
            > 0
        ),
    )


def learn_endpoint_weights(
    y_true,
    probability_matrix,
    n_random=6000,
    seed=13,
):

    n_models = (
        probability_matrix
        .shape[1]
    )


    if n_models == 1:

        return np.array([
            1.0
        ])


    random_generator = (
        np.random.default_rng(
            seed
        )
    )


    candidate_weights = [
        np.ones(
            n_models
        ) / n_models
    ]


    for model_index in range(
        n_models
    ):

        one_hot_weight = np.zeros(
            n_models
        )

        one_hot_weight[
            model_index
        ] = 1.0

        candidate_weights.append(
            one_hot_weight
        )


    # Sparse and smooth candidate mixtures.
    candidate_weights.extend(
        random_generator.dirichlet(
            alpha=np.full(
                n_models,
                0.35,
            ),
            size=(
                n_random
                // 2
            ),
        )
    )

    candidate_weights.extend(
        random_generator.dirichlet(
            alpha=np.full(
                n_models,
                1.5,
            ),
            size=(
                n_random
                - n_random
                // 2
            ),
        )
    )


    # Deterministic pairwise blends improve reproducibility.
    if n_models >= 2:

        pair_grid = np.linspace(
            0.10,
            0.90,
            9,
        )


        for first_model in range(
            n_models
        ):

            for second_model in range(
                first_model + 1,
                n_models,
            ):

                for first_weight in (
                    pair_grid
                ):

                    weight = np.zeros(
                        n_models,
                        dtype=float,
                    )

                    weight[
                        first_model
                    ] = first_weight

                    weight[
                        second_model
                    ] = (
                        1.0
                        - first_weight
                    )

                    candidate_weights.append(
                        weight
                    )


    best_weight = (
        candidate_weights[
            0
        ]
    )

    best_objective = -np.inf


    for weight in (
        candidate_weights
    ):

        blended = (
            rowwise_weighted_blend(
                probability_matrix,
                weight,
            )
        )


        if (
            len(
                np.unique(
                    y_true
                )
            )
            != 2
        ):

            continue


        auc = roc_auc_score(
            y_true,
            blended,
        )


        # Very small regularization avoids unstable one-sample optima
        # without forcing weak models into the blend.
        concentration_penalty = (
            0.00025
            * np.sum(
                weight
                ** 2
            )
        )


        objective = (
            auc
            - concentration_penalty
        )


        if (
            objective
            > best_objective
        ):

            best_objective = (
                objective
            )

            best_weight = (
                weight.copy()
            )


    return (
        best_weight
        / best_weight.sum()
    )


def learn_weight_matrix(
    y_true,
    mask,
    prediction_artifacts,
    training_rows=None,
    seed=13,
):

    model_names = list(
        prediction_artifacts
    )

    n_tasks = (
        y_true.shape[1]
    )

    weight_matrix = np.zeros(
        (
            n_tasks,
            len(
                model_names
            ),
        ),
        dtype=float,
    )


    if training_rows is None:

        training_rows = np.ones(
            len(
                y_true
            ),
            dtype=bool,
        )


    for task in range(
        n_tasks
    ):

        task_probabilities = np.stack([
            prediction_artifacts[
                model_name
            ][
                "oof_prob"
            ][
                :,
                task
            ]

            for model_name
            in model_names
        ], axis=1)


        valid = (
            training_rows
            & mask[
                :,
                task
            ]
            .astype(
                bool
            )
            & np.isfinite(
                task_probabilities
            )
            .any(
                axis=1
            )
        )


        y_task = (
            y_true[
                valid,
                task
            ]
            .astype(
                int
            )
        )

        probability_task = (
            task_probabilities[
                valid
            ]
        )


        if (
            len(
                y_task
            )
            < 30
            or len(
                np.unique(
                    y_task
                )
            )
            != 2
        ):

            weight_matrix[
                task
            ] = (
                1.0
                / len(
                    model_names
                )
            )

            continue


        weight_matrix[
            task
        ] = (
            learn_endpoint_weights(
                y_task,
                probability_task,
                n_random=(
                    1500
                    if EXECUTION_MODE
                    == "QUICK"
                    else 6000
                ),
                seed=(
                    seed
                    + task
                ),
            )
        )


    return (
        model_names,
        weight_matrix,
    )


def blend_artifacts(
    prediction_artifacts,
    model_names,
    weight_matrix,
    key,
):

    n_rows = (
        next(
            iter(
                prediction_artifacts
                .values()
            )
        )[
            key
        ].shape[0]
    )

    n_tasks = (
        weight_matrix.shape[0]
    )

    blended = np.zeros(
        (
            n_rows,
            n_tasks,
        ),
        dtype=float,
    )


    for task in range(
        n_tasks
    ):

        task_probabilities = np.stack([
            prediction_artifacts[
                model_name
            ][
                key
            ][
                :,
                task
            ]

            for model_name
            in model_names
        ], axis=1)


        blended[
            :,
            task
        ] = (
            rowwise_weighted_blend(
                task_probabilities,
                weight_matrix[
                    task
                ],
            )
        )


    return blended

## 52. Build the unbiased cross-fitted OOF ensemble

In [ ]:
CROSS_FITTED_ENSEMBLE_OOF = np.full(
    (
        len(
            train_df
        ),
        len(
            TARGET_COLS
        ),
    ),
    np.nan,
    dtype=np.float32,
)


CROSS_FITTED_WEIGHT_MATRICES = []


for fold in range(
    N_CV_FOLDS
):

    training_rows = (
        CV_FOLD_IDS
        != fold
    )

    validation_rows = (
        CV_FOLD_IDS
        == fold
    )


    (
        fold_model_names,
        fold_weight_matrix,
    ) = learn_weight_matrix(
        Y_TRAIN,
        M_TRAIN,
        PREDICTION_ARTIFACTS,
        training_rows=(
            training_rows
        ),
        seed=(
            PRIMARY_SEED
            + 100
            + fold
        ),
    )


    fold_predictions = (
        blend_artifacts(
            PREDICTION_ARTIFACTS,
            fold_model_names,
            fold_weight_matrix,
            key="oof_prob",
        )
    )


    CROSS_FITTED_ENSEMBLE_OOF[
        validation_rows
    ] = fold_predictions[
        validation_rows
    ]


    CROSS_FITTED_WEIGHT_MATRICES.append(
        fold_weight_matrix
    )


print(
    "Cross-fitted ensemble OOF predictions তৈরি হয়েছে।"
)

## 53. Freeze final OOF weights and produce final test probabilities

In [ ]:
(
    ENSEMBLE_MODEL_NAMES,
    FINAL_ENSEMBLE_WEIGHTS,
) = learn_weight_matrix(
    Y_TRAIN,
    M_TRAIN,
    PREDICTION_ARTIFACTS,
    training_rows=None,
    seed=(
        PRIMARY_SEED
        + 999
    ),
)


FINAL_ENSEMBLE_TEST = (
    blend_artifacts(
        PREDICTION_ARTIFACTS,
        ENSEMBLE_MODEL_NAMES,
        FINAL_ENSEMBLE_WEIGHTS,
        key="test_prob",
    )
)


(
    ENSEMBLE_FOLD_AUC,
    ENSEMBLE_FOLD_ACCURACY,
    ENSEMBLE_THRESHOLDS,
) = recompute_cv_statistics(
    CROSS_FITTED_ENSEMBLE_OOF
)


np.savez_compressed(
    DIRS["outputs"]
    / "07_final_cross_fitted_ensemble.npz",
    oof_probability=(
        CROSS_FITTED_ENSEMBLE_OOF
    ),
    test_probability=(
        FINAL_ENSEMBLE_TEST
    ),
    weights=(
        FINAL_ENSEMBLE_WEIGHTS
    ),
    model_names=np.array(
        ENSEMBLE_MODEL_NAMES
    ),
    fold_auc=(
        ENSEMBLE_FOLD_AUC
    ),
    fold_accuracy=(
        ENSEMBLE_FOLD_ACCURACY
    ),
    thresholds=(
        ENSEMBLE_THRESHOLDS
    ),
)


print(
    "বাংলা বার্তা: Endpoint-wise ensemble weights "
    "OOF predictions থেকে freeze করা হয়েছে। "
    "Final test labels weight learning-এ ব্যবহার হয়নি।"
)

## 54. Ensemble weight visualization

In [ ]:
plt.figure(
    figsize=(
        max(
            10,
            1.4
            * len(
                ENSEMBLE_MODEL_NAMES
            ),
        ),
        7,
    )
)


weight_dataframe = (
    pd.DataFrame(
        FINAL_ENSEMBLE_WEIGHTS,
        index=(
            TARGET_COLS
        ),
        columns=(
            ENSEMBLE_MODEL_NAMES
        ),
    )
)


sns.heatmap(
    weight_dataframe,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    vmin=0,
    vmax=1,
)


plt.title(
    "Endpoint-wise OOF-Optimized Ensemble Weights",
    fontweight="bold",
)


plt.xlabel(
    "Model"
)


plt.ylabel(
    "Tox21 endpoint"
)


plt.tight_layout()


plt.savefig(
    DIRS["figures"]
    / "04_ensemble_weights.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()

## 55. Final evaluation policy

### Primary outputs

- **5-fold CV mean ROC-AUC ± SD**
- **5-fold CV mean Accuracy ± SD**
- **Frozen test mean ROC-AUC**
- **Frozen test mean Accuracy**
- **Bootstrap 95% CI for frozen-test mean ROC-AUC**

### Important interpretation

The previous run was CPU-only and skipped the KG model because coverage was only 8 molecules. This revision fixes that mapping path and removes the unstable neural fusion. A valid score is still determined by the data, split difficulty, GPU training depth and real generalization; `0.95` is not inserted into the output.


In [ ]:
TEST_Y = Y_FILLED[
    test_idx
]

TEST_M = M[
    test_idx
]


FINAL_TEST_REPORT = (
    evaluate_multitask_nan_safe(
        TEST_Y,
        TEST_M,
        FINAL_ENSEMBLE_TEST,
        thresholds=(
            ENSEMBLE_THRESHOLDS
        ),
        endpoints=(
            TARGET_COLS
        ),
    )
)


FINAL_TEST_REPORT.to_csv(
    DIRS["outputs"]
    / "08_final_test_per_endpoint.csv",
    index=False,
)


FINAL_TEST_MEAN_AUC = float(
    FINAL_TEST_REPORT[
        "AUC"
    ].mean()
)


FINAL_TEST_MEAN_ACCURACY = float(
    FINAL_TEST_REPORT[
        "Accuracy"
    ].mean()
)


FINAL_TEST_CI = (
    bootstrap_macro_auc(
        TEST_Y,
        TEST_M,
        FINAL_ENSEMBLE_TEST,
        n_bootstrap=(
            300
            if EXECUTION_MODE
            == "QUICK"
            else 1500
        ),
        seed=PRIMARY_SEED,
    )
)


print(
    "Final 5-fold CV Mean AUC:",
    f"{np.mean(ENSEMBLE_FOLD_AUC):.4f}",
    "±",
    f"{np.std(ENSEMBLE_FOLD_AUC):.4f}",
)


print(
    "Final 5-fold CV Mean Accuracy:",
    f"{np.mean(ENSEMBLE_FOLD_ACCURACY):.4f}",
    "±",
    f"{np.std(ENSEMBLE_FOLD_ACCURACY):.4f}",
)


print(
    "Frozen Test Mean AUC:",
    f"{FINAL_TEST_MEAN_AUC:.4f}",
)


print(
    "Frozen Test Mean Accuracy:",
    f"{FINAL_TEST_MEAN_ACCURACY:.4f}",
)


print(
    "Bootstrap 95% CI for Test Mean AUC:",
    f"[{FINAL_TEST_CI[0]:.4f}, "
    f"{FINAL_TEST_CI[2]:.4f}]",
)


display(
    FINAL_TEST_REPORT.round(
        4
    )
)

if FINAL_TEST_MEAN_AUC >= TARGET_MEAN_AUC:

    print(
        "বাংলা বার্তা: Target mean ROC-AUC অর্জিত হয়েছে। "
        "Confidence interval, endpoint support এবং leakage audit সহ result report করুন।"
    )

else:

    print(
        "বাংলা বার্তা: Target mean ROC-AUC এখনও অর্জিত হয়নি। "
        "Result পরিবর্তন বা test tuning করবেন না। "
        "প্রথমে CUDA training, KG coverage, weak-endpoint analysis এবং model gate report পরীক্ষা করুন।"
    )


# Part IX — Clean visualizations, result tables and reproducibility


## 56. Model leaderboard — 5-fold mean AUC and mean Accuracy

In [ ]:
leaderboard_rows = []

MODEL_CV_DETAILS = {}

for model_name in (
    ENSEMBLE_MODEL_NAMES
):

    artifact = (
        PREDICTION_ARTIFACTS[
            model_name
        ]
    )


    (
        fold_auc,
        fold_accuracy,
        final_thresholds,
    ) = recompute_cv_statistics(
        artifact[
            "oof_prob"
        ]
    )


    test_report = (
        evaluate_multitask_nan_safe(
            TEST_Y,
            TEST_M,
            artifact[
                "test_prob"
            ],
            thresholds=(
                final_thresholds
            ),
            endpoints=(
                TARGET_COLS
            ),
        )
    )


    oof_coverage = float(
        np.isfinite(
            artifact[
                "oof_prob"
            ]
        )
        .any(
            axis=1
        )
        .mean()
    )


    test_coverage = float(
        np.isfinite(
            artifact[
                "test_prob"
            ]
        )
        .any(
            axis=1
        )
        .mean()
    )


    leaderboard_rows.append({
        "Model": model_name,
        "CV Mean AUC": (
            np.nanmean(
                fold_auc
            )
        ),
        "CV AUC SD": (
            np.nanstd(
                fold_auc
            )
        ),
        "CV Mean Accuracy": (
            np.nanmean(
                fold_accuracy
            )
        ),
        "CV Accuracy SD": (
            np.nanstd(
                fold_accuracy
            )
        ),
        "Test Mean AUC": (
            test_report[
                "AUC"
            ].mean()
        ),
        "Test Mean Accuracy": (
            test_report[
                "Accuracy"
            ].mean()
        ),
        "OOF Coverage": (
            oof_coverage
        ),
        "Test Coverage": (
            test_coverage
        ),
    })


    MODEL_CV_DETAILS[
        model_name
    ] = {
        "fold_auc": (
            fold_auc
        ),
        "fold_accuracy": (
            fold_accuracy
        ),
        "thresholds": (
            final_thresholds
        ),
        "test_report": (
            test_report
        ),
    }


leaderboard_rows.append({
    "Model": (
        "Model_5_CrossFitted_Ensemble"
    ),
    "CV Mean AUC": (
        np.mean(
            ENSEMBLE_FOLD_AUC
        )
    ),
    "CV AUC SD": (
        np.std(
            ENSEMBLE_FOLD_AUC
        )
    ),
    "CV Mean Accuracy": (
        np.mean(
            ENSEMBLE_FOLD_ACCURACY
        )
    ),
    "CV Accuracy SD": (
        np.std(
            ENSEMBLE_FOLD_ACCURACY
        )
    ),
    "Test Mean AUC": (
        FINAL_TEST_MEAN_AUC
    ),
    "Test Mean Accuracy": (
        FINAL_TEST_MEAN_ACCURACY
    ),
    "OOF Coverage": 1.0,
    "Test Coverage": 1.0,
})


LEADERBOARD = (
    pd.DataFrame(
        leaderboard_rows
    )
    .sort_values(
        "CV Mean AUC",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


LEADERBOARD.to_csv(
    DIRS["outputs"]
    / "09_model_leaderboard.csv",
    index=False,
)


display(
    LEADERBOARD
    .style
    .format({
        "CV Mean AUC": "{:.4f}",
        "CV AUC SD": "{:.4f}",
        "CV Mean Accuracy": "{:.4f}",
        "CV Accuracy SD": "{:.4f}",
        "Test Mean AUC": "{:.4f}",
        "Test Mean Accuracy": "{:.4f}",
        "OOF Coverage": "{:.1%}",
        "Test Coverage": "{:.1%}",
    })
    .background_gradient(
        subset=[
            "CV Mean AUC",
            "Test Mean AUC",
        ],
        cmap="YlGn",
    )
)

## 57. Five-fold model comparison with uncertainty bars

In [ ]:
plot_leaderboard = (
    LEADERBOARD
    .sort_values(
        "CV Mean AUC",
        ascending=True,
    )
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(17, 7),
)


axes[0].barh(
    plot_leaderboard[
        "Model"
    ],
    plot_leaderboard[
        "CV Mean AUC"
    ],
    xerr=(
        plot_leaderboard[
            "CV AUC SD"
        ]
    ),
    capsize=4,
)


axes[0].axvline(
    TARGET_MEAN_AUC,
    linestyle="--",
    linewidth=1.5,
    label=(
        f"Stretch target "
        f"{TARGET_MEAN_AUC:.2f}"
    ),
)


axes[0].set_xlim(
    0.5,
    1.0,
)


axes[0].set_title(
    "5-Fold CV — Mean ROC-AUC",
    fontweight="bold",
)


axes[0].set_xlabel(
    "Mean ROC-AUC"
)


axes[0].legend()


axes[1].barh(
    plot_leaderboard[
        "Model"
    ],
    plot_leaderboard[
        "CV Mean Accuracy"
    ],
    xerr=(
        plot_leaderboard[
            "CV Accuracy SD"
        ]
    ),
    capsize=4,
)


axes[1].set_xlim(
    0.5,
    1.0,
)


axes[1].set_title(
    "5-Fold CV — Mean Accuracy",
    fontweight="bold",
)


axes[1].set_xlabel(
    "Mean Accuracy"
)


plt.suptitle(
    "Optimized Tox21 Model Comparison",
    fontsize=16,
    fontweight="bold",
)


plt.tight_layout()


plt.savefig(
    DIRS["figures"]
    / "05_cv_model_comparison.png",
    dpi=240,
    bbox_inches="tight",
)


plt.show()

## 58. Model × endpoint OOF AUC heatmap

In [ ]:
heatmap_rows = {}


for model_name in (
    ENSEMBLE_MODEL_NAMES
):

    artifact = (
        PREDICTION_ARTIFACTS[
            model_name
        ]
    )


    report = (
        evaluate_multitask_nan_safe(
            Y_TRAIN,
            M_TRAIN,
            artifact[
                "oof_prob"
            ],
            thresholds=np.full(
                len(
                    TARGET_COLS
                ),
                0.5,
            ),
            endpoints=(
                TARGET_COLS
            ),
        )
    )


    heatmap_rows[
        model_name
    ] = (
        report
        .set_index(
            "Endpoint"
        )[
            "AUC"
        ]
        .reindex(
            TARGET_COLS
        )
    )


ensemble_oof_report = (
    evaluate_multitask_nan_safe(
        Y_TRAIN,
        M_TRAIN,
        CROSS_FITTED_ENSEMBLE_OOF,
        thresholds=np.full(
            len(
                TARGET_COLS
            ),
            0.5,
        ),
        endpoints=(
            TARGET_COLS
        ),
    )
)


heatmap_rows[
    "Final_Ensemble"
] = (
    ensemble_oof_report
    .set_index(
        "Endpoint"
    )[
        "AUC"
    ]
    .reindex(
        TARGET_COLS
    )
)


AUC_HEATMAP = (
    pd.DataFrame(
        heatmap_rows
    )
)


plt.figure(
    figsize=(
        max(
            11,
            1.6
            * AUC_HEATMAP
            .shape[1]
        ),
        8,
    )
)


sns.heatmap(
    AUC_HEATMAP,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    vmin=0.5,
    vmax=1.0,
    linewidths=0.4,
)


plt.title(
    "OOF ROC-AUC Heatmap — Models × Tox21 Endpoints",
    fontweight="bold",
)


plt.xlabel(
    "Model"
)


plt.ylabel(
    "Endpoint"
)


plt.tight_layout()


plt.savefig(
    DIRS["figures"]
    / "06_oof_auc_heatmap.png",
    dpi=240,
    bbox_inches="tight",
)


plt.show()

## 59. Final ensemble per-endpoint AUC and Accuracy

In [ ]:
endpoint_plot = (
    FINAL_TEST_REPORT
    .set_index(
        "Endpoint"
    )
    .reindex(
        TARGET_COLS
    )
    .reset_index()
)


x_positions = np.arange(
    len(
        endpoint_plot
    )
)


bar_width = 0.38


plt.figure(
    figsize=(17, 7)
)


plt.bar(
    x_positions
    - bar_width / 2,
    endpoint_plot[
        "AUC"
    ],
    width=(
        bar_width
    ),
    label="ROC-AUC",
)


plt.bar(
    x_positions
    + bar_width / 2,
    endpoint_plot[
        "Accuracy"
    ],
    width=(
        bar_width
    ),
    label="Accuracy",
)


plt.axhline(
    TARGET_MEAN_AUC,
    linestyle="--",
    linewidth=1.4,
    label=(
        f"AUC target "
        f"{TARGET_MEAN_AUC:.2f}"
    ),
)


plt.xticks(
    x_positions,
    endpoint_plot[
        "Endpoint"
    ],
    rotation=35,
    ha="right",
)


plt.ylim(
    0,
    1.05,
)


plt.ylabel(
    "Score"
)


plt.title(
    "Final Ensemble — Per-Endpoint Test Performance",
    fontweight="bold",
)


plt.legend(
    loc="lower right"
)


plt.tight_layout()


plt.savefig(
    DIRS["figures"]
    / "07_final_endpoint_auc_accuracy.png",
    dpi=240,
    bbox_inches="tight",
)


plt.show()

## 60. Per-endpoint ROC curves

In [ ]:
fig, axes = plt.subplots(
    3,
    4,
    figsize=(18, 14),
)


for task, axis in enumerate(
    axes.ravel()
):

    valid = (
        TEST_M[
            :,
            task
        ]
        .astype(
            bool
        )
    )


    y_task = (
        TEST_Y[
            valid,
            task
        ]
    )


    p_task = (
        FINAL_ENSEMBLE_TEST[
            valid,
            task
        ]
    )


    if (
        len(
            np.unique(
                y_task
            )
        )
        == 2
    ):

        false_positive_rate, true_positive_rate, _ = (
            roc_curve(
                y_task,
                p_task,
            )
        )


        auc_value = (
            roc_auc_score(
                y_task,
                p_task,
            )
        )


        axis.plot(
            false_positive_rate,
            true_positive_rate,
            linewidth=2,
            label=(
                f"AUC={auc_value:.3f}"
            ),
        )


    axis.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1,
    )


    axis.set_title(
        TARGET_COLS[
            task
        ],
        fontweight="bold",
    )


    axis.set_xlabel(
        "False Positive Rate"
    )


    axis.set_ylabel(
        "True Positive Rate"
    )


    axis.legend(
        loc="lower right"
    )


plt.suptitle(
    "Final Ensemble — ROC Curves for 12 Tox21 Endpoints",
    fontsize=16,
    fontweight="bold",
)


plt.tight_layout()


plt.savefig(
    DIRS["figures"]
    / "08_final_roc_curves.png",
    dpi=240,
    bbox_inches="tight",
)


plt.show()

## 61. Best and hardest endpoint confusion matrices

In [ ]:
best_endpoint_row = (
    FINAL_TEST_REPORT
    .sort_values(
        "AUC",
        ascending=False,
    )
    .iloc[0]
)


hardest_endpoint_row = (
    FINAL_TEST_REPORT
    .sort_values(
        "AUC",
        ascending=True,
    )
    .iloc[0]
)


selected_endpoint_rows = [
    best_endpoint_row,
    hardest_endpoint_row,
]


fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5.5),
)


for axis, endpoint_row in zip(
    axes,
    selected_endpoint_rows,
):

    endpoint = (
        endpoint_row[
            "Endpoint"
        ]
    )

    task = TARGET_COLS.index(
        endpoint
    )

    valid = (
        TEST_M[
            :,
            task
        ]
        .astype(
            bool
        )
    )


    y_task = (
        TEST_Y[
            valid,
            task
        ]
        .astype(
            int
        )
    )


    p_task = (
        FINAL_ENSEMBLE_TEST[
            valid,
            task
        ]
    )


    prediction = (
        p_task
        >= ENSEMBLE_THRESHOLDS[
            task
        ]
    ).astype(
        int
    )


    matrix = confusion_matrix(
        y_task,
        prediction,
        labels=[0, 1],
    )


    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axis,
        xticklabels=[
            "Non-Toxic",
            "Toxic",
        ],
        yticklabels=[
            "Non-Toxic",
            "Toxic",
        ],
    )


    axis.set_title(
        f"{endpoint}\n"
        f"AUC={endpoint_row['AUC']:.3f}, "
        f"ACC={endpoint_row['Accuracy']:.3f}",
        fontweight="bold",
    )


    axis.set_xlabel(
        "Predicted label"
    )


    axis.set_ylabel(
        "True label"
    )


plt.suptitle(
    "Final Ensemble — Best vs. Hardest Endpoint",
    fontsize=16,
    fontweight="bold",
)


plt.tight_layout()


plt.savefig(
    DIRS["figures"]
    / "09_best_hardest_confusion_matrices.png",
    dpi=240,
    bbox_inches="tight",
)


plt.show()


print(
    "Best endpoint:",
    best_endpoint_row[
        "Endpoint"
    ],
)


print(
    "Hardest endpoint:",
    hardest_endpoint_row[
        "Endpoint"
    ],
)

## 62. Model diversity analysis

In [ ]:
diversity_columns = {}


for model_name in (
    ENSEMBLE_MODEL_NAMES
):

    probabilities = (
        PREDICTION_ARTIFACTS[
            model_name
        ][
            "oof_prob"
        ]
    )


    diversity_columns[
        model_name
    ] = np.nanmean(
        probabilities,
        axis=1,
    )


diversity_dataframe = (
    pd.DataFrame(
        diversity_columns
    )
)


correlation_matrix = (
    diversity_dataframe
    .corr(
        method="spearman"
    )
)


plt.figure(
    figsize=(9, 7)
)


sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)


plt.title(
    "OOF Prediction Diversity — Spearman Correlation",
    fontweight="bold",
)


plt.tight_layout()


plt.savefig(
    DIRS["figures"]
    / "10_oof_model_diversity.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()

## 63. Final compact dashboard

In [ ]:
dashboard = pd.DataFrame({
    "Measure": [
        "5-fold CV Mean ROC-AUC",
        "5-fold CV ROC-AUC SD",
        "5-fold CV Mean Accuracy",
        "5-fold CV Accuracy SD",
        "Frozen Test Mean ROC-AUC",
        "Frozen Test Mean Accuracy",
        "Test AUC 95% CI lower",
        "Test AUC 95% CI upper",
        "Models in final ensemble",
        "Train molecules",
        "Test molecules",
        "Scaffold overlap",
    ],

    "Value": [
        np.mean(
            ENSEMBLE_FOLD_AUC
        ),
        np.std(
            ENSEMBLE_FOLD_AUC
        ),
        np.mean(
            ENSEMBLE_FOLD_ACCURACY
        ),
        np.std(
            ENSEMBLE_FOLD_ACCURACY
        ),
        FINAL_TEST_MEAN_AUC,
        FINAL_TEST_MEAN_ACCURACY,
        FINAL_TEST_CI[
            0
        ],
        FINAL_TEST_CI[
            2
        ],
        len(
            ENSEMBLE_MODEL_NAMES
        ),
        len(
            train_idx
        ),
        len(
            test_idx
        ),
        len(
            train_scaffolds
            & test_scaffolds
        ),
    ],
})


display(
    dashboard
    .style
    .format({
        "Value": (
            lambda value:
            f"{value:.4f}"
            if isinstance(
                value,
                (
                    float,
                    np.floating,
                )
            )
            else str(
                value
            )
        )
    })
)


dashboard.to_csv(
    DIRS["outputs"]
    / "10_final_dashboard.csv",
    index=False,
)

## 64. Reproducibility manifest and model card

In [ ]:
def sha256_file(
    path,
    chunk_size=1 << 20,
):

    hasher = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as file:

        while True:

            block = file.read(
                chunk_size
            )

            if not block:

                break

            hasher.update(
                block
            )


    return hasher.hexdigest()


REPRODUCIBILITY_MANIFEST = {
    "dataset_path": str(
        TOX21_PATH
    ),
    "dataset_sha256": (
        sha256_file(
            TOX21_PATH
        )
    ),
    "curated_dataset": str(
        CURATED_CSV
    ),
    "split_manifest": str(
        SPLIT_FILE
    ),
    "cv_manifest": str(
        CV_FILE
    ),
    "primary_seed": (
        PRIMARY_SEED
    ),
    "cv_folds": (
        N_CV_FOLDS
    ),
    "train_fraction": (
        TRAIN_FRACTION
    ),
    "target_mean_auc": (
        TARGET_MEAN_AUC
    ),
    "models_completed": (
        ENSEMBLE_MODEL_NAMES
    ),
    "final_cv_mean_auc": (
        float(
            np.mean(
                ENSEMBLE_FOLD_AUC
            )
        )
    ),
    "final_cv_mean_accuracy": (
        float(
            np.mean(
                ENSEMBLE_FOLD_ACCURACY
            )
        )
    ),
    "final_test_mean_auc": (
        FINAL_TEST_MEAN_AUC
    ),
    "final_test_mean_accuracy": (
        FINAL_TEST_MEAN_ACCURACY
    ),
    "test_auc_ci_95": [
        float(
            FINAL_TEST_CI[
                0
            ]
        ),
        float(
            FINAL_TEST_CI[
                2
            ]
        ),
    ],
    "zero_scaffold_overlap": (
        len(
            train_scaffolds
            & test_scaffolds
        )
        == 0
    ),
    "target_assay_edges_allowed": (
        False
    ),
}


with open(
    DIRS["reports"]
    / "Auc_High_reproducibility_manifest.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        REPRODUCIBILITY_MANIFEST,
        file,
        indent=2,
        ensure_ascii=False,
    )


MODEL_CARD = {
    "name": (
        "Auc_High Tox21 Black-Box Ensemble"
    ),
    "task": (
        "12-endpoint multitask binary toxicity prediction"
    ),
    "models": (
        ENSEMBLE_MODEL_NAMES
    ),
    "primary_metric": (
        "Macro mean ROC-AUC"
    ),
    "secondary_metric": (
        "Macro mean Accuracy"
    ),
    "validation": (
        "75/25 scaffold holdout + 5-fold scaffold-aware CV"
    ),
    "missing_labels": (
        "Per-task masked; never imputed as non-toxic"
    ),
    "knowledge_graph": (
        "Leakage-safe chemical-gene-pathway core relations only"
    ),
    "limitations": [
        (
            "AUC above 0.95 is a target, not a guaranteed result."
        ),
        (
            "KG coverage may be incomplete and is reported separately."
        ),
        (
            "Literature results are not directly comparable unless "
            "dataset cohort and split protocol match."
        ),
        (
            "Accuracy can be optimistic on imbalanced endpoints; "
            "diagnostic minority-class metrics are retained."
        ),
    ],
}


with open(
    DIRS["reports"]
    / "Auc_High_model_card.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        MODEL_CARD,
        file,
        indent=2,
        ensure_ascii=False,
    )


print(
    "Reproducibility manifest and model card saved."
)

# End of notebook

## Final interpretation rule

The final scientific result is the **highest reproducible, leakage-free score** under the frozen scaffold protocol.

- Do not repeatedly change the model after seeing final test labels.
- Do not report the best fold as the overall result.
- Do not compare directly with a paper unless the cohort and split protocol are matched.
- Report the five-fold mean ± standard deviation, final frozen-test metrics, coverage and uncertainty.

All artifacts are saved under:

- `outputs/`
- `models/`
- `figures/`
- `reports/`